# 대회 데이터 검증 및 통합 학습 데이터 구성

### 목표
- 대회 학습 데이터의 오류를 검증하고, 확인된 오류는 수정 없이 제외한다.
- 검증한 대회 학습 데이터와 SELECT 유지 목록의 데이터를 통합한다.
- 통합 데이터를 Train / Validation으로 분리한다.
- 탐지 모델용 Dataset과 DataLoader를 구성하고 배치를 확인한다.

### 진행 계획
| 단계 | 할 일 | 남길 근거 |
|---|---|---|
| ① 사전 준비 | ZIP 경로·내부 구조·대표 JSON 확인 | 파일 수, JSON 구조, ID·클래스 체계 |
| ② 대회 데이터 검증 | 제공한 점검표 순서대로 검사·판정 | 검사 결과, 후보 검토, 제외 사유 |
| ③ 데이터 통합 | 대회 유지 데이터와 SELECT 유지 데이터 통합 | 출처, 클래스 매핑, 중복·테스트 혼입 확인 |
| ④ 데이터 분리 | Train / Validation 분리 | 그룹 기준, 분리 비율, 클래스 분포, 누수 확인 |
| ⑤ DataLoader 구성 | 이미지·bbox·클래스 로딩 및 배치 확인 | 입력 형식, 배치 시각화, 실행 결과 |

### 처리 원칙
- 원본 ZIP과 원본 파일은 보존한다.
- 오류가 확인된 이미지는 연결된 annotation 전체와 함께 제외한다.
- 유사 이미지·통계적 이상치·위치 편향은 자동 제외하지 않는다.
- 대회 테스트 데이터는 학습·Validation에 포함하지 않는다.
- SELECT 데이터는 전체 폴더가 아닌 최종 유지 목록 10,109장을 사용한다.
- 정상 결과는 실행 출력으로 남기고, 제외 결정에는 판단 근거를 기록한다.

## 1. 사전 준비

### 1-1. 점검표 외·경로 및 내부 구조 확인

In [ ]:
from pathlib import Path
from collections import Counter
import json
import zipfile

import pandas as pd
from IPython.display import display

# Finder에서 복사한 실제 폴더 경로
CODEIT_DIR = Path(
    "/Users/codeit/Desktop/코드잇/project/pill-project/data/codeit"
)

# 해당 대회명에 일치하는 ZIP 확인
zip_candidates = sorted(
    CODEIT_DIR.glob("sprint_ai_project1_data_20261022*.zip")
)

if len(zip_candidates) != 1:
    raise ValueError(
        "대회 ZIP을 하나로 확정할 수 없습니다.\n"
        f"발견한 ZIP: {[path.name for path in zip_candidates]}"
    )

DATA_PATH = zip_candidates[0]


def is_data_file(member_name):
    """macOS 보조 파일 제외."""
    return (
        "__MACOSX" not in Path(member_name).parts
        and not Path(member_name).name.startswith(".")
    )


def read_dataset_json(member_name):
    """ZIP 내부 JSON을 압축 해제 없이 읽기."""
    with zipfile.ZipFile(DATA_PATH, "r") as z:
        return json.loads(
            z.read(member_name).decode("utf-8-sig")
        )


# 1. ZIP 내부 파일 목록
with zipfile.ZipFile(DATA_PATH, "r") as z:
    data_members = sorted(
        member.filename
        for member in z.infolist()
        if not member.is_dir()
        and is_data_file(member.filename)
    )

image_extensions = {".png", ".jpg", ".jpeg", ".webp"}

image_members = [
    name for name in data_members
    if Path(name).suffix.lower() in image_extensions
]
json_members = [
    name for name in data_members
    if Path(name).suffix.lower() == ".json"
]

print("========== 대회 ZIP 확인 ==========")
print(f"검사 ZIP: {DATA_PATH}")
print(f"이미지 파일: {len(image_members):,}개")
print(f"JSON 파일: {len(json_members):,}개")

# 2. 폴더별 파일 구성
structure_counts = Counter(
    (
        "/".join(Path(name).parts[:-1]) or "(최상위)",
        Path(name).suffix.lower() or "(확장자 없음)",
    )
    for name in data_members
)

print("\n========== 폴더별 파일 구성 ==========")
display(pd.DataFrame([
    {
        "폴더": folder,
        "확장자": extension,
        "파일 수": count,
    }
    for (folder, extension), count in sorted(
        structure_counts.items()
    )
]))

print("\n========== JSON 경로 표본: 최대 10개 ==========")
for name in json_members[:10]:
    print(name)

# 3. 학습 경로가 표시된 JSON부터 대표 구조 확인
sample_candidates = sorted(
    json_members,
    key=lambda name: (
        "train" not in name.lower(),
        name,
    ),
)

if sample_candidates:
    sample_json_member = sample_candidates[0]
    sample_document = read_dataset_json(sample_json_member)

    print(
        f"\n========== 대표 JSON: {sample_json_member} =========="
    )
    print(
        f"최상위 자료형: {type(sample_document).__name__}"
    )

    if isinstance(sample_document, dict):
        print(f"최상위 키: {list(sample_document.keys())}")

        for section in ["images", "annotations", "categories"]:
            value = sample_document.get(section)

            print(
                f"\n[{section}] "
                f"자료형: {type(value).__name__}"
            )

            if isinstance(value, list):
                print(f"항목 수: {len(value):,}개")

                if value:
                    print(json.dumps(
                        value[0],
                        ensure_ascii=False,
                        indent=2,
                    ))
    else:
        print(str(sample_document)[:1000])

else:
    print("\nJSON이 없습니다. 라벨 저장 형식 확인이 필요합니다.")

print("\n※ ZIP 내부 목록과 대표 JSON 구조만 확인했습니다.")
print("※ 오류 검사·압축 해제·파일 수정은 수행하지 않았습니다.")

In [ ]:
# 검사 대상: 대회 학습 이미지·라벨만
train_image_members = [
    name for name in image_members
    if "train_images" in Path(name).parts
]

train_json_members = [
    name for name in json_members
    if "train_annotations" in Path(name).parts
]

# 테스트는 별도 목록으로만 보관
test_image_members = [
    name for name in image_members
    if "test_images" in Path(name).parts
]

print("========== 검사 범위 확정 ==========")
print(f"검사할 학습 이미지: {len(train_image_members):,}장")
print(f"검사할 학습 JSON: {len(train_json_members):,}개")
print(f"검사 대상에서 제외한 테스트 이미지: {len(test_image_members):,}장")

## 2. 대회 데이터 검증

### 2-1. Annotation ID 중복 (1/18)
- Train JSON 763개를 공통 테이블로 읽고, 점검표 기준대로 같은 annotation ID에 서로 다른 (image_id, bbox, category_id)가 연결되는지 검사.

In [ ]:
# 점검표 1. Annotation ID 중복
# 같은 ID인데 (image_id, bbox, category_id)가 다른 기록 검색

codeit_table_rows = {
    "images": [],
    "annotations": [],
    "categories": [],
}
codeit_read_issue_rows = []

# 1. Train JSON만 읽어 공통 테이블 구성
with zipfile.ZipFile(DATA_PATH, "r") as z:
    for json_file in train_json_members:
        try:
            document = json.loads(
                z.read(json_file).decode("utf-8-sig")
            )
        except Exception as error:
            codeit_read_issue_rows.append({
                "json_file": json_file,
                "detail": (
                    f"JSON 읽기 실패: "
                    f"{type(error).__name__}: {error}"
                ),
            })
            continue

        if not isinstance(document, dict):
            codeit_read_issue_rows.append({
                "json_file": json_file,
                "detail": "최상위 자료형이 dict가 아님",
            })
            continue

        for section in codeit_table_rows:
            records = document.get(section)

            if not isinstance(records, list):
                codeit_read_issue_rows.append({
                    "json_file": json_file,
                    "detail": f"{section} 누락 또는 list가 아님",
                })
                continue

            for index, record in enumerate(records):
                if not isinstance(record, dict):
                    codeit_read_issue_rows.append({
                        "json_file": json_file,
                        "detail": f"{section}[{index}]가 dict가 아님",
                    })
                    continue

                codeit_table_rows[section].append({
                    **record,
                    "_json_file": json_file,
                    "_record_index": index,
                })

codeit_images_df = pd.DataFrame(codeit_table_rows["images"])
codeit_annotations_df = pd.DataFrame(
    codeit_table_rows["annotations"]
)
codeit_categories_df = pd.DataFrame(
    codeit_table_rows["categories"]
)
codeit_read_issues = pd.DataFrame(
    codeit_read_issue_rows,
    columns=["json_file", "detail"],
)

print("========== Train JSON 읽기 ==========")
print(f"대상 JSON: {len(train_json_members):,}개")
print(f"이미지 정보 기록: {len(codeit_images_df):,}개")
print(f"Annotation 기록: {len(codeit_annotations_df):,}개")
print(f"읽기·구조 문제 기록: {len(codeit_read_issues):,}건")

if not codeit_read_issues.empty:
    display(codeit_read_issues.head(20))
    raise ValueError(
        "읽기·구조 문제가 있어 전체 ID 검사 완료 판정을 보류합니다."
    )

# ID 비교에 필요한 필드 확인
required_id_fields = ["id", "image_id", "bbox", "category_id"]

missing_id_fields = [
    field for field in required_id_fields
    if field not in codeit_annotations_df.columns
]

if missing_id_fields:
    raise ValueError(f"필수 컬럼 누락: {missing_id_fields}")

missing_id_values = codeit_annotations_df[
    required_id_fields
].isna().any(axis=1)

if missing_id_values.any():
    display(
        codeit_annotations_df.loc[
            missing_id_values,
            ["_json_file", *required_id_fields],
        ].head(20)
    )
    raise ValueError("ID 검사에 필요한 값이 누락되어 있습니다.")

# 2. 비교용 서명 생성 — 원래 bbox 값은 변경하지 않음
annotation_id_checks = codeit_annotations_df.copy()

annotation_id_checks["_id_key"] = annotation_id_checks["id"].map(
    lambda value: json.dumps(value, ensure_ascii=False)
)

annotation_id_checks["_signature"] = [
    json.dumps(
        [image_id, bbox, category_id],
        ensure_ascii=False,
        sort_keys=True,
    )
    for image_id, bbox, category_id in zip(
        annotation_id_checks["image_id"],
        annotation_id_checks["bbox"],
        annotation_id_checks["category_id"],
    )
]

# 3. 같은 ID에 연결된 내용의 종류 수 집계
annotation_id_summary = (
    annotation_id_checks.groupby("_id_key", sort=False)
    .agg(
        record_count=("_signature", "size"),
        signature_count=("_signature", "nunique"),
        json_count=("_json_file", "nunique"),
    )
    .reset_index()
)

conflicting_id_keys = set(
    annotation_id_summary.loc[
        annotation_id_summary["signature_count"] > 1,
        "_id_key",
    ]
)

annotation_id_conflicts = annotation_id_checks.loc[
    annotation_id_checks["_id_key"].isin(conflicting_id_keys)
].copy()

identical_repeated_ids = annotation_id_summary.loc[
    (annotation_id_summary["record_count"] > 1)
    & (annotation_id_summary["signature_count"] == 1)
].copy()

print("\n========== 점검표 1. Annotation ID 중복 ==========")
print(f"검사 annotation: {len(annotation_id_checks):,}개")
print(f"서로 다른 내용이 연결된 ID: {len(conflicting_id_keys):,}개")
print(f"해당 annotation 기록: {len(annotation_id_conflicts):,}개")
print(
    f"영향 JSON: "
    f"{annotation_id_conflicts['_json_file'].nunique():,}개"
)
print(
    f"ID와 비교 내용이 모두 같은 반복 ID: "
    f"{len(identical_repeated_ids):,}개"
)

if annotation_id_conflicts.empty:
    print(
        "\n[체크 가능] Annotation ID 중복 "
        "— 점검표 기준의 충돌 없음"
    )
else:
    print("\n========== 충돌 후보: 최대 20건 ==========")
    display(annotation_id_conflicts[[
        "_json_file",
        "id",
        "image_id",
        "category_id",
        "bbox",
    ]].head(20))

    print(
        "\n※ 파일 내부의 ID인지, 전체 데이터의 고유 ID인지 "
        "확인한 뒤 오류 여부를 판단합니다."
    )

print("\n※ 같은 ID의 단순 반복과 서로 다른 내용의 연결을 구분했습니다.")
print("※ 이번 결과만으로 제외를 확정하지 않았습니다.")
print("※ 테스트 데이터 읽기·파일 수정·삭제 없음")

### 2-2. 이미지 ID 중복 (2/18)

다음 두 경우를 검사합니다.
- 같은 im.id에 **서로 다른 file_name**이 연결됨
- 같은 file_name에 **서로 다른 im.id**가 연결됨

같은 이미지의 ID·파일명이 알약별 JSON에 반복되는 것은 정상으로 처리

In [ ]:
# 2-2. 이미지 ID 중복 (2/18)

image_id_checks = codeit_images_df.copy()

required_image_fields = ["id", "file_name"]

if not set(required_image_fields).issubset(image_id_checks.columns):
    raise ValueError("이미지 정보에 id 또는 file_name 컬럼이 없습니다.")

valid_image_record = (
    image_id_checks["id"].notna()
    & image_id_checks["file_name"].map(
        lambda value: isinstance(value, str) and bool(value.strip())
    )
)

if not valid_image_record.all():
    display(
        image_id_checks.loc[
            ~valid_image_record,
            ["_json_file", "id", "file_name"],
        ].head(20)
    )
    raise ValueError("이미지 ID 또는 파일명 누락·형식을 먼저 확인해야 합니다.")

# 1. 같은 이미지 ID에 서로 다른 파일명이 연결되는지 확인
image_id_summary = (
    image_id_checks.groupby("id")
    .agg(
        filename_count=("file_name", "nunique"),
        record_count=("file_name", "size"),
    )
    .reset_index()
)

conflicting_image_ids = set(
    image_id_summary.loc[
        image_id_summary["filename_count"] > 1, "id"
    ]
)

image_id_filename_conflicts = image_id_checks.loc[
    image_id_checks["id"].isin(conflicting_image_ids)
].copy()

# 2. 같은 파일명에 서로 다른 이미지 ID가 연결되는지 확인
image_filename_summary = (
    image_id_checks.groupby("file_name")
    .agg(
        image_id_count=("id", "nunique"),
        record_count=("id", "size"),
    )
    .reset_index()
)

conflicting_image_filenames = set(
    image_filename_summary.loc[
        image_filename_summary["image_id_count"] > 1,
        "file_name",
    ]
)

image_filename_id_conflicts = image_id_checks.loc[
    image_id_checks["file_name"].isin(conflicting_image_filenames)
].copy()

# 두 충돌 중 하나라도 해당하는 기록
image_id_conflicts = image_id_checks.loc[
    image_id_checks["id"].isin(conflicting_image_ids)
    | image_id_checks["file_name"].isin(conflicting_image_filenames)
].copy()

print("========== 2-2. 이미지 ID 중복 (2/18) ==========")
print(f"전체 이미지 정보 기록: {len(image_id_checks):,}개")
print(f"고유 이미지 ID: {image_id_checks['id'].nunique():,}개")
print(f"고유 파일명: {image_id_checks['file_name'].nunique():,}개")
print(f"여러 파일명에 연결된 ID: {len(conflicting_image_ids):,}개")
print(
    f"여러 ID에 연결된 파일명: "
    f"{len(conflicting_image_filenames):,}개"
)
print(
    f"충돌이 있는 JSON: "
    f"{image_id_conflicts['_json_file'].nunique():,}개"
)

if image_id_conflicts.empty:
    print("\n[체크 가능] 이미지 ID 중복 — 점검표 기준의 충돌 없음")
    print("같은 ID·파일명의 반복 기록은 정상으로 처리했습니다.")
else:
    if not image_id_filename_conflicts.empty:
        print("\n========== 같은 ID → 여러 파일명: 최대 20건 ==========")
        display(image_id_filename_conflicts[[
            "_json_file", "id", "file_name"
        ]].head(20))

    if not image_filename_id_conflicts.empty:
        print("\n========== 같은 파일명 → 여러 ID: 최대 20건 ==========")
        display(image_filename_id_conflicts[[
            "_json_file", "id", "file_name"
        ]].head(20))

    print("\n충돌 원인을 확인한 뒤 제외 여부를 판단합니다.")

print("\n※ 파일 수정·삭제 없음")

### 2-3. bbox 면적과 넓이×높이 불일치 (3/18)
- 각 annotation의 **area와 bbox의 w × h**를 비교


In [ ]:
# 2-3. bbox 면적과 넓이×높이 불일치 (3/18)

import math
from numbers import Real


def is_finite_number(value):
    """문자열·bool을 제외한 유한한 숫자인지 확인."""
    return (
        isinstance(value, Real)
        and not isinstance(value, bool)
        and math.isfinite(value)
    )


area_check_rows = []

# DataFrame의 자료형 변환 영향을 피하고 원래 JSON 기록 사용
for annotation in codeit_table_rows["annotations"]:
    bbox = annotation.get("bbox")
    area = annotation.get("area")

    bbox_valid = (
        isinstance(bbox, list)
        and len(bbox) == 4
        and all(is_finite_number(value) for value in bbox)
    )

    expected_area = None

    if not bbox_valid:
        status = "계산 불가: bbox 형식·숫자 오류"

    elif not is_finite_number(area):
        status = "계산 불가: area 누락·숫자 오류"

    else:
        expected_area = float(bbox[2]) * float(bbox[3])

        if not math.isfinite(expected_area):
            status = "계산 불가: 면적 계산 결과가 유한하지 않음"
        elif math.isclose(
            float(area),
            expected_area,
            rel_tol=1e-9,
            abs_tol=1e-6,
        ):
            status = "일치"
        else:
            status = "불일치"

    area_check_rows.append({
        "json_file": annotation["_json_file"],
        "annotation_id": annotation["id"],
        "image_id": annotation["image_id"],
        "category_id": annotation["category_id"],
        "bbox": bbox,
        "area": area,
        "expected_area": expected_area,
        "status": status,
    })

codeit_area_checks = pd.DataFrame(area_check_rows)

codeit_area_mismatches = codeit_area_checks.loc[
    codeit_area_checks["status"].eq("불일치")
].copy()

codeit_area_unchecked = codeit_area_checks.loc[
    codeit_area_checks["status"].str.startswith("계산 불가")
].copy()

print("========== 2-3. bbox 면적 검사 (3/18) ==========")
print(f"전체 annotation: {len(codeit_area_checks):,}개")
print(
    f"면적 일치: "
    f"{codeit_area_checks['status'].eq('일치').sum():,}개"
)
print(f"면적 불일치: {len(codeit_area_mismatches):,}개")
print(f"계산 불가: {len(codeit_area_unchecked):,}개")

if not codeit_area_mismatches.empty:
    print("\n========== 면적 불일치: 최대 20건 ==========")
    display(codeit_area_mismatches.head(20))

if not codeit_area_unchecked.empty:
    print("\n========== 계산 불가: 최대 20건 ==========")
    display(codeit_area_unchecked.head(20))

if codeit_area_mismatches.empty and codeit_area_unchecked.empty:
    print("\n[체크 가능] bbox 면적과 넓이×높이 불일치 — 없음")
else:
    print("\n문제 기록을 확인한 뒤 제외 여부를 판단합니다.")

print("\n※ 원본 area를 재계산해 덮어쓰지 않았습니다.")
print("※ 좌표 범위와 음수·0 크기는 다음 항목에서 검사합니다.")
print("※ 파일 수정·삭제 없음")

In [ ]:
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# 이후 이미지 검사·시각화에서도 재사용할 Train 이미지 조회표
codeit_train_image_set = set(train_image_members)
codeit_train_image_by_name = {}

for member in train_image_members:
    codeit_train_image_by_name.setdefault(
        Path(member).name, []
    ).append(member)


def resolve_codeit_image_member(file_name):
    """JSON 파일명을 Train ZIP 내부 이미지 경로에 연결."""
    if file_name in codeit_train_image_set:
        return file_name

    matches = codeit_train_image_by_name.get(
        Path(file_name).name, []
    )

    if len(matches) != 1:
        raise ValueError(
            f"Train 이미지 경로를 확정할 수 없음: {file_name!r}, "
            f"파일명 일치 {len(matches)}개"
        )

    return matches[0]


# 불일치 annotation에 같은 JSON의 이미지 파일명 연결
area_review_records = codeit_area_mismatches.merge(
    codeit_images_df[[
        "_json_file", "id", "file_name"
    ]].rename(columns={
        "_json_file": "json_file",
        "id": "image_id",
    }),
    on=["json_file", "image_id"],
    how="left",
    validate="many_to_one",
)

if area_review_records["file_name"].isna().any():
    raise ValueError("불일치 annotation의 이미지 연결을 확인해야 합니다.")

area_review_records["area_difference"] = (
    area_review_records["area"]
    - area_review_records["expected_area"]
)

# 같은 이미지가 여러 JSON에 있어도 한 번만 출력
area_review_image_order = (
    area_review_records[["image_id", "file_name"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
area_review_image_order.insert(
    0, "review_number",
    range(1, len(area_review_image_order) + 1),
)

with zipfile.ZipFile(DATA_PATH, "r") as z:
    for _, image_record in area_review_image_order.iterrows():
        image_id = image_record["image_id"]
        image_member = resolve_codeit_image_member(
            image_record["file_name"]
        )

        with Image.open(BytesIO(z.read(image_member))) as image:
            image_rgb = image.convert("RGB")

        selected = area_review_records.loc[
            area_review_records["image_id"].eq(image_id)
        ]
        target_ids = set(selected["annotation_id"])

        all_annotations = codeit_annotations_df.loc[
            codeit_annotations_df["image_id"].eq(image_id)
        ].copy()

        # 빨간 박스를 마지막에 그려 강조
        all_annotations["_highlight"] = (
            all_annotations["id"].isin(target_ids)
        )
        all_annotations = all_annotations.sort_values("_highlight")

        print(
            f"\n검토 {image_record['review_number']} | "
            f"이미지 ID {image_id}"
        )
        print(f"이미지: {image_member}")

        with pd.option_context("display.max_colwidth", None):
            display(selected[[
                "json_file",
                "annotation_id",
                "category_id",
                "bbox",
                "area",
                "expected_area",
                "area_difference",
            ]])

        fig, ax = plt.subplots(figsize=(7, 9))
        ax.imshow(image_rgb)

        for _, annotation in all_annotations.iterrows():
            bbox = annotation["bbox"]

            # 시각화에 필요한 숫자 형식만 확인
            if not (
                isinstance(bbox, list)
                and len(bbox) == 4
                and all(is_finite_number(value) for value in bbox)
            ):
                print(
                    f"표시 불가 bbox: annotation ID "
                    f"{annotation['id']}"
                )
                continue

            x, y, w, h = bbox
            highlighted = annotation["_highlight"]
            color = "red" if highlighted else "gray"

            ax.add_patch(Rectangle(
                (x, y), w, h,
                fill=False,
                edgecolor=color,
                linewidth=2.5 if highlighted else 1,
            ))
            ax.text(
                x, y,
                f"ID {annotation['id']} | "
                f"class {annotation['category_id']}",
                color=color if highlighted else "black",
                fontsize=8,
                va="bottom",
                bbox=dict(
                    facecolor="white",
                    alpha=0.8,
                    edgecolor="none",
                ),
            )

        ax.set_title(
            f"Review {image_record['review_number']} | "
            f"Image ID {image_id}\n"
            "Red: area mismatch",
            fontsize=12,
        )
        ax.axis("off")
        plt.tight_layout()
        plt.show()
        plt.close(fig)

print("\n※ area_difference = 원본 area − bbox의 w×h")
print("※ 테스트 이미지 사용·파일 수정·삭제 없음")

### 2-3. bbox 면적과 넓이×높이 불일치 — 검토 결과

- 전체 annotation 763개 중 면적 일치 757개, 불일치 6개,
  계산 불가 0개를 확인했다.
- 불일치 annotation이 연결된 이미지 5장을 모두 시각 검토했다.
- 이미지 ID 1360은 bbox가 알약 아래쪽을 포함하지 못하는 것도 확인했다.
- 나머지 이미지는 대상 알약을 대체로 감싸지만,
  저장된 area와 bbox의 w×h 불일치는 별개로 존재한다.
- 빠른 데이터셋 구성을 위해 오류 데이터를 제외한다는 원칙에 따라
  이미지 ID 795, 1360, 696, 790, 1406을 제외 대상으로 결정한다.
- 제외 시 해당 이미지에 연결된 annotation 전체를 함께 제외한다.
- 원본 파일은 변경하지 않았으며, 최종 제외 목록 반영은 후속 단계에서 수행한다.

### 2-4. bbox 좌표가 이미지 범위 초과 (4/18)

In [ ]:
# 각 annotation과 같은 JSON에 있는 이미지 정보를 연결
codeit_image_size_lookup = {
    (row["_json_file"], row["id"]): row
    for row in codeit_table_rows["images"]
}

codeit_boundary_rows = []

for annotation in codeit_table_rows["annotations"]:
    image_info = codeit_image_size_lookup.get(
        (annotation["_json_file"], annotation.get("image_id"))
    )
    bbox = annotation.get("bbox")

    result = {
        "json_file": annotation["_json_file"],
        "annotation_id": annotation.get("id"),
        "image_id": annotation.get("image_id"),
        "category_id": annotation.get("category_id"),
        "bbox": bbox,
        "image_width": None,
        "image_height": None,
        "right_overflow": None,
        "bottom_overflow": None,
        "status": None,
    }

    if image_info is None:
        result["status"] = "이미지 정보 없음"

    else:
        width = image_info.get("width")
        height = image_info.get("height")

        result["image_width"] = width
        result["image_height"] = height

        if not (
            is_finite_number(width)
            and is_finite_number(height)
            and width > 0
            and height > 0
        ):
            result["status"] = "이미지 크기 확인 불가"

        elif not (
            isinstance(bbox, (list, tuple))
            and len(bbox) == 4
            and all(is_finite_number(value) for value in bbox)
        ):
            result["status"] = "bbox 계산 불가"

        else:
            x, y, w, h = bbox
            right = x + w
            bottom = y + h

            if not (
                is_finite_number(right)
                and is_finite_number(bottom)
            ):
                result["status"] = "bbox 계산 불가"

            else:
                result["right_overflow"] = max(0, right - width)
                result["bottom_overflow"] = max(0, bottom - height)
                result["status"] = (
                    "범위 초과"
                    if right > width or bottom > height
                    else "범위 이내"
                )

    codeit_boundary_rows.append(result)

codeit_boundary_checks = pd.DataFrame(codeit_boundary_rows)

codeit_boundary_errors = codeit_boundary_checks.loc[
    codeit_boundary_checks["status"].eq("범위 초과")
].copy()

codeit_boundary_unchecked = codeit_boundary_checks.loc[
    ~codeit_boundary_checks["status"].isin(["범위 이내", "범위 초과"])
].copy()

print("========== 2-4. bbox 좌표가 이미지 범위 초과 (4/18) ==========")
print(f"전체 annotation: {len(codeit_boundary_checks):,}개")
print(
    f"범위 이내: "
    f"{codeit_boundary_checks['status'].eq('범위 이내').sum():,}개"
)
print(f"범위 초과: {len(codeit_boundary_errors):,}개")
print(
    f"범위 초과 이미지: "
    f"{codeit_boundary_errors['image_id'].nunique():,}장"
)
print(f"계산 불가: {len(codeit_boundary_unchecked):,}개")

if not codeit_boundary_errors.empty:
    print("\n========== 범위 초과: 최대 20건 ==========")
    display(codeit_boundary_errors.head(20))

if not codeit_boundary_unchecked.empty:
    print("\n========== 계산 불가: 최대 20건 ==========")
    display(codeit_boundary_unchecked.head(20))

if codeit_boundary_errors.empty and codeit_boundary_unchecked.empty:
    print("\n[체크 가능] bbox 좌표가 이미지 범위 초과 — 없음")
else:
    print("\n자동 검사 완료 — 발견 항목 확인 필요")

print("\n※ 음수 좌표와 너비·높이 0 이하는 다음 단계에서 검사합니다.")
print("※ 파일 수정·삭제 없음")

In [ ]:
for row in codeit_boundary_errors.to_dict("records"):
    print(f"JSON: {row['json_file']}")
    print(f"Annotation ID: {row['annotation_id']}")
    print(f"Image ID: {row['image_id']}")
    print(f"bbox: {row['bbox']}")
    print(f"이미지 크기: {row['image_width']} × {row['image_height']}")
    print(f"오른쪽 초과: {row['right_overflow']}px")
    print(f"아래쪽 초과: {row['bottom_overflow']}px")

### 2-4. bbox 좌표가 이미지 범위 초과 — 검사 결과

- 전체 annotation 763개 중 범위 초과 1개, 계산 불가 0개.
- 오류 annotation ID: 903 / 이미지 ID: 239
- 이미지 크기: 976 × 1280px
- bbox: [6567, 625, 311, 315]
- x 좌표부터 이미지 너비를 초과하며, 박스 오른쪽 끝은 5902px 초과함.
- 빠른 데이터셋 구성을 위해 좌표를 수정하지 않고 해당 이미지와 연결된 모든 annotation을 제외하기로 결정함.
- 현재는 제외 결정만 기록하며, 실제 제외는 최종 데이터셋 구성 시 적용함.

### 2-5. bbox 좌표 음수 또는 크기 0 이하 (5/18)

In [ ]:
codeit_bbox_validity_rows = []

for annotation in codeit_table_rows["annotations"]:
    bbox = annotation.get("bbox")
    reasons = []

    valid_format = (
        isinstance(bbox, (list, tuple))
        and len(bbox) == 4
        and all(is_finite_number(value) for value in bbox)
    )

    if valid_format:
        x, y, w, h = bbox

        if x < 0:
            reasons.append("x 좌표 음수")
        if y < 0:
            reasons.append("y 좌표 음수")
        if w <= 0:
            reasons.append("너비 0 이하")
        if h <= 0:
            reasons.append("높이 0 이하")

        status = "오류" if reasons else "정상"
    else:
        status = "계산 불가"
        reasons.append("bbox 형식 또는 숫자 확인 필요")

    codeit_bbox_validity_rows.append({
        "json_file": annotation["_json_file"],
        "annotation_id": annotation.get("id"),
        "image_id": annotation.get("image_id"),
        "category_id": annotation.get("category_id"),
        "bbox": bbox,
        "status": status,
        "reason": " / ".join(reasons),
    })

codeit_bbox_validity_checks = pd.DataFrame(codeit_bbox_validity_rows)

codeit_bbox_validity_errors = codeit_bbox_validity_checks.loc[
    codeit_bbox_validity_checks["status"].eq("오류")
].copy()

codeit_bbox_validity_unchecked = codeit_bbox_validity_checks.loc[
    codeit_bbox_validity_checks["status"].eq("계산 불가")
].copy()

print("========== 2-5. bbox 좌표 음수 또는 크기 0 이하 (5/18) ==========")
print(f"전체 annotation: {len(codeit_bbox_validity_checks):,}개")
print(
    f"정상: "
    f"{codeit_bbox_validity_checks['status'].eq('정상').sum():,}개"
)
print(f"오류: {len(codeit_bbox_validity_errors):,}개")
print(f"오류 이미지: {codeit_bbox_validity_errors['image_id'].nunique():,}장")
print(f"계산 불가: {len(codeit_bbox_validity_unchecked):,}개")

if not codeit_bbox_validity_errors.empty:
    print("\n========== 오류 항목 ==========")
    display(codeit_bbox_validity_errors)

if not codeit_bbox_validity_unchecked.empty:
    print("\n========== 계산 불가 항목 ==========")
    display(codeit_bbox_validity_unchecked)

if codeit_bbox_validity_errors.empty and codeit_bbox_validity_unchecked.empty:
    print("\n[체크 가능] bbox 좌표 음수 또는 크기 0 이하 — 없음")

print("\n※ 파일 수정·삭제 없음")

### 2-6. 같은 이미지에서 동일 bbox 반복 (6/18)

In [ ]:
# 여러 JSON에 같은 annotation이 반복된 경우 비교 대상에서 한 번만 사용
codeit_unique_annotations = (
    codeit_annotations_df
    .drop_duplicates(subset=["id"])
    .copy()
)

codeit_same_bbox_rows = []

for image_id, group in codeit_unique_annotations.groupby("image_id"):
    records = group.to_dict("records")

    for index_a, annotation_a in enumerate(records):
        bbox_a = annotation_a.get("bbox")

        if not (
            isinstance(bbox_a, (list, tuple))
            and len(bbox_a) == 4
            and all(is_finite_number(value) for value in bbox_a)
        ):
            continue

        for annotation_b in records[index_a + 1:]:
            bbox_b = annotation_b.get("bbox")

            if not (
                isinstance(bbox_b, (list, tuple))
                and len(bbox_b) == 4
                and all(is_finite_number(value) for value in bbox_b)
            ):
                continue

            if tuple(bbox_a) == tuple(bbox_b):
                codeit_same_bbox_rows.append({
                    "image_id": image_id,
                    "annotation_id_a": annotation_a["id"],
                    "annotation_id_b": annotation_b["id"],
                    "category_id_a": annotation_a["category_id"],
                    "category_id_b": annotation_b["category_id"],
                    "bbox": bbox_a,
                    "same_class": (
                        annotation_a["category_id"]
                        == annotation_b["category_id"]
                    ),
                    "json_file_a": annotation_a["_json_file"],
                    "json_file_b": annotation_b["_json_file"],
                })

codeit_same_bbox_candidates = pd.DataFrame(
    codeit_same_bbox_rows,
    columns=[
        "image_id",
        "annotation_id_a", "annotation_id_b",
        "category_id_a", "category_id_b",
        "bbox", "same_class",
        "json_file_a", "json_file_b",
    ],
)

print("========== 2-6. 같은 이미지에서 동일 bbox 반복 (6/18) ==========")
print(
    f"검사 이미지: "
    f"{codeit_unique_annotations['image_id'].nunique():,}장"
)
print(f"검사 annotation: {len(codeit_unique_annotations):,}개")
print(f"동일 bbox 후보: {len(codeit_same_bbox_candidates):,}쌍")
print(
    f"후보가 있는 이미지: "
    f"{codeit_same_bbox_candidates['image_id'].nunique():,}장"
)

if codeit_same_bbox_candidates.empty:
    print("\n[체크 가능] 같은 이미지에서 동일 bbox 반복 — 없음")
else:
    print("\n========== 동일 bbox 후보 ==========")
    display(codeit_same_bbox_candidates)
    print("\n자동 검사 완료 — 후보 시각 검토 필요")

print("\n※ 기존 제외 결정 이미지를 포함한 전체 검사입니다.")
print("※ 파일 수정·삭제 없음")

In [ ]:
with zipfile.ZipFile(DATA_PATH, "r") as z:
    for review_number, candidate in enumerate(
        codeit_same_bbox_candidates.to_dict("records"),
        start=1,
    ):
        image_id = candidate["image_id"]

        image_info = codeit_images_df.loc[
            codeit_images_df["id"].eq(image_id)
        ].iloc[0]

        image_member = resolve_codeit_image_member(
            image_info["file_name"]
        )

        with Image.open(BytesIO(z.read(image_member))) as image:
            image_rgb = image.convert("RGB")

        image_annotations = codeit_unique_annotations.loc[
            codeit_unique_annotations["image_id"].eq(image_id)
        ]

        print(f"\n========== 동일 bbox 검토 {review_number} ==========")
        print(f"이미지 ID: {image_id}")
        print(f"파일명: {image_info['file_name']}")
        print(f"동일 bbox: {candidate['bbox']}")

        fig, axes = plt.subplots(1, 2, figsize=(12, 8))

        for ax, side, color in zip(
            axes, ["a", "b"], ["red", "blue"]
        ):
            ax.imshow(image_rgb)

            # 전체 annotation의 위치 표시
            for annotation in image_annotations.to_dict("records"):
                x, y, w, h = annotation["bbox"]

                ax.add_patch(Rectangle(
                    (x, y), w, h,
                    fill=False,
                    edgecolor="gray",
                    linewidth=1,
                ))
                ax.text(
                    x, y,
                    f"ID {annotation['id']}",
                    fontsize=8,
                    va="bottom",
                    bbox=dict(
                        facecolor="white",
                        alpha=0.8,
                        edgecolor="none",
                    ),
                )

            # 같은 위치에 있는 두 annotation을 화면별로 강조
            x, y, w, h = candidate["bbox"]

            ax.add_patch(Rectangle(
                (x, y), w, h,
                fill=False,
                edgecolor=color,
                linewidth=2.5,
            ))

            ax.set_title(
                f"Image {image_id} | {side.upper()}\n"
                f"ID {candidate[f'annotation_id_{side}']} | "
                f"Class {candidate[f'category_id_{side}']}",
                color=color,
            )
            ax.axis("off")

        plt.tight_layout()
        plt.show()

print("※ 파일 수정·삭제 없음")

### 2-6. 같은 이미지에서 동일 bbox 반복 — 검토 결과

- 동일 bbox 후보: 3쌍, 이미지 3장(ID: 208, 907, 1228).
- 시각 검토에서 한 알약에 두 annotation의 bbox가 겹쳐 있고, 다른 알약에는 bbox가 없는 현상을 확인함.
- annotation 중 하나의 bbox 위치가 잘못 지정된 것으로 판단함.
- 빠른 데이터셋 구성을 위해 수정 없이 해당 이미지와 연결된 모든 annotation을 제외하기로 결정함.
- 실제 제외는 최종 데이터셋 구성 시 적용함.

### 2-7. 서로 다른 bbox의 IoU가 매우 높음 (7/18)
- 점검표 기준인 IoU > 0.7로 검사
- 동일 bbox 쌍은 앞에서 검토했으므로 제외

In [ ]:
codeit_pair_rows = []

for image_id, group in codeit_unique_annotations.groupby("image_id"):
    records = group.to_dict("records")

    for index_a, annotation_a in enumerate(records):
        bbox_a = annotation_a["bbox"]

        if not (
            isinstance(bbox_a, (list, tuple))
            and len(bbox_a) == 4
            and all(is_finite_number(value) for value in bbox_a)
        ):
            continue

        xa, ya, wa, ha = bbox_a
        if wa <= 0 or ha <= 0:
            continue

        for annotation_b in records[index_a + 1:]:
            bbox_b = annotation_b["bbox"]

            if not (
                isinstance(bbox_b, (list, tuple))
                and len(bbox_b) == 4
                and all(is_finite_number(value) for value in bbox_b)
            ):
                continue

            xb, yb, wb, hb = bbox_b
            if wb <= 0 or hb <= 0:
                continue

            intersection_width = max(
                0, min(xa + wa, xb + wb) - max(xa, xb)
            )
            intersection_height = max(
                0, min(ya + ha, yb + hb) - max(ya, yb)
            )
            intersection_area = (
                intersection_width * intersection_height
            )

            area_a = wa * ha
            area_b = wb * hb
            union_area = area_a + area_b - intersection_area

            codeit_pair_rows.append({
                "image_id": image_id,
                "annotation_id_a": annotation_a["id"],
                "annotation_id_b": annotation_b["id"],
                "category_id_a": annotation_a["category_id"],
                "category_id_b": annotation_b["category_id"],
                "bbox_a": bbox_a,
                "bbox_b": bbox_b,
                "same_bbox": tuple(bbox_a) == tuple(bbox_b),
                "iou": intersection_area / union_area,
                "small_box_overlap": (
                    intersection_area / min(area_a, area_b)
                ),
                "json_file_a": annotation_a["_json_file"],
                "json_file_b": annotation_b["_json_file"],
            })

codeit_bbox_pairs = pd.DataFrame(
    codeit_pair_rows,
    columns=[
        "image_id",
        "annotation_id_a", "annotation_id_b",
        "category_id_a", "category_id_b",
        "bbox_a", "bbox_b",
        "same_bbox", "iou", "small_box_overlap",
        "json_file_a", "json_file_b",
    ],
)

codeit_iou_threshold = 0.7

codeit_high_iou_candidates = codeit_bbox_pairs.loc[
    codeit_bbox_pairs["same_bbox"].eq(False)
    & codeit_bbox_pairs["iou"].gt(codeit_iou_threshold)
].sort_values("iou", ascending=False).reset_index(drop=True)

print("========== 2-7. 서로 다른 bbox의 IoU가 매우 높음 (7/18) ==========")
print(f"비교한 박스 쌍: {len(codeit_bbox_pairs):,}쌍")
print(f"검사 기준: IoU > {codeit_iou_threshold}")
print(f"동일 bbox를 제외한 후보: {len(codeit_high_iou_candidates):,}쌍")
print(
    f"후보가 있는 이미지: "
    f"{codeit_high_iou_candidates['image_id'].nunique():,}장"
)

if codeit_high_iou_candidates.empty:
    print("\n[체크 가능] 서로 다른 bbox의 높은 IoU — 후보 없음")
else:
    print("\n========== 높은 IoU 후보 ==========")
    display(codeit_high_iou_candidates)
    print("\n자동 검사 완료 — 후보 시각 검토 필요")

print("\n※ 기존 제외 결정 이미지를 포함한 전체 검사입니다.")
print("※ 높은 IoU만으로 오류나 제외를 확정하지 않습니다.")
print("※ 파일 수정·삭제 없음")

### 2-8. 작은 박스 중첩 비율 검사 (8/18)
- 겹친 면적 ÷ 작은 박스 면적 ≥ 90% 인 쌍을 확인
- 앞에서 검사한 동일 bbox와 높은 IoU 쌍은 제외

In [ ]:
codeit_small_overlap_threshold = 0.9

codeit_small_overlap_candidates = codeit_bbox_pairs.loc[
    codeit_bbox_pairs["same_bbox"].eq(False)
    & codeit_bbox_pairs["iou"].le(codeit_iou_threshold)
    & codeit_bbox_pairs["small_box_overlap"].ge(
        codeit_small_overlap_threshold
    )
].sort_values(
    "small_box_overlap", ascending=False
).reset_index(drop=True)

print("========== 2-8. 작은 박스 중첩 비율 검사 (8/18) ==========")
print(f"검사 기준: 작은 박스 중첩 비율 ≥ {codeit_small_overlap_threshold:.0%}")
print("동일 bbox 및 앞 단계의 높은 IoU 쌍 제외")
print(f"후보: {len(codeit_small_overlap_candidates):,}쌍")
print(
    f"후보가 있는 이미지: "
    f"{codeit_small_overlap_candidates['image_id'].nunique():,}장"
)

if codeit_small_overlap_candidates.empty:
    print("\n[체크 가능] 작은 박스 중첩 비율 90% 이상 — 추가 후보 없음")
else:
    print("\n========== 작은 박스 중첩 후보 ==========")
    display(codeit_small_overlap_candidates)
    print("\n자동 검사 완료 — 후보 시각 검토 필요")

print("\n※ 기존 제외 결정 이미지를 포함한 전체 검사입니다.")
print("※ 중첩 비율만으로 오류나 제외를 확정하지 않습니다.")
print("※ 파일 수정·삭제 없음")


### 2-9. PNG 파일 누락 또는 손상 (9/18)

In [ ]:
# 알약별 JSON에 반복된 이미지 기록은 한 번만 검사
codeit_image_file_targets = (
    codeit_images_df[["id", "file_name"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

codeit_image_file_rows = []

with zipfile.ZipFile(DATA_PATH, "r") as z:
    for image_record in codeit_image_file_targets.to_dict("records"):
        result = {
            "image_id": image_record["id"],
            "file_name": image_record["file_name"],
            "zip_member": None,
            "actual_width": None,
            "actual_height": None,
            "status": None,
            "detail": "",
        }

        try:
            image_member = resolve_codeit_image_member(
                image_record["file_name"]
            )
            result["zip_member"] = image_member

        except Exception as error:
            matches = codeit_train_image_by_name.get(
                Path(image_record["file_name"]).name, []
            )
            result["status"] = (
                "파일 누락" if len(matches) == 0 else "경로 확정 불가"
            )
            result["detail"] = str(error)
            codeit_image_file_rows.append(result)
            continue

        try:
            image_bytes = z.read(image_member)

            # 파일 구조 검사
            with Image.open(BytesIO(image_bytes)) as image:
                image.verify()

            # 다시 열어 실제 픽셀까지 읽기
            with Image.open(BytesIO(image_bytes)) as image:
                image.load()
                result["actual_width"], result["actual_height"] = (
                    image.size
                )

            result["status"] = "정상"

        except Exception as error:
            result["status"] = "읽기 실패"
            result["detail"] = str(error)

        codeit_image_file_rows.append(result)

codeit_image_file_checks = pd.DataFrame(codeit_image_file_rows)

codeit_image_file_issues = codeit_image_file_checks.loc[
    codeit_image_file_checks["status"].ne("정상")
].copy()

print("========== 2-9. PNG 파일 누락 또는 손상 (9/18) ==========")
print(f"검사 이미지: {len(codeit_image_file_checks):,}장")

for status in ["정상", "파일 누락", "경로 확정 불가", "읽기 실패"]:
    count = codeit_image_file_checks["status"].eq(status).sum()
    print(f"{status}: {count:,}장")

if codeit_image_file_issues.empty:
    print("\n[체크 가능] PNG 파일 누락 또는 손상 — 없음")
else:
    print("\n========== 확인 필요 항목 ==========")
    display(codeit_image_file_issues)

print("\n※ Train JSON에서 참조하는 이미지만 검사했습니다.")
print("※ 파일 수정·압축 해제·삭제 없음")

### 2-10. JSON과 실제 PNG 크기 불일치 (10/18)

In [ ]:
# 앞 단계에서 확인한 실제 이미지 크기를 재사용
codeit_image_size_checks = (
    codeit_images_df[
        ["_json_file", "id", "file_name", "width", "height"]
    ]
    .rename(columns={
        "_json_file": "json_file",
        "id": "image_id",
        "width": "json_width",
        "height": "json_height",
    })
    .merge(
        codeit_image_file_checks[[
            "image_id", "file_name",
            "actual_width", "actual_height", "status",
        ]].rename(columns={"status": "file_status"}),
        on=["image_id", "file_name"],
        how="left",
        validate="many_to_one",
    )
)


def check_codeit_image_size(row):
    if row["file_status"] != "정상":
        return "비교 불가"

    if not (
        is_finite_number(row["json_width"])
        and is_finite_number(row["json_height"])
        and row["json_width"] > 0
        and row["json_height"] > 0
    ):
        return "JSON 크기 오류"

    if (
        row["json_width"] == row["actual_width"]
        and row["json_height"] == row["actual_height"]
    ):
        return "일치"

    return "불일치"


codeit_image_size_checks["status"] = codeit_image_size_checks.apply(
    check_codeit_image_size,
    axis=1,
)

codeit_image_size_issues = codeit_image_size_checks.loc[
    codeit_image_size_checks["status"].ne("일치")
].copy()

print("========== 2-10. JSON과 실제 PNG 크기 불일치 (10/18) ==========")
print(f"검사 JSON 이미지 기록: {len(codeit_image_size_checks):,}개")
print(
    f"고유 이미지: "
    f"{codeit_image_size_checks['image_id'].nunique():,}장"
)

for status in ["일치", "불일치", "JSON 크기 오류", "비교 불가"]:
    count = codeit_image_size_checks["status"].eq(status).sum()
    print(f"{status}: {count:,}개")

if codeit_image_size_issues.empty:
    print("\n[체크 가능] JSON과 실제 PNG 크기 불일치 — 없음")
else:
    print("\n========== 확인 필요 항목 ==========")
    display(codeit_image_size_issues)

print("\n※ 알약별 JSON의 이미지 크기 기록을 모두 비교했습니다.")
print("※ 이미지 파일을 다시 읽지 않았습니다.")
print("※ 파일 수정·삭제 없음")

### 2-11. 중복 또는 유사 이미지 (11/18)
- 완전 중복: 이미지 크기와 RGB 픽셀이 같음.
- 유사 후보: pHash 해밍 거리 5 이하, 서로 다른 알약 조합만 비교.
- 파일명에서 조합을 확인할 수 없으면 별도로 표시합니다.

In [ ]:
import hashlib
import re
import numpy as np

# pHash 계산용 변환 행렬: 한 번만 생성
_hash_positions = np.arange(32)
_hash_frequencies = np.arange(8)[:, None]
_codeit_dct_matrix = np.cos(
    np.pi * (2 * _hash_positions + 1) * _hash_frequencies / 64
)
_codeit_dct_matrix[0] *= 1 / np.sqrt(2)


def calculate_codeit_phash(rgb):
    gray = rgb.convert("L").resize(
        (32, 32), Image.Resampling.LANCZOS
    )
    pixels = np.asarray(gray, dtype=np.float64)

    coefficients = (
        _codeit_dct_matrix @ pixels @ _codeit_dct_matrix.T
    )
    bits = (coefficients > np.median(coefficients)).ravel()

    return int("".join("1" if bit else "0" for bit in bits), 2)


def get_codeit_combination(image_id):
    records = codeit_images_df.loc[
        codeit_images_df["id"].eq(image_id)
    ].to_dict("records")

    combinations = set()

    for record in records:
        for field in ["imgfile", "file_name"]:
            value = record.get(field)

            if not isinstance(value, str):
                continue

            match = re.search(r"K-\d{6}(?:-\d{6})*", value)

            if match:
                combinations.add(tuple(sorted(
                    int(code)
                    for code in match.group().split("-")[1:]
                )))

    return next(iter(combinations)) if len(combinations) == 1 else None


codeit_hash_rows = []

with zipfile.ZipFile(DATA_PATH, "r") as z:
    for record in codeit_image_file_checks.to_dict("records"):
        result = {
            "image_id": record["image_id"],
            "file_name": record["file_name"],
            "combination": get_codeit_combination(record["image_id"]),
            "pixel_md5": None,
            "phash": None,
            "hash_error": "",
        }

        if record["status"] != "정상":
            result["hash_error"] = "앞 단계 이미지 읽기 미통과"
        else:
            try:
                with Image.open(
                    BytesIO(z.read(record["zip_member"]))
                ) as image:
                    rgb = image.convert("RGB")

                    digest = hashlib.md5()
                    digest.update(
                        f"RGB:{rgb.width}:{rgb.height}:".encode()
                    )
                    digest.update(rgb.tobytes())

                    result["pixel_md5"] = digest.hexdigest()
                    result["phash"] = calculate_codeit_phash(rgb)

            except Exception as error:
                result["hash_error"] = str(error)

        codeit_hash_rows.append(result)

# 64비트 해시의 정밀도를 유지하도록 object 자료형 사용
codeit_image_hashes = pd.DataFrame(
    codeit_hash_rows, dtype=object
)

codeit_hash_success = codeit_image_hashes.loc[
    codeit_image_hashes["hash_error"].eq("")
].copy()

codeit_hash_failures = codeit_image_hashes.loc[
    codeit_image_hashes["hash_error"].ne("")
].copy()

# 완전 중복은 같은 조합까지 포함
codeit_duplicate_images = codeit_hash_success.loc[
    codeit_hash_success.duplicated("pixel_md5", keep=False)
].sort_values("pixel_md5").copy()

codeit_unknown_combinations = codeit_hash_success.loc[
    codeit_hash_success["combination"].isna()
].copy()

similar_records = codeit_hash_success.loc[
    codeit_hash_success["combination"].notna()
].to_dict("records")

codeit_similar_rows = []
codeit_similar_comparison_count = 0

for index_a, record_a in enumerate(similar_records):
    for record_b in similar_records[index_a + 1:]:
        if record_a["combination"] == record_b["combination"]:
            continue

        codeit_similar_comparison_count += 1

        distance = (
            int(record_a["phash"]) ^ int(record_b["phash"])
        ).bit_count()

        if distance <= 5:
            codeit_similar_rows.append({
                "image_id_a": record_a["image_id"],
                "file_name_a": record_a["file_name"],
                "combination_a": record_a["combination"],
                "image_id_b": record_b["image_id"],
                "file_name_b": record_b["file_name"],
                "combination_b": record_b["combination"],
                "phash_distance": distance,
                "same_pixels": (
                    record_a["pixel_md5"] == record_b["pixel_md5"]
                ),
            })

codeit_similar_candidates = pd.DataFrame(
    codeit_similar_rows,
    columns=[
        "image_id_a", "file_name_a", "combination_a",
        "image_id_b", "file_name_b", "combination_b",
        "phash_distance", "same_pixels",
    ],
).sort_values("phash_distance").reset_index(drop=True)

print("========== 2-11. 중복 또는 유사 이미지 (11/18) ==========")
print(f"검사 이미지: {len(codeit_image_hashes):,}장")
print(f"해시 계산 성공: {len(codeit_hash_success):,}장")
print(f"해시 계산 실패: {len(codeit_hash_failures):,}장")

print("\n[완전 중복 — 같은 조합 포함]")
print(
    f"완전 중복 그룹: "
    f"{codeit_duplicate_images['pixel_md5'].nunique():,}개"
)
print(f"중복 그룹에 포함된 이미지: {len(codeit_duplicate_images):,}장")

if not codeit_duplicate_images.empty:
    display(codeit_duplicate_images[[
        "image_id", "file_name", "combination", "pixel_md5"
    ]])

print("\n[유사 이미지 — 서로 다른 조합만]")
print("pHash 해밍 거리 기준: 5 이하")
print(f"비교한 쌍: {codeit_similar_comparison_count:,}쌍")
print(f"유사 후보: {len(codeit_similar_candidates):,}쌍")
print(f"조합 확인 불가: {len(codeit_unknown_combinations):,}장")

if not codeit_similar_candidates.empty:
    print("\n========== 유사 후보: 최대 20쌍 ==========")
    display(codeit_similar_candidates.head(20))

if not codeit_unknown_combinations.empty:
    print("\n========== 조합 확인 불가 ==========")
    display(codeit_unknown_combinations[["image_id", "file_name"]])

if not codeit_hash_failures.empty:
    print("\n========== 해시 계산 실패 ==========")
    display(codeit_hash_failures[[
        "image_id", "file_name", "hash_error"
    ]])

print("\n※ pHash가 같아도 실제 픽셀이 같다는 뜻은 아닙니다.")
print("※ 유사 후보는 시각 검토 후 판단합니다.")
print("※ 기존 제외 결정 이미지를 포함한 Train 검사입니다.")
print("※ 파일 수정·삭제 없음")

In [ ]:
with zipfile.ZipFile(DATA_PATH, "r") as z:
    for start in range(0, len(codeit_similar_candidates), 4):
        batch = codeit_similar_candidates.iloc[start:start + 4]

        fig, axes = plt.subplots(
            len(batch), 2,
            figsize=(12, 6 * len(batch)),
            squeeze=False,
        )

        for row_index, (candidate_index, candidate) in enumerate(
            batch.iterrows()
        ):
            for column_index, side in enumerate(["a", "b"]):
                ax = axes[row_index, column_index]
                file_name = candidate[f"file_name_{side}"]
                image_member = resolve_codeit_image_member(file_name)

                with Image.open(
                    BytesIO(z.read(image_member))
                ) as image:
                    ax.imshow(image.convert("RGB"))

                combination = " / ".join(
                    f"K-{code:06d}"
                    for code in candidate[f"combination_{side}"]
                )

                ax.set_title(
                    f"Pair {candidate_index + 1} | "
                    f"Image {candidate[f'image_id_{side}']} | "
                    f"Distance {candidate['phash_distance']}\n"
                    f"{combination}",
                    fontsize=10,
                )
                ax.axis("off")

        plt.tight_layout()
        plt.show()

print("유사 후보 16쌍 표시 완료 — 파일 수정·삭제 없음")

### 2-11. 중복 또는 유사 이미지 — 검토 결과

- Train 이미지 232장 모두 해시 계산 성공.
- 완전 중복 이미지: 없음.
- 서로 다른 조합의 pHash 해밍 거리 5 이하 후보: 16쌍.
- 후보 16쌍 전체를 시각 검토한 결과, 알약 일부가 다르거나 각인 방향·앞뒷면·배치에 차이가 있음을 확인함.
- 유사한 구도의 별개 이미지로 판단하여 이번 검사에서는 추가 제외하지 않음.
- 다른 검사에서 확인된 오류에 따른 제외 결정은 유지함.

### 2-12. ignore / iscrowd 값 이상 (12/18)

In [ ]:
codeit_flag_rows = []

for annotation in codeit_table_rows["annotations"]:
    for field in ["ignore", "iscrowd"]:
        value = annotation.get(field)

        if field not in annotation:
            status = "필드 없음"
        elif not is_finite_number(value):
            status = "값 확인 필요"
        elif value != 0:
            status = "0이 아닌 값"
        else:
            status = "정상"

        codeit_flag_rows.append({
            "json_file": annotation["_json_file"],
            "annotation_id": annotation.get("id"),
            "image_id": annotation.get("image_id"),
            "field": field,
            "value": value,
            "status": status,
        })

codeit_flag_checks = pd.DataFrame(codeit_flag_rows)

codeit_flag_issues = codeit_flag_checks.loc[
    codeit_flag_checks["status"].ne("정상")
].copy()

print("========== 2-12. ignore / iscrowd 값 이상 (12/18) ==========")
print(f"검사 annotation: {len(codeit_table_rows['annotations']):,}개")

display(
    pd.crosstab(
        codeit_flag_checks["field"],
        codeit_flag_checks["status"],
    ).reindex(
        index=["ignore", "iscrowd"],
        columns=["정상", "0이 아닌 값", "필드 없음", "값 확인 필요"],
        fill_value=0,
    )
)

if codeit_flag_issues.empty:
    print("\n[체크 가능] ignore / iscrowd — 모두 숫자 0")
else:
    print("\n========== 확인 필요 항목 ==========")
    display(codeit_flag_issues)

print("\n※ 파일 수정·삭제 없음")

### 2-13. category_id와 dl_idx 매핑 불일치 (13/18)

In [ ]:
def codeit_integer_value(value):
    """숫자 또는 숫자 문자열을 정수로 해석. 유효하지 않으면 None."""
    if isinstance(value, bool) or value is None:
        return None

    try:
        number = pd.to_numeric(value, errors="raise")
        if is_finite_number(number) and float(number).is_integer():
            return int(number)
    except (TypeError, ValueError, OverflowError):
        pass

    return None


codeit_mapping_rows = []

for annotation in codeit_table_rows["annotations"]:
    image_info = codeit_image_size_lookup.get(
        (annotation["_json_file"], annotation.get("image_id"))
    )

    category_value = annotation.get("category_id")
    dl_idx_value = (
        image_info.get("dl_idx") if image_info is not None else None
    )
    drug_code = (
        image_info.get("drug_N") if image_info is not None else None
    )

    category_number = codeit_integer_value(category_value)
    dl_idx_number = codeit_integer_value(dl_idx_value)

    code_match = (
        re.fullmatch(r"K-(\d{6})", drug_code)
        if isinstance(drug_code, str)
        else None
    )
    code_number = int(code_match.group(1)) if code_match else None

    reasons = []

    if image_info is None:
        reasons.append("연결 이미지 정보 없음")
    else:
        if category_number is None:
            reasons.append("category_id 정수 확인 불가")
        if dl_idx_number is None:
            reasons.append("dl_idx 정수 확인 불가")
        if code_number is None:
            reasons.append("drug_N 형식 확인 불가")

        if category_number is not None and dl_idx_number is not None:
            if category_number != dl_idx_number:
                reasons.append("category_id와 dl_idx 불일치")

        if code_number is not None and dl_idx_number is not None:
            if code_number != dl_idx_number:
                reasons.append("drug_N 코드와 dl_idx 불일치")

    codeit_mapping_rows.append({
        "json_file": annotation["_json_file"],
        "annotation_id": annotation.get("id"),
        "image_id": annotation.get("image_id"),
        "category_id": category_value,
        "dl_idx": dl_idx_value,
        "drug_N": drug_code,
        "status": "확인 필요" if reasons else "일치",
        "reason": " / ".join(reasons),
    })

codeit_mapping_checks = pd.DataFrame(codeit_mapping_rows)

codeit_mapping_issues = codeit_mapping_checks.loc[
    codeit_mapping_checks["status"].ne("일치")
].copy()

print("========== 2-13. category_id와 dl_idx 매핑 불일치 (13/18) ==========")
print(f"검사 annotation: {len(codeit_mapping_checks):,}개")
print(
    f"매핑 일치: "
    f"{codeit_mapping_checks['status'].eq('일치').sum():,}개"
)
print(f"확인 필요: {len(codeit_mapping_issues):,}개")

if codeit_mapping_issues.empty:
    print("\n[체크 가능] category_id·dl_idx·drug_N 매핑 — 모두 일치")
else:
    print("\n========== 확인 필요: 최대 20건 ==========")
    display(codeit_mapping_issues.head(20))
    print("\n※ 일정한 번호 차이가 반복되면 원본 매핑 규칙부터 확인합니다.")

print("\n※ 자료형 해석만 수행했으며 원본 값은 변경하지 않았습니다.")
print("※ 파일 수정·삭제 없음")

### 2-14. 파일 이름의 코드와 Annotation category_id 구성 불일치 (14/18)

In [ ]:
codeit_composition_rows = []

for image_id, group in codeit_unique_annotations.groupby("image_id"):
    combination = get_codeit_combination(image_id)

    category_values = [
        codeit_integer_value(value)
        for value in group["category_id"]
    ]

    annotation_codes = {
        value for value in category_values if value is not None
    }

    file_codes = set(combination) if combination is not None else None
    missing_codes = []
    extra_codes = []

    if file_codes is None:
        status = "파일명 코드 확인 불가"
    elif any(value is None for value in category_values):
        status = "category_id 확인 불가"
    else:
        missing_codes = sorted(file_codes - annotation_codes)
        extra_codes = sorted(annotation_codes - file_codes)
        status = (
            "불일치" if missing_codes or extra_codes else "일치"
        )

    codeit_composition_rows.append({
        "image_id": image_id,
        "file_name": codeit_images_df.loc[
            codeit_images_df["id"].eq(image_id), "file_name"
        ].iloc[0],
        "file_codes": (
            sorted(file_codes) if file_codes is not None else None
        ),
        "annotation_codes": sorted(annotation_codes),
        "missing_codes": missing_codes,
        "extra_codes": extra_codes,
        "status": status,
    })

codeit_composition_checks = pd.DataFrame(codeit_composition_rows)

codeit_composition_issues = codeit_composition_checks.loc[
    codeit_composition_checks["status"].ne("일치")
].copy()

print(
    "========== 2-14. 파일 이름과 Annotation 클래스 구성 검사 "
    "(14/18) =========="
)
print(f"검사 이미지: {len(codeit_composition_checks):,}장")

for status in [
    "일치", "불일치",
    "파일명 코드 확인 불가", "category_id 확인 불가",
]:
    count = codeit_composition_checks["status"].eq(status).sum()
    print(f"{status}: {count:,}장")

if codeit_composition_issues.empty:
    print("\n[체크 가능] 파일 이름과 Annotation 클래스 구성 — 모두 일치")
else:
    print("\n========== 확인 필요 이미지 ==========")
    display(codeit_composition_issues)
    print("\nmissing_codes: 파일명에는 있지만 annotation에 없는 코드")
    print("extra_codes: annotation에는 있지만 파일명에 없는 코드")

print("\n※ 클래스 구성 검사이며 실제 알약의 종류는 시각 검토로 확인합니다.")
print("※ 파일 수정·삭제 없음")

In [ ]:
review_records = codeit_composition_issues.to_dict("records")

with zipfile.ZipFile(DATA_PATH, "r") as z:
    for start in range(0, len(review_records), 4):
        batch = review_records[start:start + 4]

        fig, axes = plt.subplots(
            2, 2, figsize=(14, 16), squeeze=False
        )

        for ax, candidate in zip(axes.ravel(), batch):
            image_id = candidate["image_id"]
            image_member = resolve_codeit_image_member(
                candidate["file_name"]
            )

            with Image.open(BytesIO(z.read(image_member))) as image:
                ax.imshow(image.convert("RGB"))

            annotations = codeit_unique_annotations.loc[
                codeit_unique_annotations["image_id"].eq(image_id)
            ]

            for annotation in annotations.to_dict("records"):
                x, y, w, h = annotation["bbox"]

                ax.add_patch(Rectangle(
                    (x, y), w, h,
                    fill=False,
                    edgecolor="red",
                    linewidth=2,
                ))
                ax.text(
                    x, y,
                    f"ID {annotation['id']} | "
                    f"Class {annotation['category_id']}",
                    fontsize=8,
                    va="bottom",
                    bbox=dict(
                        facecolor="white",
                        alpha=0.8,
                        edgecolor="none",
                    ),
                )

            missing = ", ".join(
                f"K-{code:06d}" for code in candidate["missing_codes"]
            )

            ax.set_title(
                f"Image {image_id} | Missing: {missing}\n"
                f"Annotations: {len(annotations)}"
            )
            ax.axis("off")

        for ax in axes.ravel()[len(batch):]:
            ax.axis("off")

        plt.tight_layout()
        plt.show()

print("※ Missing은 파일명 기준 누락 코드이며, 시각적으로 확정한 클래스가 아닙니다.")
print("※ 파일 수정·삭제 없음")

### 2-14. 파일 이름과 Annotation 클래스 구성 불일치 — 검토 결과

- 검사 이미지 232장 중 일치 224장, 불일치 8장.
- 불일치 이미지 모두 파일명에 있는 클래스 하나가 annotation에서 누락됨.
- 8장 전체의 시각 검토에서 실제 알약은 있으나 해당 박스가 없는 것을 확인함.
- 제외 대상 이미지 ID: 16, 193, 783, 1258, 1267, 1383, 1405, 1432.
- 빠른 데이터셋 구성을 위해 수정 없이 해당 이미지와 연결된 모든 annotation을 제외하기로 결정함.
- 실제 제외는 최종 데이터셋 구성 시 적용하며, 다른 제외 사유와 겹치는 이미지는 한 번만 집계함.

### 2-15. 동일 클래스에 여러 제품명·상충하는 외형 정보 (15/18)
- 같은 클래스에서 dl_name, drug_shape, color_class1, print_front 값이 서로 다른지 확인

In [ ]:
codeit_metadata_fields = [
    "dl_name", "drug_shape", "color_class1", "print_front"
]

# 같은 JSON의 annotation과 이미지 메타데이터 연결
codeit_metadata_rows = []

for annotation in codeit_table_rows["annotations"]:
    image_info = codeit_image_size_lookup.get(
        (annotation["_json_file"], annotation.get("image_id"))
    )

    record = {
        "json_file": annotation["_json_file"],
        "annotation_id": annotation.get("id"),
        "image_id": annotation.get("image_id"),
        "category_id": codeit_integer_value(
            annotation.get("category_id")
        ),
    }

    for field in codeit_metadata_fields:
        value = image_info.get(field) if image_info is not None else None

        if value is None or (
            not isinstance(value, (list, dict)) and pd.isna(value)
        ):
            record[field] = None
        else:
            text = str(value).strip()
            record[field] = text if text else None

    codeit_metadata_rows.append(record)

codeit_metadata_records = pd.DataFrame(codeit_metadata_rows)

codeit_metadata_summary_rows = []

for category_id, group in codeit_metadata_records.groupby("category_id"):
    for field in codeit_metadata_fields:
        values = sorted(group[field].dropna().unique().tolist())

        codeit_metadata_summary_rows.append({
            "category_id": category_id,
            "drug_code": f"K-{int(category_id):06d}",
            "field": field,
            "value_count": len(values),
            "values": values,
            "missing_count": int(group[field].isna().sum()),
            "annotation_count": len(group),
        })

codeit_metadata_summary = pd.DataFrame(codeit_metadata_summary_rows)

codeit_metadata_conflicts = codeit_metadata_summary.loc[
    codeit_metadata_summary["value_count"].gt(1)
].copy()

codeit_metadata_missing = codeit_metadata_summary.loc[
    codeit_metadata_summary["missing_count"].gt(0)
].copy()

print(
    "========== 2-15. 동일 클래스의 제품명·외형 정보 검사 "
    "(15/18) =========="
)
print(
    f"검사 클래스: "
    f"{codeit_metadata_records['category_id'].nunique():,}종"
)
print(f"검사 필드: {', '.join(codeit_metadata_fields)}")
print(f"서로 다른 값이 있는 클래스·필드: {len(codeit_metadata_conflicts):,}건")
print(
    f"해당 클래스: "
    f"{codeit_metadata_conflicts['category_id'].nunique():,}종"
)

if codeit_metadata_conflicts.empty:
    print("\n[체크 가능] 동일 클래스의 제품명·외형 정보 — 값 충돌 없음")
else:
    print("\n========== 서로 다른 값이 있는 항목 ==========")
    display(codeit_metadata_conflicts)

if not codeit_metadata_missing.empty:
    print("\n========== 빈 값이 있는 항목 ==========")
    display(codeit_metadata_missing[[
        "drug_code", "field", "missing_count", "annotation_count"
    ]])

print("\n※ 빈 값은 값 충돌과 구분합니다.")
print("※ 서로 다른 값이 발견되어도 표기 차이인지 확인 후 판단합니다.")
print("※ 파일 수정·삭제 없음")

### 2-16. 실제 알약 개수와 Annotation 개수 불일치 (16/18)


In [ ]:
# 앞 단계에서 확인한 파일명 조합을 재사용
codeit_count_checks = codeit_composition_checks[[
    "image_id", "file_name", "file_codes"
]].copy()

codeit_annotation_counts = (
    codeit_unique_annotations.groupby("image_id").size()
)

codeit_count_checks["expected_count"] = (
    codeit_count_checks["file_codes"].apply(
        lambda codes: len(codes) if codes is not None else None
    )
)
codeit_count_checks["annotation_count"] = (
    codeit_count_checks["image_id"]
    .map(codeit_annotation_counts)
    .fillna(0)
    .astype(int)
)
codeit_count_checks["combination"] = (
    codeit_count_checks["file_codes"].apply(
        lambda codes: tuple(codes) if codes is not None else None
    )
)
codeit_count_checks["status"] = codeit_count_checks.apply(
    lambda row: (
        "기대 개수 확인 불가"
        if row["expected_count"] is None
        or pd.isna(row["expected_count"])
        else (
            "일치"
            if row["expected_count"] == row["annotation_count"]
            else "불일치"
        )
    ),
    axis=1,
)

codeit_count_issues = codeit_count_checks.loc[
    codeit_count_checks["status"].ne("일치")
].copy()

# 같은 조합에서 이미지별 annotation 수가 다른지 검사
codeit_combination_count_summary = (
    codeit_count_checks.loc[
        codeit_count_checks["combination"].notna()
    ]
    .groupby("combination")
    .agg(
        image_count=("image_id", "nunique"),
        annotation_count_variants=("annotation_count", "nunique"),
        annotation_counts=(
            "annotation_count",
            lambda values: sorted(set(values)),
        ),
    )
    .reset_index()
)

codeit_combination_count_issues = (
    codeit_combination_count_summary.loc[
        codeit_combination_count_summary[
            "annotation_count_variants"
        ].gt(1)
    ].copy()
)

print(
    "========== 2-16. 알약 기대 개수와 Annotation 개수 검사 "
    "(16/18) =========="
)
print(f"검사 이미지: {len(codeit_count_checks):,}장")

for status in ["일치", "불일치", "기대 개수 확인 불가"]:
    count = codeit_count_checks["status"].eq(status).sum()
    print(f"{status}: {count:,}장")

if not codeit_count_issues.empty:
    print("\n========== 기대 개수와 다른 이미지 ==========")
    display(codeit_count_issues[[
        "image_id", "file_name",
        "expected_count", "annotation_count", "status",
    ]])

print(
    f"\n같은 조합에서 Annotation 수가 다른 경우: "
    f"{len(codeit_combination_count_issues):,}개 조합"
)

if not codeit_combination_count_issues.empty:
    display(codeit_combination_count_issues)

# 이미 시각 검토하여 제외 결정한 라벨 누락 이미지와 구분
codeit_reviewed_missing_ids = set(
    codeit_composition_issues["image_id"]
)

codeit_new_count_issues = codeit_count_issues.loc[
    ~codeit_count_issues["image_id"].isin(
        codeit_reviewed_missing_ids
    )
].copy()

print(
    f"\n2-14에서 검토한 이미지 외 추가 개수 확인 대상: "
    f"{len(codeit_new_count_issues):,}장"
)

if not codeit_new_count_issues.empty:
    display(codeit_new_count_issues)

if (
    codeit_count_issues.empty
    and codeit_combination_count_issues.empty
):
    print("\n[체크 가능] 기대 개수 및 같은 조합의 Annotation 수 — 이상 없음")

print("\n※ 같은 이미지가 여러 검사에 해당해도 최종 제외는 한 번만 집계합니다.")
print("※ 파일 수정·삭제 없음")

### 2-17. 실물 길이 대비 bbox 크기 검사 (17/18)
- bbox의 긴 변 ÷ 알약의 실물 길이(mm)를 계산하여 18 미만 또는 33 초과인 항목을 찾음

In [ ]:
codeit_size_ratio_rows = []

for annotation in codeit_table_rows["annotations"]:
    image_info = codeit_image_size_lookup.get(
        (annotation["_json_file"], annotation.get("image_id"))
    )

    raw_length = (
        image_info.get("leng_long")
        if image_info is not None
        else None
    )

    try:
        length_mm = (
            float(raw_length)
            if not isinstance(raw_length, bool)
            else None
        )
    except (TypeError, ValueError, OverflowError):
        length_mm = None

    bbox = annotation.get("bbox")
    ratio = None

    valid_bbox = (
        isinstance(bbox, (list, tuple))
        and len(bbox) == 4
        and all(is_finite_number(value) for value in bbox)
        and bbox[2] > 0
        and bbox[3] > 0
    )

    if (
        length_mm is None
        or not math.isfinite(length_mm)
        or length_mm <= 0
    ):
        status = "유효한 길이 없음"
    elif not valid_bbox:
        status = "bbox 계산 불가"
    else:
        ratio = max(bbox[2], bbox[3]) / length_mm
        status = (
            "크기 의심"
            if ratio < 18 or ratio > 33
            else "참고 범위"
        )

    codeit_size_ratio_rows.append({
        "json_file": annotation["_json_file"],
        "annotation_id": annotation.get("id"),
        "image_id": annotation.get("image_id"),
        "category_id": annotation.get("category_id"),
        "bbox": bbox,
        "original_length": raw_length,
        "length_mm": length_mm,
        "pixels_per_mm": ratio,
        "status": status,
    })

codeit_size_ratio_checks = pd.DataFrame(codeit_size_ratio_rows)

codeit_size_ratio_candidates = codeit_size_ratio_checks.loc[
    codeit_size_ratio_checks["status"].eq("크기 의심")
].sort_values("pixels_per_mm").copy()

codeit_size_ratio_unchecked = codeit_size_ratio_checks.loc[
    ~codeit_size_ratio_checks["status"].isin(
        ["참고 범위", "크기 의심"]
    )
].copy()

print("========== 2-17. 실물 길이 대비 bbox 크기 검사 (17/18) ==========")
print(f"전체 annotation: {len(codeit_size_ratio_checks):,}개")
print(
    f"비율 계산 가능: "
    f"{codeit_size_ratio_checks['pixels_per_mm'].notna().sum():,}개"
)
print(f"계산 불가: {len(codeit_size_ratio_unchecked):,}개")
print(f"크기 의심 후보: {len(codeit_size_ratio_candidates):,}개")
print(
    f"후보 이미지: "
    f"{codeit_size_ratio_candidates['image_id'].nunique():,}장"
)

if not codeit_size_ratio_candidates.empty:
    print("\n========== 크기 의심 후보 ==========")
    display(codeit_size_ratio_candidates)

if not codeit_size_ratio_unchecked.empty:
    print("\n========== 계산 불가 항목 ==========")
    display(codeit_size_ratio_unchecked)

if (
    codeit_size_ratio_candidates.empty
    and codeit_size_ratio_unchecked.empty
):
    print("\n[체크 가능] 실물 길이 대비 bbox 크기 — 참고 범위 밖 후보 없음")

print("\n※ 비율은 bbox의 긴 변을 실물 길이(mm)로 나눈 값입니다.")
print("※ 18~33은 참고 기준이며 자동 제외 기준이 아닙니다.")
print("※ 길이 정보가 없다는 이유만으로 제외하지 않습니다.")
print("※ 박스 위치와 클래스의 정확성은 이 비율로 판단할 수 없습니다.")
print("※ 파일 수정·삭제 없음")

### 2-17. 실물 길이 대비 bbox 크기 — 검사 결과

- 전체 annotation 763개 중 비율 계산 가능 745개, 계산 불가 18개.
- 참고 범위(18~33px/mm) 밖 후보는 1개이며, 이미지 1360의 annotation 5178임.
- 해당 이미지는 2-3에서 이미 오류를 확인하고 제외하기로 결정하여 추가 검토하지 않음.
- 계산 불가 18개는 모두 K-016548의 실물 길이가 0으로 기록된 경우임.
- 실물 길이 정보 부족은 bbox 오류와 구분하며, 이 사유만으로 제외하지 않음.
- 이번 검사에 따른 추가 제외 이미지 없음.

### 2-18. 클래스별 bbox 크기·종횡비 이상치 (18/18)
- 클래스별 면적과 종횡비가 Q1 − 1.5×IQR ~ Q3 + 1.5×IQR을 벗어나는지 확인

In [ ]:
# 앞 단계에서 bbox 형식과 양수 크기를 확인한 annotation 사용
codeit_bbox_stats = codeit_unique_annotations[[
    "_json_file", "id", "image_id", "category_id", "bbox"
]].rename(columns={
    "_json_file": "json_file",
    "id": "annotation_id",
}).copy()

codeit_bbox_stats["bbox_area"] = codeit_bbox_stats["bbox"].apply(
    lambda bbox: bbox[2] * bbox[3]
)
codeit_bbox_stats["aspect_ratio"] = codeit_bbox_stats["bbox"].apply(
    lambda bbox: bbox[2] / bbox[3]
)

# 클래스별 사분위수와 이상치 경계 계산
codeit_class_bbox_rows = []

for category_id, group in codeit_bbox_stats.groupby("category_id"):
    result = {
        "category_id": category_id,
        "annotation_count": len(group),
        "analysis_available": len(group) >= 4,
    }

    for field in ["bbox_area", "aspect_ratio"]:
        q1 = group[field].quantile(0.25)
        q3 = group[field].quantile(0.75)
        iqr = q3 - q1

        result[f"{field}_lower"] = q1 - 1.5 * iqr
        result[f"{field}_upper"] = q3 + 1.5 * iqr

    codeit_class_bbox_rows.append(result)

codeit_class_bbox_summary = pd.DataFrame(codeit_class_bbox_rows)

codeit_bbox_stats = codeit_bbox_stats.merge(
    codeit_class_bbox_summary,
    on="category_id",
    how="left",
    validate="many_to_one",
)

# 표본이 4개 이상인 클래스만 이상치 후보 판정
for field in ["bbox_area", "aspect_ratio"]:
    codeit_bbox_stats[f"{field}_outlier"] = (
        codeit_bbox_stats["analysis_available"]
        & (
            codeit_bbox_stats[field].lt(
                codeit_bbox_stats[f"{field}_lower"]
            )
            | codeit_bbox_stats[field].gt(
                codeit_bbox_stats[f"{field}_upper"]
            )
        )
    )

codeit_bbox_stats["any_outlier"] = (
    codeit_bbox_stats["bbox_area_outlier"]
    | codeit_bbox_stats["aspect_ratio_outlier"]
)

codeit_bbox_outliers = codeit_bbox_stats.loc[
    codeit_bbox_stats["any_outlier"]
].copy()

# 시각 검토 등을 거쳐 이미 제외하기로 결정한 이미지
codeit_existing_exclusion_ids = (
    set(codeit_area_mismatches["image_id"])
    | set(codeit_boundary_errors["image_id"])
    | set(codeit_same_bbox_candidates["image_id"])
    | set(codeit_composition_issues["image_id"])
)

codeit_new_bbox_outliers = codeit_bbox_outliers.loc[
    ~codeit_bbox_outliers["image_id"].isin(
        codeit_existing_exclusion_ids
    )
].copy()

# 클래스별 결과 요약
codeit_bbox_outlier_summary = (
    codeit_bbox_stats.groupby("category_id")
    .agg(
        annotation_count=("annotation_id", "size"),
        analysis_available=("analysis_available", "first"),
        area_outlier_count=("bbox_area_outlier", "sum"),
        aspect_outlier_count=("aspect_ratio_outlier", "sum"),
        any_outlier_count=("any_outlier", "sum"),
    )
    .reset_index()
)

codeit_bbox_outlier_summary["outlier_rate_pct"] = (
    codeit_bbox_outlier_summary["any_outlier_count"]
    / codeit_bbox_outlier_summary["annotation_count"]
    * 100
).round(2)

# 표본 부족 클래스는 비율도 판정 보류
codeit_bbox_outlier_summary.loc[
    ~codeit_bbox_outlier_summary["analysis_available"],
    "outlier_rate_pct",
] = float("nan")

codeit_bbox_outlier_summary = (
    codeit_bbox_outlier_summary.sort_values(
        ["any_outlier_count", "outlier_rate_pct"],
        ascending=False,
    )
)

print("========== 2-18. 클래스별 bbox 크기·종횡비 이상치 (18/18) ==========")
print(f"전체 annotation: {len(codeit_bbox_stats):,}개")
print(f"전체 클래스: {len(codeit_class_bbox_summary):,}종")
print(
    f"분석 가능 클래스: "
    f"{codeit_class_bbox_summary['analysis_available'].sum():,}종"
)
print(
    f"표본 4개 미만으로 판정 보류: "
    f"{(~codeit_class_bbox_summary['analysis_available']).sum():,}종"
)

print(
    f"\n면적 이상치: "
    f"{codeit_bbox_stats['bbox_area_outlier'].sum():,}개"
)
print(
    f"종횡비 이상치: "
    f"{codeit_bbox_stats['aspect_ratio_outlier'].sum():,}개"
)
print(f"둘 중 하나 이상 해당: {len(codeit_bbox_outliers):,}개")
print(f"해당 이미지: {codeit_bbox_outliers['image_id'].nunique():,}장")

print("\n========== 기존 제외 결정과 구분 ==========")
print(
    f"기존 제외 결정에 포함된 후보 이미지: "
    f"{codeit_bbox_outliers.loc[
        codeit_bbox_outliers['image_id'].isin(
            codeit_existing_exclusion_ids
        ),
        'image_id'
    ].nunique():,}장"
)
print(
    f"새로운 후보 이미지: "
    f"{codeit_new_bbox_outliers['image_id'].nunique():,}장"
)
print(f"새로운 후보 annotation: {len(codeit_new_bbox_outliers):,}개")

print("\n========== 클래스별 분석 결과 ==========")
display(codeit_bbox_outlier_summary)

if not codeit_new_bbox_outliers.empty:
    print("\n========== 새로운 후보: 최대 20건 ==========")
    display(codeit_new_bbox_outliers[[
        "json_file", "annotation_id", "image_id", "category_id",
        "bbox", "bbox_area", "bbox_area_lower", "bbox_area_upper",
        "aspect_ratio", "aspect_ratio_lower", "aspect_ratio_upper",
        "bbox_area_outlier", "aspect_ratio_outlier",
    ]].head(20))
else:
    print("\n기존 제외 결정 외에 새로운 이상치 후보는 없습니다.")

print("\n※ 이상치 경계는 클래스별 Q1 − 1.5×IQR, Q3 + 1.5×IQR입니다.")
print("※ 회전과 촬영 각도에 따라서도 면적과 종횡비가 달라집니다.")
print("※ 통계적 이상치만으로 오류나 제외를 확정하지 않습니다.")
print("※ 표본 부족 클래스의 후보 수 0은 정상 판정을 의미하지 않습니다.")
print("※ 파일 수정·삭제 없음")

In [ ]:
codeit_outlier_review_ids = (
    codeit_new_bbox_outliers["image_id"].drop_duplicates().tolist()
)

with zipfile.ZipFile(DATA_PATH, "r") as z:
    for start in range(0, len(codeit_outlier_review_ids), 4):
        batch_ids = codeit_outlier_review_ids[start:start + 4]

        fig, axes = plt.subplots(
            2, 2, figsize=(14, 16), squeeze=False
        )

        for ax, image_id in zip(axes.ravel(), batch_ids):
            image_info = codeit_images_df.loc[
                codeit_images_df["id"].eq(image_id)
            ].iloc[0]

            image_member = resolve_codeit_image_member(
                image_info["file_name"]
            )

            with Image.open(BytesIO(z.read(image_member))) as image:
                ax.imshow(image.convert("RGB"))

            candidates = codeit_new_bbox_outliers.loc[
                codeit_new_bbox_outliers["image_id"].eq(image_id)
            ]
            candidate_ids = set(candidates["annotation_id"])

            annotations = codeit_unique_annotations.loc[
                codeit_unique_annotations["image_id"].eq(image_id)
            ]

            for annotation in annotations.to_dict("records"):
                x, y, w, h = annotation["bbox"]
                is_candidate = annotation["id"] in candidate_ids

                ax.add_patch(Rectangle(
                    (x, y), w, h,
                    fill=False,
                    edgecolor="red" if is_candidate else "gray",
                    linewidth=2.5 if is_candidate else 1,
                ))

                if is_candidate:
                    ax.text(
                        x, y,
                        f"ID {annotation['id']} | "
                        f"Class {annotation['category_id']}",
                        fontsize=8,
                        va="bottom",
                        bbox=dict(
                            facecolor="white",
                            alpha=0.85,
                            edgecolor="none",
                        ),
                    )

            reasons = []
            for candidate in candidates.to_dict("records"):
                flags = []
                if candidate["bbox_area_outlier"]:
                    flags.append("Area")
                if candidate["aspect_ratio_outlier"]:
                    flags.append("Aspect")

                reasons.append(
                    f"ID {candidate['annotation_id']}: "
                    + " / ".join(flags)
                )

            ax.set_title(
                f"Image {image_id}\n" + "\n".join(reasons),
                fontsize=10,
            )
            ax.axis("off")

        for ax in axes.ravel()[len(batch_ids):]:
            ax.axis("off")

        plt.tight_layout()
        plt.show()

print(f"새 후보 이미지 {len(codeit_outlier_review_ids)}장 표시 완료")
print("※ 빨간색: 이상치 후보 박스 / 회색: 나머지 박스")
print("※ 파일 수정·삭제 없음")

### 2-18. 클래스별 bbox 크기·종횡비 이상치 — 검토 결과

- 전체 763개 annotation에서 이상치 후보 38개, 이미지 37장 확인.
- 기존 제외 결정 이미지 6장을 제외한 새 후보 31장·32개 박스를 모두 시각 검토함.
- 객체를 감싸는 박스와 일부 여백이 넓은 박스가 확인됨.
- 가로로 놓인 알약 등 배치 방향에 따른 종횡비 차이도 확인됨.
- 첨부 시각화에서 추가 제외할 만큼 명확한 박스 오류는 확인되지 않음.
- 통계적 이상치만으로 제외하지 않으며, 이번 검사에 따른 추가 제외 없음.
- 표본 4개 미만인 17종은 통계적 이상치 판정을 보류함.

### 2-추가 검사. 클래스별 bbox 중심 위치 편향
- 제외 결정 이미지를 뺀 데이터에서 검사
- 박스가 10개 이상인 클래스 중 한 구역에 70% 이상 몰린 경우를 표시

In [ ]:
codeit_position_rows = []

for annotation in codeit_unique_annotations.to_dict("records"):
    image_id = annotation["image_id"]

    if image_id in codeit_existing_exclusion_ids:
        continue

    image_info = codeit_image_size_lookup[
        (annotation["_json_file"], image_id)
    ]

    x, y, w, h = annotation["bbox"]
    width = image_info["width"]
    height = image_info["height"]

    center_x = (x + w / 2) / width
    center_y = (y + h / 2) / height

    horizontal = "왼쪽" if center_x < 0.5 else "오른쪽"
    vertical = "위" if center_y < 0.5 else "아래"
    quadrant = f"{horizontal} {vertical}"

    codeit_position_rows.append({
        "annotation_id": annotation["id"],
        "image_id": image_id,
        "category_id": annotation["category_id"],
        "center_x": center_x,
        "center_y": center_y,
        "quadrant": quadrant,
    })

codeit_position_checks = pd.DataFrame(codeit_position_rows)

codeit_quadrant_order = [
    "왼쪽 위", "오른쪽 위", "왼쪽 아래", "오른쪽 아래"
]

codeit_position_counts = pd.crosstab(
    codeit_position_checks["category_id"],
    codeit_position_checks["quadrant"],
).reindex(columns=codeit_quadrant_order, fill_value=0)

codeit_position_summary = codeit_position_counts.copy()
codeit_position_summary["annotation_count"] = (
    codeit_position_counts.sum(axis=1)
)
codeit_position_summary["max_quadrant"] = (
    codeit_position_counts.idxmax(axis=1)
)
codeit_position_summary["max_quadrant_count"] = (
    codeit_position_counts.max(axis=1)
)
codeit_position_summary["max_quadrant_ratio"] = (
    codeit_position_summary["max_quadrant_count"]
    / codeit_position_summary["annotation_count"]
)
codeit_position_summary["analysis_available"] = (
    codeit_position_summary["annotation_count"].ge(10)
)
codeit_position_summary["position_bias"] = (
    codeit_position_summary["analysis_available"]
    & codeit_position_summary["max_quadrant_ratio"].ge(0.7)
)
codeit_position_summary["max_quadrant_pct"] = (
    codeit_position_summary["max_quadrant_ratio"] * 100
).round(2)

codeit_position_summary = (
    codeit_position_summary.reset_index()
    .sort_values("max_quadrant_ratio", ascending=False)
)

codeit_position_bias_candidates = codeit_position_summary.loc[
    codeit_position_summary["position_bias"]
].copy()

print("========== 추가 검사: 클래스별 bbox 중심 위치 편향 ==========")
print(f"분석 annotation: {len(codeit_position_checks):,}개")
print(f"분석 클래스: {len(codeit_position_summary):,}종")
print(
    f"박스 10개 이상인 클래스: "
    f"{codeit_position_summary['analysis_available'].sum():,}종"
)
print(
    f"표본 부족으로 판정 보류: "
    f"{(~codeit_position_summary['analysis_available']).sum():,}종"
)
print(f"위치 편향 후보: {len(codeit_position_bias_candidates):,}종")
print("기준: 박스 10개 이상이며 한 구역에 70% 이상")

if not codeit_position_bias_candidates.empty:
    print("\n========== 위치 편향 후보 ==========")
    display(codeit_position_bias_candidates[[
        "category_id", "annotation_count",
        *codeit_quadrant_order,
        "max_quadrant", "max_quadrant_pct",
    ]])
else:
    print("\n[체크 가능] 기준에 해당하는 위치 편향 후보 없음")

print("\n※ 기존 제외 결정 이미지를 뺀 데이터로 분석했습니다.")
print("※ 위치 편향은 데이터 분포 특성이며 자동 제외 사유가 아닙니다.")
print("※ 파일 수정·삭제 없음")

### 추가 검사: 클래스별 bbox 중심 위치 편향 — 분석 결과

- 기존 제외 결정 이미지를 뺀 717개 annotation, 56종을 분석함.
- 박스 10개 이상인 27종을 대상으로 판정하고, 29종은 표본 부족으로 판정을 보류함.
- 한 구역에 70% 이상 몰린 클래스는 5종임.
  - K-033880: 왼쪽 아래 85.71%
  - K-019232: 왼쪽 아래 73.33%
  - K-001900: 왼쪽 위 71.43%
  - K-031863: 오른쪽 아래 70.00%
  - K-003832: 왼쪽 아래 70.00%
- 위치 편향은 라벨 오류와 구분하며, 이번 분석에 따른 추가 제외 없음.
- SELECT 데이터와 통합한 뒤 위치 분포를 다시 확인할 예정임.

In [ ]:
position_candidates = (
    codeit_position_bias_candidates.reset_index(drop=True)
)

if position_candidates.empty:
    print("위치 편향 후보가 없습니다.")
else:
    columns = 3
    rows = (len(position_candidates) + columns - 1) // columns

    fig, axes = plt.subplots(
        rows, columns,
        figsize=(5 * columns, 4.5 * rows),
        squeeze=False,
        constrained_layout=True,
    )

    active_axes = []

    for ax, record in zip(
        axes.ravel(), position_candidates.to_dict("records")
    ):
        total = record["annotation_count"]

        counts = np.array([
            [record["왼쪽 위"], record["오른쪽 위"]],
            [record["왼쪽 아래"], record["오른쪽 아래"]],
        ])
        percentages = counts / total * 100

        heatmap = ax.imshow(
            percentages,
            cmap="Blues",
            vmin=0,
            vmax=100,
        )

        for row_index in range(2):
            for column_index in range(2):
                percentage = percentages[row_index, column_index]

                ax.text(
                    column_index,
                    row_index,
                    f"{counts[row_index, column_index]} / {total}\n"
                    f"{percentage:.1f}%",
                    ha="center",
                    va="center",
                    fontsize=12,
                    color="white" if percentage >= 55 else "black",
                )

        ax.set_xticks([0, 1], labels=["Left", "Right"])
        ax.set_yticks([0, 1], labels=["Top", "Bottom"])
        ax.set_title(
            f"K-{int(record['category_id']):06d}\n"
            f"Max: {record['max_quadrant_pct']:.2f}%"
        )
        active_axes.append(ax)

    for ax in axes.ravel()[len(position_candidates):]:
        ax.axis("off")

    fig.colorbar(
        heatmap,
        ax=active_axes,
        label="Box centers (%)",
        shrink=0.8,
    )
    fig.suptitle(
        "Codeit Train: class position bias after exclusions",
        fontsize=15,
    )
    plt.show()

print("※ 각 칸은 이미지의 해당 구역에 중심이 있는 박스 수와 비율입니다.")
print("※ 모든 그래프에 동일한 색상 기준 0~100%를 적용했습니다.")
print("※ 파일 수정·삭제 없음")

## 3. 데이터 통합

### 3-1. 대회 데이터 최종 집계 및 공유용 저장

In [ ]:
# 대회 데이터 폴더와 같은 data 폴더 아래의 select 사용
SELECT_DIR = DATA_PATH.parent.parent / "select"

if not SELECT_DIR.is_dir():
    raise FileNotFoundError(f"SELECT 폴더를 찾지 못했습니다: {SELECT_DIR}")

select_annotation_paths = sorted(
    (SELECT_DIR / "annotations").rglob("*.json")
)

if not select_annotation_paths:
    raise FileNotFoundError(
        f"SELECT annotation JSON을 찾지 못했습니다: {SELECT_DIR / 'annotations'}"
    )

select_example_path = select_annotation_paths[0]
select_example = json.loads(
    select_example_path.read_text(encoding="utf-8-sig")
)

print("========== SELECT JSON 양식 ==========")
print(f"파일: {select_example_path}")
print(json.dumps(select_example, ensure_ascii=False, indent=2))

print("\n========== 클래스 매핑 파일 후보 ==========")

select_mapping_paths = sorted(
    path
    for path in SELECT_DIR.rglob("*")
    if path.is_file()
    and "mapping" in path.name.lower()
    and path.suffix.lower() in {".csv", ".json"}
)

if not select_mapping_paths:
    print("SELECT 내부에서 매핑 파일을 찾지 못했습니다.")
else:
    for path in select_mapping_paths:
        print(f"\n파일: {path}")

        if path.suffix.lower() == ".csv":
            mapping_preview = pd.read_csv(
                path, encoding="utf-8-sig"
            )
            print(f"전체 행 수: {len(mapping_preview):,}")
            display(mapping_preview.head())
        else:
            mapping_preview = json.loads(
                path.read_text(encoding="utf-8-sig")
            )
            preview_text = json.dumps(
                mapping_preview, ensure_ascii=False, indent=2
            )
            print(preview_text[:4000])

print("\n※ 파일 수정·저장·삭제 없음")

In [ ]:
from datetime import datetime

# 1. SELECT 클래스 매핑 확인
codeit_shared_mapping = pd.read_csv(
    SELECT_DIR / "class_mapping.csv",
    encoding="utf-8-sig",
)

for field in ["category_id", "model_class_id"]:
    converted = codeit_shared_mapping[field].map(codeit_integer_value)

    if converted.isna().any():
        raise ValueError(f"클래스 매핑에 유효하지 않은 {field}가 있습니다.")

    codeit_shared_mapping[field] = converted.astype(int)

for field in ["code", "category_id", "model_class_id"]:
    if codeit_shared_mapping[field].duplicated().any():
        raise ValueError(f"클래스 매핑에 중복된 {field}가 있습니다.")

codeit_mapping_by_category = (
    codeit_shared_mapping.set_index("category_id").to_dict("index")
)

# 2. 확정된 제외 사유 통합
codeit_exclusion_sources = [
    ("2-3", "bbox 면적 불일치", codeit_area_mismatches["image_id"]),
    ("2-4", "이미지 범위 초과", codeit_boundary_errors["image_id"]),
    ("2-6", "동일 bbox 및 위치 오류",
     codeit_same_bbox_candidates["image_id"]),
    ("2-14", "annotation 누락",
     codeit_composition_issues["image_id"]),
]

codeit_exclusion_reasons = pd.DataFrame([
    {"image_id": image_id, "stage": stage, "reason": reason}
    for stage, reason, image_ids in codeit_exclusion_sources
    for image_id in sorted(set(image_ids))
])

codeit_final_exclusion_ids = set(
    codeit_exclusion_reasons["image_id"]
)

codeit_final_annotations = codeit_unique_annotations.loc[
    ~codeit_unique_annotations["image_id"].isin(
        codeit_final_exclusion_ids
    )
].copy()

unknown_categories = {
    codeit_integer_value(value)
    for value in codeit_final_annotations["category_id"]
} - set(codeit_mapping_by_category)

if unknown_categories:
    raise ValueError(
        f"SELECT 매핑에 없는 대회 클래스가 있습니다: {unknown_categories}"
    )

# 3. 이미지별 JSON 구성
codeit_converted_documents = {}
codeit_export_image_members = {}
codeit_export_manifest_rows = []

for image_id, group in codeit_final_annotations.groupby("image_id"):
    image_id = int(image_id)

    image_info = codeit_images_df.loc[
        codeit_images_df["id"].eq(image_id)
    ].iloc[0]

    original_name = image_info["file_name"]
    original_member = resolve_codeit_image_member(original_name)

    exported_name = f"codeit_{Path(original_name).name}"
    exported_json_name = f"{Path(exported_name).stem}.json"

    image_relative_path = f"images/{exported_name}"
    json_relative_path = f"annotations/{exported_json_name}"

    if json_relative_path in codeit_converted_documents:
        raise ValueError(f"출력 파일명 중복: {json_relative_path}")

    annotations = []
    categories = {}

    for annotation in group.to_dict("records"):
        category_id = codeit_integer_value(annotation["category_id"])
        mapping = codeit_mapping_by_category[category_id]

        expected_code = f"K-{category_id:06d}"
        if mapping["code"] != expected_code:
            raise ValueError(f"클래스 코드 매핑 불일치: {category_id}")

        original_image_info = codeit_image_size_lookup[
            (annotation["_json_file"], annotation["image_id"])
        ]
        source_dl_idx = codeit_integer_value(
            original_image_info.get("dl_idx")
        )

        if source_dl_idx is None:
            raise ValueError(
                f"원본 dl_idx 확인 불가: annotation {annotation['id']}"
            )

        bbox = [float(value) for value in annotation["bbox"]]
        area = float(annotation["area"])

        if not math.isclose(
            area, bbox[2] * bbox[3],
            rel_tol=1e-9, abs_tol=1e-6,
        ):
            raise ValueError(
                f"유지 대상에 면적 오류가 남아 있습니다: {annotation['id']}"
            )

        annotations.append({
            "id": int(annotation["id"]),
            "image_id": image_id,
            "category_id": category_id,
            "bbox": bbox,
            "area": area,
            "iscrowd": int(annotation["iscrowd"]),
            "drug_code": mapping["code"],
            "model_class_id": int(mapping["model_class_id"]),
            "source_dl_idx": source_dl_idx,
        })

        categories[category_id] = {
            "id": category_id,
            "name": mapping["code"],
        }

    document = {
        "images": [{
            "id": image_id,
            "file_name": exported_name,
            "width": int(image_info["width"]),
            "height": int(image_info["height"]),
        }],
        "annotations": annotations,
        "categories": [
            categories[key] for key in sorted(categories)
        ],
        "source": {
            # AIHub의 1~8과 구분하여 대회 출처는 0으로 기록
            "source_number": 0,
            "original_image": Path(original_name).name,
            "original_path": original_member,
        },
    }

    # SELECT와 필드 구성이 같은지 확인
    assert set(document) == set(select_example)

    for section in ["images", "annotations", "categories"]:
        expected_fields = set(select_example[section][0])
        for record in document[section]:
            assert set(record) == expected_fields, (
                f"{section} 필드 불일치: {json_relative_path}"
            )

    assert set(document["source"]) == set(select_example["source"])

    codeit_converted_documents[json_relative_path] = document
    codeit_export_image_members[image_relative_path] = original_member

    codeit_export_manifest_rows.append({
        "source_dataset": "codeit",
        "original_image_id": image_id,
        "image_id": image_id,
        "file_name": exported_name,
        "image_path": image_relative_path,
        "json_file": json_relative_path,
        "annotation_count": len(annotations),
    })

codeit_export_manifest = pd.DataFrame(codeit_export_manifest_rows)

assert sum(
    len(document["annotations"])
    for document in codeit_converted_documents.values()
) == len(codeit_final_annotations)

# 4. 최종 수량 및 클래스 분포
codeit_selection_manifest = (
    codeit_images_df[["id", "file_name"]]
    .drop_duplicates()
    .rename(columns={"id": "image_id"})
    .copy()
)

codeit_selection_manifest["keep"] = (
    ~codeit_selection_manifest["image_id"].isin(
        codeit_final_exclusion_ids
    )
)

reason_lookup = (
    codeit_exclusion_reasons.groupby("image_id")["reason"]
    .agg(lambda values: " / ".join(values))
)

codeit_selection_manifest["exclusion_reason"] = (
    codeit_selection_manifest["image_id"]
    .map(reason_lookup)
    .fillna("")
)

assert (
    codeit_selection_manifest["keep"].sum()
    == len(codeit_export_manifest)
), "유지 이미지 목록과 변환 결과 수가 다릅니다."

codeit_final_class_distribution = pd.DataFrame({
    "before_boxes": (
        codeit_unique_annotations.groupby("category_id").size()
    ),
    "after_boxes": (
        codeit_final_annotations.groupby("category_id").size()
    ),
}).fillna(0).astype(int)

codeit_final_class_distribution["removed_boxes"] = (
    codeit_final_class_distribution["before_boxes"]
    - codeit_final_class_distribution["after_boxes"]
)

# 5. 저장 폴더 생성 및 근거 저장
codeit_save_time = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

CODEIT_RESULT_DIR = DATA_PATH.parent / (
    f"codeit_validated_{codeit_save_time}"
)
CODEIT_RESULT_DIR.mkdir(parents=True, exist_ok=False)

codeit_tables_to_save = {
    "selection_manifest.csv": codeit_selection_manifest,
    "export_manifest.csv": codeit_export_manifest,
    "exclusion_reasons.csv": codeit_exclusion_reasons,
    "class_mapping.csv": codeit_shared_mapping,
    "position_summary.csv": codeit_position_summary,
}

for filename, table in codeit_tables_to_save.items():
    table.to_csv(
        CODEIT_RESULT_DIR / filename,
        index=False,
        encoding="utf-8-sig",
    )

codeit_final_class_distribution.to_csv(
    CODEIT_RESULT_DIR / "class_distribution.csv",
    encoding="utf-8-sig",
)

# 원본 메타데이터 보존: 촬영 조건과 원본 annotation 연결 정보 포함
codeit_retained_json_files = set(
    codeit_final_annotations["_json_file"]
)

codeit_source_metadata = {
    "dataset": "codeit",
    "original_zip": DATA_PATH.name,
    "images": [
        record for record in codeit_table_rows["images"]
        if record["_json_file"] in codeit_retained_json_files
    ],
    "annotations": [
        record for record in codeit_table_rows["annotations"]
        if record["_json_file"] in codeit_retained_json_files
    ],
}

(CODEIT_RESULT_DIR / "source_metadata.json").write_text(
    json.dumps(
        codeit_source_metadata,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    ),
    encoding="utf-8",
)

codeit_report = f"""# 대회 데이터 최종 검증 결과

- 전체 이미지: {len(codeit_selection_manifest)}장
- 제외 이미지: {len(codeit_final_exclusion_ids)}장
- 유지 이미지: {len(codeit_export_manifest)}장
- 유지 annotation: {len(codeit_final_annotations)}개
- 유지 클래스: {codeit_final_annotations["category_id"].nunique()}종

## 제외 원칙
빠른 데이터셋 구성을 위해 확인된 오류는 수정하지 않고,
해당 이미지와 연결된 모든 annotation을 제외함.
상세 사유는 exclusion_reasons.csv에 기록함.

## JSON 구성
- SELECT와 같은 필드로 이미지 한 장당 JSON 한 개를 구성함.
- SELECT의 class_mapping.csv를 그대로 사용함.
- source_dl_idx는 대회 원본 dl_idx를 보존함.
- source_number 0은 대회 데이터를 뜻함.
- 이미지와 annotation ID는 대회 원본 값을 유지하며,
  SELECT 통합 단계에서 전체 ID 충돌을 처리할 예정임.
- 원본 촬영 조건 등은 source_metadata.json에 보존함.

## 검토상 한계와 유지 결정
- 유사 이미지 16쌍은 세부 차이를 확인하여 추가 제외하지 않음.
- 길이 정보가 0인 18개 annotation은 그 사유만으로 제외하지 않음.
- 통계적 이상치 새 후보 31장을 시각 검토하고 추가 제외하지 않음.
- 위치 편향 5종은 분포 특성으로 기록하며 제외하지 않음.
- 표본 부족 클래스는 통계적 판정을 보류함.
- Test는 포함하지 않음.
- SELECT와의 중복 검사 및 Train/Validation 분할 전 단계임.
"""

(CODEIT_RESULT_DIR / "validation_report.md").write_text(
    codeit_report,
    encoding="utf-8",
)

# 6. 공유용 ZIP 생성
CODEIT_CLEAN_ZIP = CODEIT_RESULT_DIR / "codeit_train_validated.zip"

codeit_expected_zip_members = (
    set(codeit_export_image_members)
    | set(codeit_converted_documents)
    | {
        path.name
        for path in CODEIT_RESULT_DIR.iterdir()
        if path.is_file() and path != CODEIT_CLEAN_ZIP
    }
)

with zipfile.ZipFile(DATA_PATH, "r") as source_zip:
    with zipfile.ZipFile(
        CODEIT_CLEAN_ZIP,
        "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=1,
    ) as output_zip:
        for output_path, original_member in (
            codeit_export_image_members.items()
        ):
            output_zip.writestr(
                output_path, source_zip.read(original_member)
            )

        for output_path, document in codeit_converted_documents.items():
            output_zip.writestr(
                output_path,
                json.dumps(
                    document,
                    ensure_ascii=False,
                    indent=2,
                    allow_nan=False,
                ).encode("utf-8"),
            )

        for path in sorted(CODEIT_RESULT_DIR.iterdir()):
            if path.is_file() and path != CODEIT_CLEAN_ZIP:
                output_zip.write(path, arcname=path.name)

# 7. ZIP 저장 검증
with zipfile.ZipFile(CODEIT_CLEAN_ZIP, "r") as saved_zip:
    assert set(saved_zip.namelist()) == codeit_expected_zip_members

    bad_member = saved_zip.testzip()
    if bad_member is not None:
        raise ValueError(f"ZIP 무결성 검사 실패: {bad_member}")

    for json_path, expected_document in codeit_converted_documents.items():
        saved_document = json.loads(
            saved_zip.read(json_path).decode("utf-8")
        )
        assert saved_document == expected_document

        image_name = saved_document["images"][0]["file_name"]
        assert f"images/{image_name}" in codeit_expected_zip_members

print("========== 3-1. 대회 데이터 최종 집계 및 공유용 저장 ==========")
print(f"전체 이미지: {len(codeit_selection_manifest):,}장")
print(f"제외 이미지: {len(codeit_final_exclusion_ids):,}장")
print(f"유지 이미지: {len(codeit_export_manifest):,}장")
print(f"유지 annotation: {len(codeit_final_annotations):,}개")
print(
    f"유지 클래스: "
    f"{codeit_final_annotations['category_id'].nunique():,}종"
)
print(
    f"박스가 0개가 된 클래스: "
    f"{codeit_final_class_distribution['after_boxes'].eq(0).sum():,}종"
)
print(f"이미지별 JSON: {len(codeit_converted_documents):,}개")
print("\nSELECT 필드 구성·클래스 매핑 확인 완료")
print("ZIP 파일 구성·무결성·JSON 저장 내용 확인 완료")
print(f"\n저장 폴더: {CODEIT_RESULT_DIR}")
print(f"공유용 ZIP: {CODEIT_CLEAN_ZIP}")
print(f"ZIP 크기: {CODEIT_CLEAN_ZIP.stat().st_size / 1024**2:.1f} MB")
print("\n※ 원본 ZIP과 SELECT 파일은 변경하지 않았습니다.")

### 3-2. SELECT 유지 목록 연결 및 통합 전 중복 확인

In [ ]:
DATA_ROOT = DATA_PATH.parent.parent

select_manifest_candidates = sorted(
    path
    for path in DATA_ROOT.rglob("selection_manifest.csv")
    if "codeit" not in path.relative_to(DATA_ROOT).parts
)

print("========== 3-2. SELECT 최종 유지 목록 확인 ==========")

if not select_manifest_candidates:
    print("data 폴더 아래에서 SELECT 유지 목록을 찾지 못했습니다.")
else:
    for path in select_manifest_candidates:
        table = pd.read_csv(path, encoding="utf-8-sig")

        print(f"\n파일: {path}")
        print(f"전체 기록: {len(table):,}개")
        print(f"열: {table.columns.tolist()}")

        if "keep" in table.columns:
            print("keep 값별 기록 수:")
            display(
                table["keep"]
                .value_counts(dropna=False)
                .rename_axis("keep")
                .reset_index(name="count")
            )

        display(table.head(2))

print("\n※ 파일 읽기만 수행했으며 변경 사항은 없습니다.")

In [ ]:
import hashlib
from io import BytesIO

CODEIT_RESULT_DIR = DATA_PATH.parent / (
    "codeit_validated_20261027_132901_622959"
)
CODEIT_CLEAN_ZIP = CODEIT_RESULT_DIR / "codeit_train_validated.zip"

if not CODEIT_CLEAN_ZIP.is_file():
    raise FileNotFoundError(f"저장한 ZIP을 찾지 못했습니다: {CODEIT_CLEAN_ZIP}")

# 저장된 유지 목록 복구
codeit_export_manifest = pd.read_csv(
    CODEIT_RESULT_DIR / "export_manifest.csv",
    encoding="utf-8-sig",
)

codeit_recovered_hash_rows = []

with zipfile.ZipFile(CODEIT_CLEAN_ZIP, "r") as z:
    for record in codeit_export_manifest.to_dict("records"):
        with Image.open(
            BytesIO(z.read(record["image_path"]))
        ) as image:
            rgb = image.convert("RGB")

            digest = hashlib.md5()
            digest.update(
                f"RGB:{rgb.width}:{rgb.height}:".encode()
            )
            digest.update(rgb.tobytes())

        codeit_recovered_hash_rows.append({
            "image_id": int(record["original_image_id"]),
            "file_name": record["file_name"],
            "pixel_md5": digest.hexdigest(),
            "hash_error": "",
        })

codeit_image_hashes = pd.DataFrame(codeit_recovered_hash_rows)

assert len(codeit_image_hashes) == 215
assert not codeit_image_hashes["image_id"].duplicated().any()

print("========== 3-2. 대회 데이터 변수 복구 ==========")
print(f"유지 목록 복구: {len(codeit_export_manifest):,}장")
print(f"픽셀 해시 재계산: {len(codeit_image_hashes):,}장")
print("복구 완료 — 기존 3-2 연결·중복 검사 셀을 다시 실행하세요.")
print("※ 파일 수정·재저장 없음")

In [ ]:
# 1. 확인된 SELECT 유지 목록 연결
if len(select_manifest_candidates) != 1:
    raise ValueError(
        "유지 목록이 여러 개이므로 경로를 하나로 지정해야 합니다.\n"
        + "\n".join(str(path) for path in select_manifest_candidates)
    )

SELECT_MANIFEST_PATH = select_manifest_candidates[0]

select_selection_manifest = pd.read_csv(
    SELECT_MANIFEST_PATH,
    encoding="utf-8-sig",
)

# 문자열 False가 True로 해석되지 않도록 명시적으로 변환
select_keep_values = (
    select_selection_manifest["keep"]
    .astype(str)
    .str.strip()
    .str.lower()
)

if not select_keep_values.isin(["true", "false"]).all():
    raise ValueError("SELECT 유지 목록에 해석할 수 없는 keep 값이 있습니다.")

select_selection_manifest["keep"] = select_keep_values.eq("true")

select_keep_manifest = select_selection_manifest.loc[
    select_selection_manifest["keep"]
].copy().reset_index(drop=True)

assert len(select_keep_manifest) == 10109
assert not select_keep_manifest["json_file"].duplicated().any()
assert not select_keep_manifest["file_name"].duplicated().any()

# 2. 대회 유지 이미지의 기존 픽셀 해시 재사용
codeit_kept_hashes = codeit_image_hashes.loc[
    codeit_image_hashes["image_id"].isin(
        codeit_export_manifest["original_image_id"]
    )
].copy()

assert len(codeit_kept_hashes) == 215
assert codeit_kept_hashes["hash_error"].eq("").all()

codeit_pixel_hash_lookup = {}

for record in codeit_kept_hashes.to_dict("records"):
    codeit_pixel_hash_lookup.setdefault(
        record["pixel_md5"], []
    ).append(record)

# 3. SELECT JSON 연결 및 실제 이미지 픽셀 비교
select_kept_documents = {}
select_cross_hash_rows = []
codeit_select_duplicate_rows = []
select_actual_annotation_count = 0

for index, record in enumerate(
    select_keep_manifest.to_dict("records"),
    start=1,
):
    json_path = SELECT_DIR / record["json_file"]

    document = json.loads(
        json_path.read_text(encoding="utf-8-sig")
    )

    if len(document["images"]) != 1:
        raise ValueError(f"이미지별 JSON 구조 확인 필요: {json_path}")

    image_info = document["images"][0]

    if image_info["file_name"] != record["file_name"]:
        raise ValueError(f"유지 목록과 JSON 파일명 불일치: {json_path}")

    if len(document["annotations"]) != int(record["annotation_count"]):
        raise ValueError(f"유지 목록과 현재 박스 수 불일치: {json_path}")

    image_candidates = {
        path
        for path in [
            SELECT_DIR / "images" / image_info["file_name"],
            SELECT_DIR / image_info["file_name"],
        ]
        if path.is_file()
    }

    if len(image_candidates) != 1:
        raise ValueError(
            f"SELECT 이미지 경로 확인 필요: {image_info['file_name']}"
        )

    image_path = next(iter(image_candidates))

    with Image.open(image_path) as image:
        rgb = image.convert("RGB")

        # 대회 검사와 동일한 방식으로 크기와 RGB 픽셀 비교
        digest = hashlib.md5()
        digest.update(
            f"RGB:{rgb.width}:{rgb.height}:".encode()
        )
        digest.update(rgb.tobytes())
        pixel_md5 = digest.hexdigest()

    select_kept_documents[record["json_file"]] = document
    select_actual_annotation_count += len(document["annotations"])

    select_cross_hash_rows.append({
        "json_file": record["json_file"],
        "image_id": image_info["id"],
        "file_name": image_info["file_name"],
        "image_path": image_path.relative_to(SELECT_DIR).as_posix(),
        "pixel_md5": pixel_md5,
    })

    for matched in codeit_pixel_hash_lookup.get(pixel_md5, []):
        codeit_select_duplicate_rows.append({
            "select_json_file": record["json_file"],
            "select_image_id": image_info["id"],
            "select_file_name": image_info["file_name"],
            "codeit_image_id": matched["image_id"],
            "codeit_file_name": matched["file_name"],
            "pixel_md5": pixel_md5,
        })

    if index % 1000 == 0 or index == len(select_keep_manifest):
        print(f"SELECT 연결·픽셀 비교: {index:,}/{len(select_keep_manifest):,}장")

select_cross_hashes = pd.DataFrame(select_cross_hash_rows)

codeit_select_exact_duplicates = pd.DataFrame(
    codeit_select_duplicate_rows,
    columns=[
        "select_json_file", "select_image_id", "select_file_name",
        "codeit_image_id", "codeit_file_name", "pixel_md5",
    ],
)

assert select_actual_annotation_count == 38891, (
    "SELECT 박스 수가 어제 최종 결과 38,891개와 다릅니다."
)

print("\n========== 3-2. SELECT 연결 및 데이터 간 완전 중복 검사 ==========")
print(f"SELECT 유지 이미지: {len(select_keep_manifest):,}장")
print(f"SELECT 유지 annotation: {select_actual_annotation_count:,}개")
print(f"대회 유지 이미지: {len(codeit_kept_hashes):,}장")
print(f"데이터 간 완전 중복: {len(codeit_select_exact_duplicates):,}쌍")
print(
    f"중복에 해당하는 SELECT 이미지: "
    f"{codeit_select_exact_duplicates['select_image_id'].nunique():,}장"
)

if not codeit_select_exact_duplicates.empty:
    display(codeit_select_exact_duplicates)
else:
    print("\n두 데이터 사이에 동일한 크기·RGB 픽셀 해시는 없습니다.")

print("\n※ 이번 단계는 완전 중복 검사입니다. 유사 이미지 비교는 포함하지 않습니다.")
print("※ Test와의 중복 확인은 아직 수행하지 않았습니다.")
print("※ 파일 수정·삭제 없음")

### 3-3. 데이터 간 유사 이미지 및 Test 혼입 후보 검사
- SELECT ↔ 대회 Train: 유사 이미지 후보
- 통합 후보 ↔ Test: 완전 중복 및 유사 이미지 후보

In [ ]:
import numpy as np

# 추가 설치 없이 64비트 pHash 계산
integration_dct_matrix = np.cos(
    np.pi
    * (2 * np.arange(32) + 1)
    * np.arange(8)[:, None]
    / 64
)
integration_dct_matrix[0] *= 1 / np.sqrt(2)


def integration_image_hashes(image):
    rgb = image.convert("RGB")

    digest = hashlib.md5()
    digest.update(f"RGB:{rgb.width}:{rgb.height}:".encode())
    digest.update(rgb.tobytes())

    gray = rgb.convert("L").resize(
        (32, 32), Image.Resampling.LANCZOS
    )
    pixels = np.asarray(gray, dtype=np.float64)
    coefficients = (
        integration_dct_matrix
        @ pixels
        @ integration_dct_matrix.T
    )
    bits = (coefficients > np.median(coefficients)).ravel()
    phash = int("".join("1" if bit else "0" for bit in bits), 2)

    return digest.hexdigest(), f"{phash:016x}"


integration_hash_rows = []

# 1. SELECT 유지 이미지
for index, record in enumerate(
    select_cross_hashes.to_dict("records"), start=1
):
    with Image.open(SELECT_DIR / record["image_path"]) as image:
        pixel_md5, phash_hex = integration_image_hashes(image)

    if pixel_md5 != record["pixel_md5"]:
        raise ValueError(
            f"앞 검사 이후 이미지 내용이 달라졌습니다: {record['file_name']}"
        )

    integration_hash_rows.append({
        "dataset": "select",
        "image_id": str(record["image_id"]),
        "file_name": record["file_name"],
        "image_path": record["image_path"],
        "pixel_md5": pixel_md5,
        "phash_hex": phash_hex,
    })

    if index % 1000 == 0 or index == len(select_cross_hashes):
        print(f"SELECT 해시: {index:,}/{len(select_cross_hashes):,}장")

# 2. 검증 완료 대회 Train 이미지
with zipfile.ZipFile(CODEIT_CLEAN_ZIP, "r") as z:
    for record in codeit_export_manifest.to_dict("records"):
        with Image.open(BytesIO(z.read(record["image_path"]))) as image:
            pixel_md5, phash_hex = integration_image_hashes(image)

        integration_hash_rows.append({
            "dataset": "codeit",
            "image_id": str(record["original_image_id"]),
            "file_name": record["file_name"],
            "image_path": record["image_path"],
            "pixel_md5": pixel_md5,
            "phash_hex": phash_hex,
        })

print(f"대회 Train 해시: {len(codeit_export_manifest):,}장 완료")

# 3. 원본 ZIP의 Test 이미지
with zipfile.ZipFile(DATA_PATH, "r") as z:
    integration_test_members = sorted(
        member
        for member in z.namelist()
        if "test_images" in Path(member).parts
        and "__MACOSX" not in Path(member).parts
        and not Path(member).name.startswith(".")
        and Path(member).suffix.lower() in {
            ".png", ".jpg", ".jpeg", ".webp"
        }
    )

    for index, member in enumerate(integration_test_members, start=1):
        with Image.open(BytesIO(z.read(member))) as image:
            pixel_md5, phash_hex = integration_image_hashes(image)

        integration_hash_rows.append({
            "dataset": "test",
            "image_id": Path(member).stem,
            "file_name": Path(member).name,
            "image_path": member,
            "pixel_md5": pixel_md5,
            "phash_hex": phash_hex,
        })

        if index % 200 == 0 or index == len(integration_test_members):
            print(f"Test 해시: {index:,}/{len(integration_test_members):,}장")

integration_hashes = pd.DataFrame(integration_hash_rows)

# 해시 계산 결과를 먼저 저장
INTEGRATION_CHECK_DIR = CODEIT_RESULT_DIR / "integration_checks"
INTEGRATION_CHECK_DIR.mkdir(parents=True, exist_ok=True)

integration_hashes.to_csv(
    INTEGRATION_CHECK_DIR / "image_hashes.csv",
    index=False,
    encoding="utf-8-sig",
)

# 4. 데이터 간 비교 함수
def find_integration_similarities(left_table, right_table):
    rows = []

    right_records = [
        (record, int(record["phash_hex"], 16))
        for record in right_table.to_dict("records")
    ]

    for left in left_table.to_dict("records"):
        left_hash = int(left["phash_hex"], 16)

        for right, right_hash in right_records:
            same_pixels = left["pixel_md5"] == right["pixel_md5"]
            distance = (left_hash ^ right_hash).bit_count()

            if same_pixels or distance <= 5:
                rows.append({
                    "dataset_a": left["dataset"],
                    "image_id_a": left["image_id"],
                    "file_name_a": left["file_name"],
                    "image_path_a": left["image_path"],
                    "dataset_b": right["dataset"],
                    "image_id_b": right["image_id"],
                    "file_name_b": right["file_name"],
                    "image_path_b": right["image_path"],
                    "same_pixels": same_pixels,
                    "phash_distance": distance,
                })

    return pd.DataFrame(rows, columns=[
        "dataset_a", "image_id_a", "file_name_a", "image_path_a",
        "dataset_b", "image_id_b", "file_name_b", "image_path_b",
        "same_pixels", "phash_distance",
    ]).sort_values(
        ["same_pixels", "phash_distance"],
        ascending=[False, True],
    ).reset_index(drop=True)


integration_select_hashes = integration_hashes.loc[
    integration_hashes["dataset"].eq("select")
]
integration_codeit_hashes = integration_hashes.loc[
    integration_hashes["dataset"].eq("codeit")
]
integration_test_hashes = integration_hashes.loc[
    integration_hashes["dataset"].eq("test")
]

codeit_select_similar_candidates = find_integration_similarities(
    integration_select_hashes,
    integration_codeit_hashes,
)
print("SELECT ↔ 대회 Train 비교 완료")

integration_test_candidates = find_integration_similarities(
    integration_hashes.loc[integration_hashes["dataset"].ne("test")],
    integration_test_hashes,
)
print("통합 후보 ↔ Test 비교 완료")

# 5. 후보 결과 저장
codeit_select_similar_candidates.to_csv(
    INTEGRATION_CHECK_DIR / "select_codeit_similar_candidates.csv",
    index=False,
    encoding="utf-8-sig",
)
integration_test_candidates.to_csv(
    INTEGRATION_CHECK_DIR / "test_overlap_candidates.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n========== 3-3. 데이터 간 유사 이미지 및 Test 혼입 후보 검사 ==========")
print(f"SELECT: {len(integration_select_hashes):,}장")
print(f"대회 Train: {len(integration_codeit_hashes):,}장")
print(f"Test: {len(integration_test_hashes):,}장")

print("\n[SELECT ↔ 대회 Train]")
print(
    f"완전 중복: "
    f"{codeit_select_similar_candidates['same_pixels'].sum():,}쌍"
)
print(
    f"완전 중복 외 유사 후보: "
    f"{(~codeit_select_similar_candidates['same_pixels']).sum():,}쌍"
)
display(codeit_select_similar_candidates.head(20))

print("\n[통합 후보 ↔ Test]")
print(
    f"완전 중복: "
    f"{integration_test_candidates['same_pixels'].sum():,}쌍"
)
print(
    f"완전 중복 외 유사 후보: "
    f"{(~integration_test_candidates['same_pixels']).sum():,}쌍"
)
display(integration_test_candidates.head(20))

print(f"\n검사 결과 저장 폴더: {INTEGRATION_CHECK_DIR}")
print("※ 유사 후보 기준: pHash 해밍 거리 5 이하")
print("※ 같은 조합도 포함하여 비교했습니다.")
print("※ pHash만으로 모든 변형 이미지의 중복을 찾을 수는 없습니다.")
print("※ Test는 비교 대상으로만 읽었으며 학습 데이터에 포함하지 않았습니다.")
print("※ 원본 이미지·라벨 수정 및 삭제 없음")

In [ ]:
# 후보 번호는 전체 Test 후보 목록 기준으로 고정
integration_test_review = (
    integration_test_candidates.reset_index(drop=True).copy()
)
integration_test_review.insert(
    0, "review_number", range(1, len(integration_test_review) + 1)
)


def show_integration_pairs(review_table, pairs_per_figure=4):
    if review_table.empty:
        print("표시할 후보가 없습니다.")
        return

    def read_review_image(dataset, image_path, codeit_zip, test_zip):
        if dataset == "select":
            with Image.open(SELECT_DIR / image_path) as image:
                return image.convert("RGB")

        source_zip = {
            "codeit": codeit_zip,
            "test": test_zip,
        }[dataset]

        with Image.open(BytesIO(source_zip.read(image_path))) as image:
            return image.convert("RGB")

    with zipfile.ZipFile(CODEIT_CLEAN_ZIP, "r") as codeit_zip:
        with zipfile.ZipFile(DATA_PATH, "r") as test_zip:
            for start in range(0, len(review_table), pairs_per_figure):
                batch = review_table.iloc[
                    start:start + pairs_per_figure
                ]

                fig, axes = plt.subplots(
                    len(batch), 2,
                    figsize=(12, 6 * len(batch)),
                    squeeze=False,
                )

                for row_index, record in enumerate(
                    batch.to_dict("records")
                ):
                    for column_index, side in enumerate(["a", "b"]):
                        ax = axes[row_index, column_index]

                        image_rgb = read_review_image(
                            record[f"dataset_{side}"],
                            record[f"image_path_{side}"],
                            codeit_zip,
                            test_zip,
                        )
                        ax.imshow(image_rgb)
                        ax.set_title(
                            f"Review {record['review_number']} | "
                            f"{record[f'dataset_{side}']} | "
                            f"Image {record[f'image_id_{side}']}\n"
                            f"pHash distance: {record['phash_distance']}",
                            fontsize=10,
                        )
                        ax.axis("off")

                plt.tight_layout()
                plt.show()


integration_test_zero_review = integration_test_review.loc[
    integration_test_review["phash_distance"].eq(0)
].copy()

print("========== 3-3. Test와 pHash 거리 0인 후보 검토 ==========")
print(f"검토 후보: {len(integration_test_zero_review):,}쌍")

show_integration_pairs(integration_test_zero_review)

print("※ 왼쪽: 학습 통합 후보 / 오른쪽: Test")
print("※ 파일 수정·삭제 없음")

In [ ]:
# Test 후보: 이미 확인한 거리 0을 제외하고 거리별 최대 4쌍
integration_test_sample_review = (
    integration_test_review.loc[
        integration_test_review["phash_distance"].gt(0)
    ]
    .groupby("phash_distance", group_keys=False)
    .head(4)
    .copy()
)

# SELECT와 대회 Train 후보: 거리별 최대 4쌍
integration_cross_review = (
    codeit_select_similar_candidates.reset_index(drop=True).copy()
)
integration_cross_review.insert(
    0, "review_number", range(1, len(integration_cross_review) + 1)
)

integration_cross_sample_review = (
    integration_cross_review
    .groupby("phash_distance", group_keys=False)
    .head(4)
    .copy()
)

print("========== 3-3. 남은 유사 후보 표본 검토 ==========")

print(
    f"\n통합 후보 ↔ Test: "
    f"{len(integration_test_sample_review)}쌍"
)
show_integration_pairs(integration_test_sample_review)

print(
    f"\nSELECT ↔ 대회 Train: "
    f"{len(integration_cross_sample_review)}쌍"
)
show_integration_pairs(integration_cross_sample_review)

print("※ 거리별 목록 앞부분을 확인하는 표본 검토입니다.")
print("※ 미검토 후보 전체에 동일한 결론을 확정하는 검사는 아닙니다.")
print("※ 파일 수정·삭제 없음")

### 3-3. 데이터 간 유사 이미지 및 Test 혼입 후보 검사 결과

- SELECT 10,109장, 대회 Train 215장, Test 842장을 비교함.
- SELECT와 대회 Train 사이 완전 중복: 0쌍, 유사 후보: 122쌍.
- 통합 후보와 Test 사이 완전 중복: 0쌍, 유사 후보: 558쌍.
- Test와 pHash 거리 0인 17쌍 전체 및 나머지 거리별 표본을 시각 검토함.
- SELECT와 대회 Train 사이 후보도 거리별 표본을 시각 검토함.
- 검토한 후보는 구도가 비슷하지만 알약 일부가 다른 별개 이미지로 확인됨.
- 이번 검사에 따른 추가 제외 없음.
- 유사 후보 전체를 시각 검토한 것은 아니며, pHash 검사로 모든 변형 이미지의 중복을 배제할 수는 없음.

## 3-4. 통합 라벨 구성 및 ID 충돌 처리

In [ ]:
from datetime import datetime
from collections import Counter

# 1. 저장된 공통 클래스 매핑 읽기
integration_class_mapping = pd.read_csv(
    CODEIT_RESULT_DIR / "class_mapping.csv",
    encoding="utf-8-sig",
)

integration_class_lookup = {
    int(record["category_id"]): record
    for record in integration_class_mapping.to_dict("records")
}

# 2. SELECT와 대회 데이터 연결
integration_inputs = []

for record in select_cross_hashes.sort_values(
    "json_file"
).to_dict("records"):
    integration_inputs.append({
        "dataset": "select",
        "original_json_file": record["json_file"],
        "image_path": record["image_path"],
        "document": select_kept_documents[record["json_file"]],
    })

with zipfile.ZipFile(CODEIT_CLEAN_ZIP, "r") as z:
    for record in codeit_export_manifest.sort_values(
        "json_file"
    ).to_dict("records"):
        integration_inputs.append({
            "dataset": "codeit",
            "original_json_file": record["json_file"],
            "image_path": record["image_path"],
            "document": json.loads(
                z.read(record["json_file"]).decode("utf-8")
            ),
        })

# 3. 통합 ID 부여
integration_documents = {}
integration_manifest_rows = []
integration_annotation_id_rows = []
integration_annotation_count = 0
integration_category_counts = Counter()

for new_image_id, item in enumerate(integration_inputs, start=1):
    dataset = item["dataset"]
    original_document = item["document"]

    if len(original_document["images"]) != 1:
        raise ValueError(
            f"이미지별 JSON 구조 확인 필요: {item['original_json_file']}"
        )

    original_image = original_document["images"][0]
    new_image = dict(original_image)
    new_image["id"] = new_image_id

    new_annotations = []
    image_categories = {}

    for original_annotation in original_document["annotations"]:
        if original_annotation["image_id"] != original_image["id"]:
            raise ValueError(
                f"원본 이미지 연결 불일치: {item['original_json_file']}"
            )

        category_id = int(original_annotation["category_id"])

        if category_id not in integration_class_lookup:
            raise ValueError(f"공통 매핑에 없는 클래스: {category_id}")

        mapping = integration_class_lookup[category_id]

        if (
            original_annotation["drug_code"] != mapping["code"]
            or int(original_annotation["model_class_id"])
            != int(mapping["model_class_id"])
        ):
            raise ValueError(
                f"클래스 매핑 불일치: {dataset}, "
                f"{item['original_json_file']}"
            )

        integration_annotation_count += 1

        new_annotation = dict(original_annotation)
        new_annotation["id"] = integration_annotation_count
        new_annotation["image_id"] = new_image_id
        new_annotation["category_id"] = category_id
        new_annotation["model_class_id"] = int(mapping["model_class_id"])

        new_annotations.append(new_annotation)
        integration_category_counts[category_id] += 1

        image_categories[category_id] = {
            "id": category_id,
            "name": mapping["code"],
        }

        integration_annotation_id_rows.append({
            "dataset": dataset,
            "original_json_file": item["original_json_file"],
            "original_image_id": original_image["id"],
            "original_annotation_id": original_annotation["id"],
            "image_id": new_image_id,
            "annotation_id": integration_annotation_count,
        })

    output_json_path = f"annotations/{new_image_id:06d}.json"

    integration_documents[output_json_path] = {
        "images": [new_image],
        "annotations": new_annotations,
        "categories": [
            image_categories[key] for key in sorted(image_categories)
        ],
        "source": dict(original_document["source"]),
    }

    integration_manifest_rows.append({
        "image_id": new_image_id,
        "dataset": dataset,
        "file_name": original_image["file_name"],
        "image_path": f"{dataset}/{item['image_path']}",
        "json_file": output_json_path,
        "original_image_id": original_image["id"],
        "original_json_file": item["original_json_file"],
        "annotation_count": len(new_annotations),
    })

integration_manifest = pd.DataFrame(integration_manifest_rows)
integration_annotation_id_map = pd.DataFrame(
    integration_annotation_id_rows
)

# 4. 수량과 ID 연결 검증
assert len(integration_manifest) == 10324
assert integration_annotation_count == 39608
assert integration_manifest["image_id"].is_unique
assert integration_manifest["image_path"].is_unique
assert integration_manifest["json_file"].is_unique
assert integration_annotation_id_map["annotation_id"].is_unique
assert (
    integration_manifest["annotation_count"].sum()
    == integration_annotation_count
)

for document in integration_documents.values():
    image_id = document["images"][0]["id"]
    assert all(
        annotation["image_id"] == image_id
        for annotation in document["annotations"]
    )

# 5. 통합 결과 저장
INTEGRATION_RESULT_DIR = DATA_PATH.parent.parent / (
    "integrated_dataset_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
INTEGRATION_RESULT_DIR.mkdir(parents=True, exist_ok=False)

integration_manifest.to_csv(
    INTEGRATION_RESULT_DIR / "integration_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

integration_annotation_id_map.to_csv(
    INTEGRATION_RESULT_DIR / "annotation_id_map.csv",
    index=False,
    encoding="utf-8-sig",
)

integration_class_mapping.to_csv(
    INTEGRATION_RESULT_DIR / "class_mapping.csv",
    index=False,
    encoding="utf-8-sig",
)

integration_class_distribution = pd.DataFrame([
    {
        "category_id": category_id,
        "drug_code": integration_class_lookup[category_id]["code"],
        "box_count": count,
    }
    for category_id, count in sorted(integration_category_counts.items())
])

integration_class_distribution.to_csv(
    INTEGRATION_RESULT_DIR / "class_distribution.csv",
    index=False,
    encoding="utf-8-sig",
)

# 사용 경로 및 검토 근거 기록
integration_readme = """# 통합 데이터 사용 안내

## 이미지 경로
integration_manifest.csv의 image_path를 기준으로 이미지를 읽습니다.

- select/images/...: 기존 SELECT 폴더의 이미지
- codeit/images/...: 검증 완료 대회 ZIP을 풀었을 때의 이미지

이미지별 JSON의 file_name은 기존 파일명을 유지합니다.
실제 이미지 위치는 사용 목록의 image_path로 연결합니다.
통합 라벨 ZIP에는 이미지가 포함되지 않습니다.

## ID 및 클래스
이미지 ID와 annotation ID를 통합 기준으로 새로 부여했습니다.
원본 ID 연결은 annotation_id_map.csv에 보존했습니다.
클래스 매핑과 bbox 좌표는 변경하지 않았습니다.

## 중복 및 Test 비교 결과
데이터 간 및 Test와의 완전 중복은 발견되지 않았습니다.
Test와 pHash 거리 0인 17쌍 및 나머지 거리별 표본을 검토했습니다.
SELECT와 대회 Train 후보도 거리별 표본을 검토했습니다.
검토한 후보는 구도가 비슷하지만 알약 일부가 다른 이미지였습니다.
이번 검사로 추가 제외하지 않았습니다.
유사 후보 전체를 시각 검토한 것은 아닙니다.

## 다음 단계
Train/Validation 분할 전 데이터입니다.
최종 분할 목록에 따라 이미지를 읽도록 구성해야 합니다.
"""

(INTEGRATION_RESULT_DIR / "README.md").write_text(
    integration_readme,
    encoding="utf-8",
)

# 6. 이미지 없이 통합 라벨과 목록만 ZIP 저장
INTEGRATION_LABEL_ZIP = (
    INTEGRATION_RESULT_DIR / "integrated_labels.zip"
)

integration_support_files = [
    "integration_manifest.csv",
    "annotation_id_map.csv",
    "class_mapping.csv",
    "class_distribution.csv",
    "README.md",
]

with zipfile.ZipFile(
    INTEGRATION_LABEL_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=1,
) as z:
    for json_path, document in integration_documents.items():
        z.writestr(
            json_path,
            json.dumps(
                document,
                ensure_ascii=False,
                indent=2,
                allow_nan=False,
            ).encode("utf-8"),
        )

    for filename in integration_support_files:
        z.write(
            INTEGRATION_RESULT_DIR / filename,
            arcname=filename,
        )

# 7. 저장 ZIP의 구성과 JSON 내용 확인
with zipfile.ZipFile(INTEGRATION_LABEL_ZIP, "r") as z:
    expected_members = (
        set(integration_documents) | set(integration_support_files)
    )
    assert set(z.namelist()) == expected_members

    bad_member = z.testzip()
    if bad_member is not None:
        raise ValueError(f"저장 ZIP 오류: {bad_member}")

    for json_path, expected_document in integration_documents.items():
        saved_document = json.loads(
            z.read(json_path).decode("utf-8")
        )
        assert saved_document == expected_document

print("========== 3-4. 통합 라벨 구성 및 ID 충돌 처리 ==========")
print(f"통합 이미지: {len(integration_manifest):,}장")
print(f"통합 annotation: {integration_annotation_count:,}개")
print(f"실제 포함 클래스: {len(integration_category_counts):,}종")
print("이미지 ID 및 annotation ID 중복 없음")
print("이미지와 annotation 연결 및 저장 ZIP 검사 통과")
print(f"\n저장 폴더: {INTEGRATION_RESULT_DIR}")
print(f"공유용 라벨 ZIP: {INTEGRATION_LABEL_ZIP}")
print(
    f"라벨 ZIP 크기: "
    f"{INTEGRATION_LABEL_ZIP.stat().st_size / 1024**2:.1f} MB"
)
print("\n※ 이미지 파일 복사 및 원본 JSON 수정 없음")
print("※ 클래스 매핑과 bbox 좌표 변경 없음")

## 4. Train/Validation 분할

### 4-1. 조합별 그룹 구성 및 분할 가능 여부 확인

In [ ]:
import re

split_image_rows = []
split_group_class_rows = []

for record in integration_manifest.to_dict("records"):
    document = integration_documents[record["json_file"]]

    # 접두사가 아닌 실제 알약 코드 조합 추출
    original_name = document["source"].get("original_image", "")
    match = re.search(
        r"K-\d{6}(?:-\d{6})*",
        original_name,
    )

    if match is None:
        match = re.search(
            r"K-\d{6}(?:-\d{6})*",
            record["file_name"],
        )

    if match is None:
        raise ValueError(
            f"조합 확인 불가: {record['file_name']}"
        )

    combination = tuple(sorted(
        int(code) for code in match.group().split("-")[1:]
    ))
    group_id = "K-" + "-".join(
        f"{code:06d}" for code in combination
    )

    class_ids = sorted({
        int(annotation["category_id"])
        for annotation in document["annotations"]
    })

    split_image_rows.append({
        **record,
        "group_id": group_id,
    })

    for category_id in class_ids:
        split_group_class_rows.append({
            "group_id": group_id,
            "category_id": category_id,
            "image_id": record["image_id"],
        })

split_image_manifest = pd.DataFrame(split_image_rows)
split_group_class_records = pd.DataFrame(split_group_class_rows)

# 같은 조합은 출처가 달라도 하나의 그룹으로 처리
split_group_summary = (
    split_image_manifest.groupby("group_id")
    .agg(
        image_count=("image_id", "size"),
        annotation_count=("annotation_count", "sum"),
        datasets=(
            "dataset",
            lambda values: " / ".join(sorted(set(values))),
        ),
    )
    .reset_index()
)

split_class_group_summary = (
    split_group_class_records.groupby("category_id")
    .agg(
        group_count=("group_id", "nunique"),
        image_count=("image_id", "nunique"),
    )
    .reset_index()
)

split_class_group_summary["drug_code"] = (
    split_class_group_summary["category_id"].map(
        lambda value: f"K-{int(value):06d}"
    )
)

split_single_group_classes = split_class_group_summary.loc[
    split_class_group_summary["group_count"].eq(1)
].copy()

assert len(split_image_manifest) == 10324
assert split_image_manifest["image_id"].is_unique
assert len(split_class_group_summary) == 118

# 재시작 후에도 그룹 구성부터 다시 계산하지 않도록 저장
split_image_manifest.to_csv(
    INTEGRATION_RESULT_DIR / "split_image_groups.csv",
    index=False,
    encoding="utf-8-sig",
)

split_group_summary.to_csv(
    INTEGRATION_RESULT_DIR / "split_group_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

split_class_group_summary.to_csv(
    INTEGRATION_RESULT_DIR / "split_class_group_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

print("========== 4-1. 조합별 그룹 구성 및 분할 가능 여부 ==========")
print(f"전체 이미지: {len(split_image_manifest):,}장")
print(f"전체 조합 그룹: {len(split_group_summary):,}개")
print(f"전체 클래스: {len(split_class_group_summary):,}종")
print(
    f"조합이 2개 이상인 클래스: "
    f"{split_class_group_summary['group_count'].ge(2).sum():,}종"
)
print(f"한 조합에만 있는 클래스: {len(split_single_group_classes):,}종")

if not split_single_group_classes.empty:
    print("\n========== 한 조합에만 있는 클래스 ==========")
    display(split_single_group_classes[[
        "drug_code", "group_count", "image_count"
    ]])

print("\n========== 그룹별 이미지 수 분포 ==========")
display(split_group_summary["image_count"].describe())

print(f"\n그룹 구성 저장 완료: {INTEGRATION_RESULT_DIR}")
print("※ 이미지와 라벨 원본 변경 없음")
print("※ 아직 Train/Validation 배정은 수행하지 않았습니다.")

### 4-2. Train/Validation 그룹 분할
- Train: 80%
- Validation: 20%
- 118종이 양쪽에 모두 포함되는 분할 찾기

In [ ]:
# 1. 분할 설정
split_seed = 42
split_validation_ratio = 0.2
split_search_trials = 1000

# 2. 그룹별 이미지 수와 클래스별 박스 수 구성
split_group_ids = sorted(
    split_image_manifest["group_id"].unique()
)
split_category_ids = sorted(
    int(value)
    for value in split_class_group_summary["category_id"]
)

split_group_index = {
    group_id: index
    for index, group_id in enumerate(split_group_ids)
}
split_category_index = {
    category_id: index
    for index, category_id in enumerate(split_category_ids)
}

split_group_image_counts = (
    split_image_manifest.groupby("group_id").size()
    .reindex(split_group_ids)
    .to_numpy(dtype=int)
)

split_group_box_counts = np.zeros(
    (len(split_group_ids), len(split_category_ids)),
    dtype=int,
)

for record in split_image_manifest.to_dict("records"):
    group_index = split_group_index[record["group_id"]]
    document = integration_documents[record["json_file"]]

    for annotation in document["annotations"]:
        category_index = split_category_index[
            int(annotation["category_id"])
        ]
        split_group_box_counts[group_index, category_index] += 1

split_total_class_boxes = split_group_box_counts.sum(axis=0)
split_target_validation_images = (
    len(split_image_manifest) * split_validation_ratio
)

# 3. 여러 그룹 배정안 비교
split_rng = np.random.default_rng(split_seed)
split_best_score = float("inf")
split_best_validation_indices = None
split_feasible_trials = 0

for trial in range(split_search_trials):
    order = split_rng.permutation(len(split_group_ids))

    cumulative_images = np.cumsum(
        split_group_image_counts[order]
    )

    cut = int(np.argmin(
        np.abs(cumulative_images - split_target_validation_images)
    )) + 1

    validation_indices = order[:cut]

    validation_class_boxes = (
        split_group_box_counts[validation_indices].sum(axis=0)
    )
    train_class_boxes = (
        split_total_class_boxes - validation_class_boxes
    )

    feasible = (
        np.all(validation_class_boxes > 0)
        and np.all(train_class_boxes > 0)
    )

    if feasible:
        split_feasible_trials += 1

        class_validation_ratios = (
            validation_class_boxes / split_total_class_boxes
        )
        image_validation_ratio = (
            split_group_image_counts[validation_indices].sum()
            / len(split_image_manifest)
        )

        score = (
            np.mean(
                (class_validation_ratios - split_validation_ratio) ** 2
            )
            + (image_validation_ratio - split_validation_ratio) ** 2
        )

        if score < split_best_score:
            split_best_score = score
            split_best_validation_indices = validation_indices.copy()

    if (trial + 1) % 250 == 0:
        print(
            f"분할안 비교: {trial + 1:,}/{split_search_trials:,}회 "
            f"| 양쪽 클래스 포함 조건 충족: {split_feasible_trials:,}회"
        )

if split_best_validation_indices is None:
    raise RuntimeError(
        "모든 클래스가 양쪽에 포함되는 배정안을 찾지 못했습니다. "
        "파일은 저장하지 않았습니다. 그룹별 제약을 확인해야 합니다."
    )

# 4. 최종 배정
split_validation_groups = {
    split_group_ids[index]
    for index in split_best_validation_indices
}

split_final_manifest = split_image_manifest.copy()
split_final_manifest["split"] = np.where(
    split_final_manifest["group_id"].isin(split_validation_groups),
    "validation",
    "train",
)

split_train_manifest = split_final_manifest.loc[
    split_final_manifest["split"].eq("train")
].copy()

split_validation_manifest = split_final_manifest.loc[
    split_final_manifest["split"].eq("validation")
].copy()

# 5. 클래스별 이미지 수와 박스 수 집계
split_class_count_rows = []

for record in split_final_manifest.to_dict("records"):
    document = integration_documents[record["json_file"]]

    for annotation in document["annotations"]:
        split_class_count_rows.append({
            "split": record["split"],
            "image_id": record["image_id"],
            "category_id": int(annotation["category_id"]),
        })

split_class_count_records = pd.DataFrame(split_class_count_rows)

split_class_distribution = pd.DataFrame(
    index=pd.Index(split_category_ids, name="category_id")
)

for split_name in ["train", "validation"]:
    subset = split_class_count_records.loc[
        split_class_count_records["split"].eq(split_name)
    ]

    split_class_distribution[f"{split_name}_boxes"] = (
        subset.groupby("category_id").size()
    )
    split_class_distribution[f"{split_name}_images"] = (
        subset.groupby("category_id")["image_id"].nunique()
    )

split_class_distribution = (
    split_class_distribution.fillna(0).astype(int)
)

split_class_distribution["validation_box_pct"] = (
    split_class_distribution["validation_boxes"]
    / (
        split_class_distribution["train_boxes"]
        + split_class_distribution["validation_boxes"]
    )
    * 100
).round(2)

# 6. 分割条件検証
train_image_ids = set(split_train_manifest["image_id"])
validation_image_ids = set(split_validation_manifest["image_id"])

train_group_ids = set(split_train_manifest["group_id"])
validation_group_ids = set(split_validation_manifest["group_id"])

assert train_image_ids.isdisjoint(validation_image_ids)
assert train_group_ids.isdisjoint(validation_group_ids)
assert len(train_image_ids | validation_image_ids) == 10324
assert split_final_manifest["image_id"].is_unique
assert split_class_distribution["train_boxes"].gt(0).all()
assert split_class_distribution["validation_boxes"].gt(0).all()
assert split_final_manifest["annotation_count"].sum() == 39608

# 7. 結果保存
SPLIT_RESULT_DIR = INTEGRATION_RESULT_DIR / (
    "split_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
SPLIT_RESULT_DIR.mkdir(parents=True, exist_ok=False)

for filename, table in [
    ("split_manifest.csv", split_final_manifest),
    ("train_manifest.csv", split_train_manifest),
    ("validation_manifest.csv", split_validation_manifest),
]:
    table.to_csv(
        SPLIT_RESULT_DIR / filename,
        index=False,
        encoding="utf-8-sig",
    )

split_class_distribution.to_csv(
    SPLIT_RESULT_DIR / "split_class_distribution.csv",
    encoding="utf-8-sig",
)

split_settings = {
    "seed": split_seed,
    "target_validation_ratio": split_validation_ratio,
    "search_trials": split_search_trials,
    "feasible_trials": split_feasible_trials,
    "selected_score": float(split_best_score),
    "group_definition": "파일명에 기록된 알약 코드 조합",
    "train_images": len(split_train_manifest),
    "validation_images": len(split_validation_manifest),
    "train_groups": len(train_group_ids),
    "validation_groups": len(validation_group_ids),
    "classes_in_both": len(split_class_distribution),
}

(SPLIT_RESULT_DIR / "split_settings.json").write_text(
    json.dumps(split_settings, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\n========== 4-2. Train/Validation 그룹 분할 완료 ==========")
print(
    f"Train: {len(split_train_manifest):,}장 "
    f"({len(split_train_manifest) / len(split_final_manifest):.2%})"
)
print(
    f"Validation: {len(split_validation_manifest):,}장 "
    f"({len(split_validation_manifest) / len(split_final_manifest):.2%})"
)
print(
    f"Train annotation: "
    f"{split_train_manifest['annotation_count'].sum():,}개"
)
print(
    f"Validation annotation: "
    f"{split_validation_manifest['annotation_count'].sum():,}개"
)
print(f"Train 그룹: {len(train_group_ids):,}개")
print(f"Validation 그룹: {len(validation_group_ids):,}개")
print(f"양쪽 모두 포함된 클래스: {len(split_class_distribution):,}종")
print("이미지 ID 중복: 0개")
print("조합 그룹 중복: 0개")

print("\n========== Validation 박스 수가 적은 클래스: 최대 20종 ==========")
display(
    split_class_distribution.sort_values("validation_boxes").head(20)
)

print(f"\n분할 결과 저장 폴더: {SPLIT_RESULT_DIR}")
print("※ 탐색한 배정안 중 클래스별 목표 비율에 가장 가까운 안을 선택했습니다.")
print("※ 이미지 복사 및 원본 라벨 수정 없음")
print("※ 서로 다른 조합 사이의 모든 시각적 유사성까지 제거한 것은 아닙니다.")

### 4-2. Train/Validation 그룹 분할 결과

- 동일 알약 조합을 하나의 그룹으로 묶어 분할함.
- 난수 시드 42, 목표 비율 80:20으로 1,000개 배정안을 비교함.
- 118종 모두 양쪽에 포함되는 배정안 중 클래스별 박스 비율을 고려하여 선택함.
- Train: 8,260장, 31,689개 박스.
- Validation: 2,064장, 7,919개 박스.
- 이미지 ID 및 조합 그룹 중복 없음.
- Validation 박스 수가 가장 적은 클래스는 K-003483으로 12개임.

## 5. 데이터 로드

### 5-1. 공통 데이터와 모델별 로딩 방식 구성
아래 3개의 탐지기를 기준으로 로딩 방식 구성
- Faster R-CNN ResNet50-FPN
- RetinaNet ResNet50-FPN
- YOLOX-S

| 공통으로 고정할 것 | 모델에 맞게 처리할 것 |
|---|---|
| Train 8,260장·Validation 2,064장 | Dataset·DataLoader |
| 클래스 118종과 공통 매핑 | 모델 내부 클래스 번호 |
| bbox와 제외 대상 | bbox 입력 형식·전처리 |
| 평가 대상·평가 지표 | 모델 출력의 공통 평가 형식 변환 |

In [ ]:
import importlib.util

# 1. 공통 학습 파일 구성
TRAINING_PACKAGE_DIR = SPLIT_RESULT_DIR / "training_package"
(TRAINING_PACKAGE_DIR / "annotations").mkdir(
    parents=True, exist_ok=True
)

training_mapping = integration_class_mapping.sort_values(
    "category_id"
).copy()

assert training_mapping["model_class_id"].astype(int).tolist() == (
    list(range(118))
)

training_categories = [
    {
        "id": int(record["category_id"]),
        "name": record["code"],
        "supercategory": "pill",
    }
    for record in training_mapping.to_dict("records")
]

for split_name, manifest in [
    ("train", split_train_manifest),
    ("validation", split_validation_manifest),
]:
    images = []
    annotations = []

    for record in manifest.to_dict("records"):
        document = integration_documents[record["json_file"]]
        image_info = dict(document["images"][0])
        image_info["file_name"] = record["image_path"]
        images.append(image_info)

        for original_annotation in document["annotations"]:
            annotation = dict(original_annotation)
            annotation["segmentation"] = []
            annotations.append(annotation)

    coco_document = {
        "info": {"description": "Pill detection fixed group split"},
        "licenses": [],
        "images": images,
        "annotations": annotations,
        "categories": training_categories,
    }

    (
        TRAINING_PACKAGE_DIR / "annotations" / f"{split_name}.json"
    ).write_text(
        json.dumps(
            coco_document,
            ensure_ascii=False,
            allow_nan=False,
        ),
        encoding="utf-8",
    )

training_mapping.to_csv(
    TRAINING_PACKAGE_DIR / "class_mapping.csv",
    index=False,
    encoding="utf-8-sig",
)

for filename in [
    "split_manifest.csv",
    "split_class_distribution.csv",
    "split_settings.json",
]:
    (TRAINING_PACKAGE_DIR / filename).write_bytes(
        (SPLIT_RESULT_DIR / filename).read_bytes()
    )

# 2. 대회 이미지만 로컬에 준비
CODEIT_IMAGE_ROOT = CODEIT_RESULT_DIR / "extracted_images"

with zipfile.ZipFile(CODEIT_CLEAN_ZIP, "r") as z:
    for record in codeit_export_manifest.to_dict("records"):
        relative_path = Path(record["image_path"])

        if relative_path.is_absolute() or ".." in relative_path.parts:
            raise ValueError("안전하지 않은 이미지 경로입니다.")

        destination = CODEIT_IMAGE_ROOT / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)

        if not destination.exists():
            destination.write_bytes(z.read(record["image_path"]))

# 3. 팀원들이 재사용할 로더 모듈
loader_source = '''
import json
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader


def resolve_image_path(file_name, image_roots):
    parts = file_name.split("/", 1)
    if len(parts) != 2 or parts[0] not in image_roots:
        raise ValueError(f"이미지 출처 확인 필요: {file_name}")
    path = Path(image_roots[parts[0]]) / parts[1]
    if not path.is_file():
        raise FileNotFoundError(path)
    return path


class PillDataset(Dataset):
    def __init__(self, annotation_path, image_roots):
        with open(annotation_path, encoding="utf-8") as stream:
            self.data = json.load(stream)

        self.images = sorted(self.data["images"], key=lambda x: x["id"])
        self.image_roots = image_roots
        self.annotations = defaultdict(list)

        for annotation in self.data["annotations"]:
            self.annotations[annotation["image_id"]].append(annotation)

        category_ids = sorted(
            category["id"] for category in self.data["categories"]
        )
        self.category_to_label = {
            category_id: index + 1
            for index, category_id in enumerate(category_ids)
        }
        self.label_to_category = {
            label: category_id
            for category_id, label in self.category_to_label.items()
        }

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image_info = self.images[index]
        image_path = resolve_image_path(
            image_info["file_name"], self.image_roots
        )

        with Image.open(image_path) as image:
            rgb = np.array(image.convert("RGB"), copy=True)

        image_tensor = (
            torch.from_numpy(rgb)
            .permute(2, 0, 1)
            .contiguous()
            .float()
            / 255.0
        )

        records = self.annotations[image_info["id"]]
        boxes = []
        labels = []

        for annotation in records:
            x, y, width, height = annotation["bbox"]
            boxes.append([x, y, x + width, y + height])
            labels.append(
                self.category_to_label[annotation["category_id"]]
            )

        target = {
            "boxes": torch.tensor(
                boxes, dtype=torch.float32
            ).reshape(-1, 4),
            "labels": torch.tensor(labels, dtype=torch.int64),
            "image_id": torch.tensor(
                image_info["id"], dtype=torch.int64
            ),
            "area": torch.tensor(
                [a["area"] for a in records], dtype=torch.float32
            ),
            "iscrowd": torch.tensor(
                [a.get("iscrowd", 0) for a in records],
                dtype=torch.int64,
            ),
        }
        return image_tensor, target


def detection_collate(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)


def make_torchvision_loaders(
    package_dir, image_roots, batch_size=2, num_workers=0, seed=42
):
    package_dir = Path(package_dir)
    datasets = {
        split: PillDataset(
            package_dir / "annotations" / f"{split}.json",
            image_roots,
        )
        for split in ["train", "validation"]
    }

    generator = torch.Generator().manual_seed(seed)
    loaders = {
        split: DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=(split == "train"),
            num_workers=num_workers,
            collate_fn=detection_collate,
            generator=generator if split == "train" else None,
            drop_last=False,
        )
        for split, dataset in datasets.items()
    }
    return datasets, loaders


def prepare_yolox_annotations(package_dir, image_roots):
    package_dir = Path(package_dir)
    runtime_dir = package_dir / "runtime_yolox"
    (runtime_dir / "annotations").mkdir(
        parents=True, exist_ok=True
    )

    for split in ["train", "validation"]:
        with open(
            package_dir / "annotations" / f"{split}.json",
            encoding="utf-8",
        ) as stream:
            data = json.load(stream)

        # 각 실행 환경의 이미지 경로에 연결
        for image in data["images"]:
            image["file_name"] = str(
                resolve_image_path(
                    image["file_name"], image_roots
                ).resolve()
            )

        with open(
            runtime_dir / "annotations" / f"{split}.json",
            "w", encoding="utf-8",
        ) as stream:
            json.dump(data, stream, ensure_ascii=False)

    return runtime_dir


def make_yolox_loaders(
    package_dir, image_roots, batch_size=2,
    num_workers=0, image_size=(640, 640)
):
    from yolox.data import (
        COCODataset, DataLoader as YoloDataLoader,
        TrainTransform, ValTransform,
    )

    runtime_dir = prepare_yolox_annotations(
        package_dir, image_roots
    )

    datasets = {
        "train": COCODataset(
            data_dir=str(runtime_dir),
            json_file="train.json",
            name="",
            img_size=image_size,
            preproc=TrainTransform(
                max_labels=50, flip_prob=0.0, hsv_prob=0.0
            ),
        ),
        "validation": COCODataset(
            data_dir=str(runtime_dir),
            json_file="validation.json",
            name="",
            img_size=image_size,
            preproc=ValTransform(legacy=False),
        ),
    }

    loaders = {
        split: YoloDataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=(split == "train"),
            num_workers=num_workers,
            drop_last=False,
        )
        for split, dataset in datasets.items()
    }
    return datasets, loaders
'''

# 주석을 포함해 허용한 문자 범위 검사
# 한글, 영문, 숫자 및 일반 기호 외의 문자 혼입 방지
loader_lines = loader_source.splitlines()
loader_lines = [
    line
    for line in loader_lines
    if not any(
        "\u3040" <= character <= "\u30ff"
        or "\u3400" <= character <= "\u9fff"
        for character in line
    )
]
loader_source = "\n".join(loader_lines) + "\n"

loader_path = TRAINING_PACKAGE_DIR / "pill_data.py"
loader_path.write_text(loader_source, encoding="utf-8")

readme = """# 공통 학습 데이터

## 공유 자료
1. 기존 SELECT 폴더
2. codeit_train_validated.zip
3. training_package.zip

기존 SELECT를 다시 업로드할 필요는 없습니다.
대회 ZIP에서는 images 폴더를 사용합니다.
최종 학습 대상은 이 패키지의 분할 JSON으로 고정됩니다.

## 경로 설정
image_roots의 select는 기존 SELECT 폴더를 가리킵니다.
image_roots의 codeit는 대회 images 폴더의 상위 폴더를 가리킵니다.

## 클래스
공통 category_id는 원본 알약 코드의 숫자입니다.
TorchVision 로더의 labels는 1~118이며 0은 배경입니다.
이 로더를 사용하는 TorchVision 모델은 출력 클래스 수를 119로 설정합니다.
YOLOX는 클래스 118개이며 COCO category_id 순서로 연결됩니다.
평가할 때 모델 출력 번호를 공통 category_id로 되돌려야 합니다.

## 로더
Faster R-CNN과 RetinaNet:
make_torchvision_loaders 사용.

YOLOX:
YOLOX와 해당 의존성이 설치된 환경에서 make_yolox_loaders 사용.
YOLOX 학습 실험 설정에 이 분할과 클래스 수를 연결해야 합니다.
제공된 기본 로더에는 Mosaic과 MixUp을 적용하지 않습니다.

## 비교 조건
분할은 다시 생성하지 않습니다.
증강, 전처리 및 학습 조건은 실험 기록에 남깁니다.
현재 기본 로더는 추가 증강 없이 구성했습니다.
로더 확인 통과는 모델 학습 전체의 성공을 의미하지 않습니다.
"""

(TRAINING_PACKAGE_DIR / "README.md").write_text(
    readme, encoding="utf-8"
)

# 4. 이미지 없는 공유용 ZIP 저장
TRAINING_PACKAGE_ZIP = SPLIT_RESULT_DIR / "training_package.zip"

package_files = [
    TRAINING_PACKAGE_DIR / "annotations" / "train.json",
    TRAINING_PACKAGE_DIR / "annotations" / "validation.json",
    TRAINING_PACKAGE_DIR / "class_mapping.csv",
    TRAINING_PACKAGE_DIR / "split_manifest.csv",
    TRAINING_PACKAGE_DIR / "split_class_distribution.csv",
    TRAINING_PACKAGE_DIR / "split_settings.json",
    TRAINING_PACKAGE_DIR / "pill_data.py",
    TRAINING_PACKAGE_DIR / "README.md",
]

with zipfile.ZipFile(
    TRAINING_PACKAGE_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as z:
    for path in package_files:
        z.write(
            path,
            arcname=path.relative_to(
                TRAINING_PACKAGE_DIR
            ).as_posix(),
        )

with zipfile.ZipFile(TRAINING_PACKAGE_ZIP, "r") as z:
    assert z.testzip() is None

print("공통 학습 패키지 저장 완료")
print(f"공유용 ZIP: {TRAINING_PACKAGE_ZIP}")

# 5. 현재 환경에서 공통 로더 실행
spec = importlib.util.spec_from_file_location(
    "pill_data", loader_path
)
pill_data = importlib.util.module_from_spec(spec)

try:
    spec.loader.exec_module(pill_data)
except ModuleNotFoundError as error:
    raise RuntimeError(
        "공통 패키지 저장은 완료됐습니다. "
        f"로더 실행에 필요한 모듈이 없습니다: {error.name}. "
        "해당 모듈이 설치된 학습 환경에서 실행해야 합니다."
    ) from error

image_roots = {
    "select": SELECT_DIR,
    "codeit": CODEIT_IMAGE_ROOT,
}

# 전체 학습 이미지 경로 확인
for split_name in ["train", "validation"]:
    data = json.loads(
        (
            TRAINING_PACKAGE_DIR
            / "annotations"
            / f"{split_name}.json"
        ).read_text(encoding="utf-8")
    )

    for image in data["images"]:
        pill_data.resolve_image_path(
            image["file_name"], image_roots
        )

datasets, loaders = pill_data.make_torchvision_loaders(
    TRAINING_PACKAGE_DIR,
    image_roots,
    batch_size=2,
    num_workers=0,
)

# 각 분할에서 첫 배치 확인
for split_name in ["train", "validation"]:
    images, targets = next(iter(loaders[split_name]))

    for image, target in zip(images, targets):
        boxes = target["boxes"]
        labels = target["labels"]

        assert image.shape[0] == 3
        assert image.dtype == pill_data.torch.float32
        assert boxes.shape[1] == 4
        assert len(boxes) == len(labels)
        assert len(boxes) > 0
        assert (boxes[:, 2:] > boxes[:, :2]).all()
        assert labels.min().item() >= 1
        assert labels.max().item() <= 118

        height, width = image.shape[-2:]
        assert (boxes[:, :2] >= 0).all()
        assert (boxes[:, 2] <= width).all()
        assert (boxes[:, 3] <= height).all()

    print(f"\n{split_name}: {len(datasets[split_name]):,}장")
    print(f"배치 이미지 수: {len(images)}")
    print(f"첫 이미지 크기: {tuple(images[0].shape)}")
    print(f"첫 이미지 박스 수: {len(targets[0]['boxes'])}")

print("\n========== 5-1. 공통 데이터 로드 확인 완료 ==========")
print("전체 이미지 경로 확인 완료")
print("Train·Validation 첫 배치 읽기 및 기본 라벨 검사 통과")
print("YOLOX 로더 함수는 제공했으며 현재 셀에서는 실행하지 않았습니다.")
print(f"공유용 ZIP: {TRAINING_PACKAGE_ZIP}")
print("※ 원본 이미지 및 라벨 수정 없음")

### 5-3. Train/Validation 분포 비교

In [ ]:
# 1. 저장된 Train/Validation 라벨 읽기
distribution_rows = []
distribution_summary_rows = []

for split_name in ["train", "validation"]:
    document = json.loads(
        (
            TRAINING_PACKAGE_DIR
            / "annotations"
            / f"{split_name}.json"
        ).read_text(encoding="utf-8")
    )

    image_lookup = {
        image["id"]: image for image in document["images"]
    }

    distribution_summary_rows.append({
        "split": split_name,
        "images": len(document["images"]),
        "boxes": len(document["annotations"]),
        "classes": len({
            annotation["category_id"]
            for annotation in document["annotations"]
        }),
    })

    for annotation in document["annotations"]:
        image = image_lookup[annotation["image_id"]]
        x, y, w, h = annotation["bbox"]

        distribution_rows.append({
            "split": split_name,
            "image_id": annotation["image_id"],
            "category_id": annotation["category_id"],
            "drug_code": annotation["drug_code"],
            "width": w,
            "height": h,
            "area": w * h,
            "relative_area_pct": (
                w * h / (image["width"] * image["height"]) * 100
            ),
            "aspect_ratio": w / h,
        })

distribution_boxes = pd.DataFrame(distribution_rows)
distribution_summary = pd.DataFrame(
    distribution_summary_rows
).set_index("split")

for field in ["images", "boxes"]:
    distribution_summary[f"{field}_pct"] = (
        distribution_summary[field]
        / distribution_summary[field].sum() * 100
    ).round(2)

# 2. 색상 메타데이터 연결
color_values_by_code = {}

source_metadata_path = CODEIT_RESULT_DIR / "source_metadata.json"

if source_metadata_path.is_file():
    metadata = json.loads(
        source_metadata_path.read_text(encoding="utf-8")
    )

    for image in metadata["images"]:
        code = image.get("drug_N")
        colors = []

        for field in ["color_class1", "color_class2"]:
            value = image.get(field)
            if isinstance(value, str) and value.strip():
                colors.append(value.strip())

        if code and colors:
            color_values_by_code.setdefault(code, set()).add(
                " + ".join(sorted(set(colors)))
            )

color_lookup = {
    code: (
        next(iter(values))
        if len(values) == 1
        else "정보 충돌"
    )
    for code, values in color_values_by_code.items()
}

distribution_boxes["color"] = (
    distribution_boxes["drug_code"]
    .map(color_lookup)
    .fillna("정보 없음")
)

color_counts = pd.crosstab(
    distribution_boxes["color"],
    distribution_boxes["split"],
).reindex(columns=["train", "validation"], fill_value=0)

color_percentages = color_counts.div(
    color_counts.sum(axis=0), axis=1
) * 100

color_coverage = (
    distribution_boxes.assign(
        color_available=~distribution_boxes["color"].isin(
            ["정보 없음", "정보 충돌"]
        )
    )
    .groupby("split")
    .agg(
        boxes=("image_id", "size"),
        boxes_with_color=("color_available", "sum"),
    )
)

color_coverage["coverage_pct"] = (
    color_coverage["boxes_with_color"]
    / color_coverage["boxes"] * 100
).round(2)

# 3. 클래스별 배정 비율
distribution_class_counts = pd.crosstab(
    distribution_boxes["drug_code"],
    distribution_boxes["split"],
).reindex(columns=["train", "validation"], fill_value=0)

distribution_class_counts["validation_pct"] = (
    distribution_class_counts["validation"]
    / distribution_class_counts[["train", "validation"]].sum(axis=1)
    * 100
).round(2)

print("========== 5-3. Train/Validation 분포 비교 ==========")
print("\n[이미지·박스·클래스 수]")
display(distribution_summary)

print("\n[박스 크기와 종횡비]")
display(
    distribution_boxes.groupby("split")[[
        "width", "height", "area",
        "relative_area_pct", "aspect_ratio",
    ]].agg(["mean", "median"])
    .round(2)
)

print("\n[색상 정보 연결 범위]")
display(color_coverage)

print("\n[클래스별 Validation 박스 비율]")
display(
    distribution_class_counts["validation_pct"].describe().round(2)
)

# 4. 그래프
distribution_colors = {
    "train": "steelblue",
    "validation": "darkorange",
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, field, title in [
    (axes[0, 0], "images", "Image count"),
    (axes[0, 1], "boxes", "Box count"),
]:
    values = distribution_summary.loc[
        ["train", "validation"], field
    ]
    ax.bar(
        values.index,
        values.values,
        color=[distribution_colors[key] for key in values.index],
    )
    for index, (split_name, count) in enumerate(values.items()):
        percentage = distribution_summary.loc[
            split_name, f"{field}_pct"
        ]
        ax.text(
            index, count,
            f"{count:,}\n{percentage:.2f}%",
            ha="center", va="bottom",
        )
    ax.set_title(title)
    ax.margins(y=0.18)

# 標本数の違いを補正
for ax, field, title in [
    (axes[0, 2], "relative_area_pct", "Box area / image area (%)"),
    (axes[1, 0], "aspect_ratio", "Box aspect ratio (width / height)"),
]:
    pooled = distribution_boxes[field]
    bins = np.linspace(pooled.min(), pooled.max(), 31)

    for split_name in ["train", "validation"]:
        values = distribution_boxes.loc[
            distribution_boxes["split"].eq(split_name), field
        ]
        ax.hist(
            values,
            bins=bins,
            weights=np.full(len(values), 100 / len(values)),
            histtype="step",
            linewidth=2,
            color=distribution_colors[split_name],
            label=split_name,
        )

    ax.set_title(title)
    ax.set_ylabel("Boxes (%)")
    ax.legend()

color_plot = color_percentages.copy()
color_plot.index = [
    f"C{index + 1}" for index in range(len(color_plot))
]
color_plot.plot.bar(
    ax=axes[1, 1],
    color=[
        distribution_colors["train"],
        distribution_colors["validation"],
    ],
    rot=0,
)
axes[1, 1].set_title("Color composition")
axes[1, 1].set_ylabel("Boxes (%)")

axes[1, 2].hist(
    distribution_class_counts["validation_pct"],
    bins=np.arange(0, 105, 5),
    color="slateblue",
    edgecolor="white",
)
axes[1, 2].axvline(
    20, color="red", linestyle="--", label="Target 20%"
)
axes[1, 2].set_title("Validation share by class")
axes[1, 2].set_xlabel("Validation boxes / all boxes (%)")
axes[1, 2].set_ylabel("Classes")
axes[1, 2].legend()

plt.tight_layout()

# 5. 비교 결과 저장
DISTRIBUTION_DIR = SPLIT_RESULT_DIR / "distribution_review"
DISTRIBUTION_DIR.mkdir(parents=True, exist_ok=True)

fig.savefig(
    DISTRIBUTION_DIR / "split_distribution.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()

color_legend = color_percentages.copy()
color_legend.insert(
    0, "graph_code",
    [f"C{index + 1}" for index in range(len(color_legend))],
)

print("\n[색상 그래프 번호 설명: 값은 각 분할 내 박스 비율(%)]")
display(color_legend.round(2))

distribution_summary.to_csv(
    DISTRIBUTION_DIR / "summary.csv",
    encoding="utf-8-sig",
)
distribution_boxes.to_csv(
    DISTRIBUTION_DIR / "box_details.csv",
    index=False,
    encoding="utf-8-sig",
)
distribution_class_counts.to_csv(
    DISTRIBUTION_DIR / "class_distribution.csv",
    encoding="utf-8-sig",
)
color_legend.to_csv(
    DISTRIBUTION_DIR / "color_distribution.csv",
    encoding="utf-8-sig",
)

print(f"\n분포 비교 결과 저장: {DISTRIBUTION_DIR}")
print("※ 색상은 클래스 메타데이터이며 이미지 픽셀에서 측정한 색상이 아닙니다.")
print("※ 색상 정보가 없는 클래스도 유지하며 별도로 표시했습니다.")
print("※ 박스 크기·색상 그래프는 각 분할 내 비율로 비교합니다.")
print("※ 이번 분석은 분포 확인이며 기존 분할을 변경하지 않습니다.")

In [ ]:
# 기존 클래스별 박스 수 집계 재사용
class_box_plot = distribution_class_counts[
    ["train", "validation"]
].copy()

class_box_plot["total"] = (
    class_box_plot["train"] + class_box_plot["validation"]
)
class_box_plot = class_box_plot.sort_values(
    ["total", "train"],
    ascending=False,
)

class_box_ratios = (
    class_box_plot[["train", "validation"]]
    .div(class_box_plot["total"], axis=0)
    * 100
)

positions = np.arange(len(class_box_plot))
figure_height = max(12, len(class_box_plot) * 0.28)

fig, (ax_count, ax_ratio) = plt.subplots(
    1, 2,
    figsize=(18, figure_height),
    sharey=True,
    gridspec_kw={"width_ratios": [2, 1]},
)

# 왼쪽: 실제 박스 수
ax_count.barh(
    positions,
    class_box_plot["train"],
    color="steelblue",
    label="Train",
)
ax_count.barh(
    positions,
    class_box_plot["validation"],
    left=class_box_plot["train"],
    color="darkorange",
    label="Validation",
)

for position, (_, row) in enumerate(class_box_plot.iterrows()):
    ax_count.text(
        row["total"] + class_box_plot["total"].max() * 0.01,
        position,
        f"{int(row['train'])} / {int(row['validation'])}",
        va="center",
        fontsize=7,
    )

ax_count.set_yticks(positions)
ax_count.set_yticklabels(class_box_plot.index, fontsize=8)
ax_count.invert_yaxis()
ax_count.set_xlabel("Number of boxes")
ax_count.set_ylabel("Drug code")
ax_count.set_title("Boxes by class | Labels: Train / Validation")
ax_count.set_xlim(0, class_box_plot["total"].max() * 1.22)
ax_count.legend(loc="lower right")
ax_count.grid(axis="x", alpha=0.2)
ax_count.set_axisbelow(True)

# 오른쪽: 각 클래스 내 배정 비율
ax_ratio.barh(
    positions,
    class_box_ratios["train"],
    color="steelblue",
)
ax_ratio.barh(
    positions,
    class_box_ratios["validation"],
    left=class_box_ratios["train"],
    color="darkorange",
)

ax_ratio.axvline(
    80,
    color="black",
    linestyle="--",
    linewidth=1,
    label="Target Train 80%",
)

for position, percentage in enumerate(
    class_box_ratios["validation"]
):
    ax_ratio.text(
        101,
        position,
        f"{percentage:.1f}%",
        va="center",
        fontsize=7,
    )

ax_ratio.set_xlim(0, 114)
ax_ratio.set_xticks([0, 20, 40, 60, 80, 100])
ax_ratio.set_xlabel("Share within each class (%)")
ax_ratio.set_title("Split ratio | Labels: Validation %")
ax_ratio.legend(loc="lower right")

fig.suptitle(
    "Train / Validation class distribution — box counts",
    fontsize=16,
    y=1.002,
)

plt.tight_layout()

DISTRIBUTION_DIR.mkdir(parents=True, exist_ok=True)

class_graph_path = (
    DISTRIBUTION_DIR / "class_box_distribution.png"
)
class_graph_pdf_path = (
    DISTRIBUTION_DIR / "class_box_distribution.pdf"
)

fig.savefig(
    class_graph_path,
    dpi=180,
    bbox_inches="tight",
)
fig.savefig(
    class_graph_pdf_path,
    bbox_inches="tight",
)

plt.show()

print(f"전체 {len(class_box_plot)}종 표시 완료")
print("파랑: Train / 주황: Validation")
print("왼쪽 숫자: Train 박스 수 / Validation 박스 수")
print("오른쪽 숫자: 해당 클래스의 Validation 박스 비율")
print(f"\nPNG 저장: {class_graph_path}")
print(f"PDF 저장: {class_graph_pdf_path}")
print("※ 기존 분할과 라벨은 변경하지 않았습니다.")

## 6. 데이터 공유를 위한 분석

### 6-1. 데이터 검증에서 제외된 클래스 정리

In [ ]:
exclusion_audit_paths = {
    "대회": CODEIT_RESULT_DIR / "exclusion_reasons.csv",
    "SELECT": SELECT_MANIFEST_PATH.parent / "exclusion_reasons.csv",
}

print("========== 6-1. 데이터 검증에서 제외된 클래스 정리 준비 ==========")

for dataset_name, path in exclusion_audit_paths.items():
    print(f"\n[{dataset_name}]")
    print(f"파일: {path}")

    if not path.is_file():
        print("해당 위치에서 제외 사유 파일을 찾지 못했습니다.")
        continue

    table = pd.read_csv(path, encoding="utf-8-sig")

    print(f"기록 수: {len(table):,}개")
    print(f"열 구성: {table.columns.tolist()}")

    for record in table.head(3).to_dict("records"):
        print(json.dumps(
            record,
            ensure_ascii=False,
            indent=2,
            default=str,
        ))

print("\n※ 파일 읽기만 수행했으며 변경 사항은 없습니다.")

In [ ]:
# 1. 저장된 제외 근거 읽기
audit_codeit_reasons = pd.read_csv(
    CODEIT_RESULT_DIR / "exclusion_reasons.csv",
    encoding="utf-8-sig",
)
audit_select_reasons = pd.read_csv(
    SELECT_MANIFEST_PATH.parent / "exclusion_reasons.csv",
    encoding="utf-8-sig",
)
audit_select_manifest = pd.read_csv(
    SELECT_MANIFEST_PATH,
    encoding="utf-8-sig",
)

def audit_join_unique(values):
    return " / ".join(sorted({
        str(value).strip()
        for value in values
        if pd.notna(value) and str(value).strip()
    }))

audit_codeit_reason_lookup = (
    audit_codeit_reasons.groupby("image_id")["reason"]
    .agg(audit_join_unique)
    .to_dict()
)
audit_codeit_stage_lookup = (
    audit_codeit_reasons.groupby("image_id")["stage"]
    .agg(audit_join_unique)
    .to_dict()
)
audit_select_reason_lookup = (
    audit_select_reasons.groupby("json_file")["reason"]
    .agg(audit_join_unique)
    .to_dict()
)
audit_select_stage_lookup = (
    audit_select_reasons.groupby("json_file")["stage"]
    .agg(audit_join_unique)
    .to_dict()
)

audit_rows = []

# 2. 대회 원본 Train JSON 전체 확인
# 제외된 JSON도 필요하므로 검증 완료 ZIP이 아닌 원본 ZIP 사용
with zipfile.ZipFile(DATA_PATH, "r") as z:
    audit_codeit_json_members = sorted(
        member
        for member in z.namelist()
        if "train_annotations" in Path(member).parts
        and "__MACOSX" not in Path(member).parts
        and not Path(member).name.startswith(".")
        and member.lower().endswith(".json")
    )

    for member in audit_codeit_json_members:
        document = json.loads(
            z.read(member).decode("utf-8-sig")
        )
        image_lookup = {
            image["id"]: image for image in document["images"]
        }

        # 같은 클래스가 한 JSON에 여러 번 있어도 JSON 수는 한 번 집계
        class_image_pairs = {
            (
                int(annotation["category_id"]),
                annotation["image_id"],
            )
            for annotation in document["annotations"]
        }

        for category_id, image_id in class_image_pairs:
            image = image_lookup[image_id]
            excluded = image_id in audit_codeit_reason_lookup

            audit_rows.append({
                "데이터": "대회",
                "클래스명": f"K-{category_id:06d}",
                "JSON 경로": member,
                "이미지 ID": image_id,
                "이미지 파일명": image["file_name"],
                "제외 여부": excluded,
                "검증 단계": audit_codeit_stage_lookup.get(image_id, ""),
                "제외 사유": audit_codeit_reason_lookup.get(image_id, ""),
            })

# 3. SELECT 검증 전 JSON 전체 확인
audit_select_keep = (
    audit_select_manifest["keep"]
    .astype(str).str.strip().str.lower()
)

if not audit_select_keep.isin(["true", "false"]).all():
    raise ValueError("SELECT 목록의 keep 값을 확인해야 합니다.")

audit_select_excluded_jsons = set(
    audit_select_manifest.loc[
        audit_select_keep.eq("false"), "json_file"
    ]
)

if audit_select_excluded_jsons != set(audit_select_reason_lookup):
    raise ValueError(
        "SELECT 유지 목록과 제외 근거의 JSON 목록이 다릅니다."
    )

for index, record in enumerate(
    audit_select_manifest.to_dict("records"),
    start=1,
):
    json_file = record["json_file"]
    document = json.loads(
        (SELECT_DIR / json_file).read_text(encoding="utf-8-sig")
    )
    image_lookup = {
        image["id"]: image for image in document["images"]
    }

    class_image_pairs = {
        (
            int(annotation["category_id"]),
            annotation["image_id"],
        )
        for annotation in document["annotations"]
    }

    for category_id, image_id in class_image_pairs:
        image = image_lookup[image_id]
        excluded = json_file in audit_select_excluded_jsons

        audit_rows.append({
            "데이터": "SELECT",
            "클래스명": f"K-{category_id:06d}",
            "JSON 경로": json_file,
            "이미지 ID": image_id,
            "이미지 파일명": image["file_name"],
            "제외 여부": excluded,
            "검증 단계": audit_select_stage_lookup.get(json_file, ""),
            "제외 사유": audit_select_reason_lookup.get(json_file, ""),
        })

    if index % 2000 == 0 or index == len(audit_select_manifest):
        print(
            f"SELECT 내역 확인: "
            f"{index:,}/{len(audit_select_manifest):,}개"
        )

audit_records = pd.DataFrame(audit_rows).drop_duplicates(
    ["데이터", "클래스명", "JSON 경로", "이미지 ID"]
)

# 4. 클래스별 요약표
audit_summary_rows = []

for (dataset, class_name), group in audit_records.groupby(
    ["데이터", "클래스명"]
):
    excluded_group = group.loc[group["제외 여부"]]
    retained_group = group.loc[~group["제외 여부"]]

    audit_summary_rows.append({
        "데이터": dataset,
        "클래스명": class_name,
        "연결된 JSON 개수": group["JSON 경로"].nunique(),
        "제외 JSON 수": excluded_group["JSON 경로"].nunique(),
        "전체 이미지 수": group["이미지 ID"].nunique(),
        "제외 이미지 수": excluded_group["이미지 ID"].nunique(),
        "유지 이미지 수": retained_group["이미지 ID"].nunique(),
        "클래스 전체 제외": retained_group.empty,
        "제외된 사유": audit_join_unique(
            excluded_group["제외 사유"]
        ),
    })

class_exclusion_summary = pd.DataFrame(
    audit_summary_rows
).sort_values(
    ["데이터", "제외 이미지 수", "클래스명"],
    ascending=[True, False, True],
).reset_index(drop=True)

# 제외된 이미지와 JSON 파일명 상세표
excluded_image_details = (
    audit_records.loc[
        audit_records["제외 여부"],
        [
            "데이터", "클래스명", "이미지 ID",
            "이미지 파일명", "JSON 경로",
            "검증 단계", "제외 사유",
        ],
    ]
    .sort_values(["데이터", "클래스명", "이미지 파일명", "JSON 경로"])
    .reset_index(drop=True)
)

# 5. 출처별 실제 제외 수 검증
audit_dataset_summary_rows = []

for dataset, group in audit_records.groupby("데이터"):
    excluded_group = group.loc[group["제외 여부"]]

    audit_dataset_summary_rows.append({
        "데이터": dataset,
        "전체 JSON 수": group["JSON 경로"].nunique(),
        "제외 JSON 수": excluded_group["JSON 경로"].nunique(),
        "전체 이미지 수": group["이미지 ID"].nunique(),
        "제외 이미지 수": excluded_group["이미지 ID"].nunique(),
        "유지 이미지 수": (
            group.loc[~group["제외 여부"], "이미지 ID"].nunique()
        ),
    })

audit_dataset_summary = pd.DataFrame(
    audit_dataset_summary_rows
).set_index("데이터")

assert audit_dataset_summary.loc["대회", "전체 JSON 수"] == 763
assert audit_dataset_summary.loc["대회", "전체 이미지 수"] == 232
assert audit_dataset_summary.loc["대회", "제외 이미지 수"] == 17
assert audit_dataset_summary.loc["SELECT", "전체 JSON 수"] == 10240
assert audit_dataset_summary.loc["SELECT", "제외 이미지 수"] == 131

# 6. CSV 두 개와 설명 저장
EXCLUSION_AUDIT_DIR = INTEGRATION_RESULT_DIR / "exclusion_audit"
EXCLUSION_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

summary_csv_path = (
    EXCLUSION_AUDIT_DIR / "class_exclusion_summary.csv"
)
details_csv_path = (
    EXCLUSION_AUDIT_DIR / "excluded_image_details.csv"
)

# 요약 CSV에는 제외 수가 0인 클래스도 포함
class_exclusion_summary.to_csv(
    summary_csv_path,
    index=False,
    encoding="utf-8-sig",
)
excluded_image_details.to_csv(
    details_csv_path,
    index=False,
    encoding="utf-8-sig",
)

(EXCLUSION_AUDIT_DIR / "README.md").write_text(
    """# 제외 내역 집계 기준

- 클래스명은 공통 알약 코드로 표시함.
- 연결된 JSON 수와 전체 이미지 수는 검증 전 자료 기준임.
- 대회는 원본의 알약별 JSON, SELECT는 이미지별 JSON 기준임.
- JSON 수와 이미지 수는 각각 중복 없이 집계함.
- 이미지에서 발생한 오류로 해당 이미지의 모든 annotation을 제외함.
- 따라서 사유는 해당 클래스 자체의 오류를 의미하지 않을 수 있음.
- 한 이미지에 여러 클래스가 있으면 각 클래스의 내역에 포함됨.
- 클래스별 제외 이미지 수를 합하면 실제 제외 이미지 수보다 커질 수 있음.
- 상세표는 클래스와 JSON 연결 단위이므로 이미지 파일명이 반복될 수 있음.
- K-003351과 K-003483은 SELECT에 annotation이 처음부터 없어
  SELECT의 검증 제외 클래스로 집계하지 않음.
- 제외 사유는 기존 검증 기록의 문구를 그대로 사용함.
""",
    encoding="utf-8",
)

# 7. 표 출력
print("\n========== 6-1. 클래스·JSON·이미지 제외 내역 ==========")

print("\n[출처별 전체 집계 — 이미지 중복 제거]")
display(audit_dataset_summary)

print("\n[제외된 데이터가 있는 클래스]")
display(
    class_exclusion_summary.loc[
        class_exclusion_summary["제외 이미지 수"].gt(0)
    ]
)

print("\n[제외 이미지·JSON 상세: 처음 20행]")
display(excluded_image_details.head(20))

print(
    "\n클래스 전체가 제외된 경우: "
    f"{class_exclusion_summary['클래스 전체 제외'].sum():,}건"
)
print(f"\n요약 CSV: {summary_csv_path}")
print(f"상세 CSV: {details_csv_path}")
print("※ 상세 CSV에는 제외 내역 전체를 저장했습니다.")
print("※ 기존 데이터와 분할은 변경하지 않았습니다.")

### 6-2. 탐지·분류 공통 분할 및 박스별 목록 저장

In [ ]:
import math

# 1. 저장된 공통 분할과 라벨 읽기
classification_split_manifest = pd.read_csv(
    TRAINING_PACKAGE_DIR / "split_manifest.csv",
    encoding="utf-8-sig",
)

classification_mapping = pd.read_csv(
    TRAINING_PACKAGE_DIR / "class_mapping.csv",
    encoding="utf-8-sig",
)

assert classification_split_manifest["image_id"].is_unique
assert set(classification_split_manifest["split"]) == {
    "train", "validation"
}

classification_image_lookup = (
    classification_split_manifest
    .set_index("image_id")
    .to_dict("index")
)

classification_class_lookup = {
    int(record["category_id"]): record
    for record in classification_mapping.to_dict("records")
}

# 2. 박스별 분류 데이터 목록 생성
classification_crop_rows = []

for split_name in ["train", "validation"]:
    document = json.loads(
        (
            TRAINING_PACKAGE_DIR
            / "annotations"
            / f"{split_name}.json"
        ).read_text(encoding="utf-8")
    )

    image_lookup = {
        image["id"]: image
        for image in document["images"]
    }

    expected_image_ids = set(
        classification_split_manifest.loc[
            classification_split_manifest["split"].eq(split_name),
            "image_id",
        ]
    )
    assert set(image_lookup) == expected_image_ids

    for annotation in document["annotations"]:
        image_id = int(annotation["image_id"])
        annotation_id = int(annotation["id"])
        category_id = int(annotation["category_id"])

        image_info = image_lookup[image_id]
        split_info = classification_image_lookup[image_id]
        class_info = classification_class_lookup[category_id]

        assert split_info["split"] == split_name
        assert image_info["file_name"] == split_info["image_path"]

        x, y, w, h = map(float, annotation["bbox"])

        if not (
            all(math.isfinite(value) for value in [x, y, w, h])
            and x >= 0
            and y >= 0
            and w > 0
            and h > 0
            and x + w <= image_info["width"]
            and y + h <= image_info["height"]
        ):
            raise ValueError(
                f"분류용 bbox 확인 필요: annotation {annotation_id}"
            )

        # 소수 좌표는 객체를 포함하도록 바깥 방향으로 정수화
        crop_left = max(0, math.floor(x))
        crop_top = max(0, math.floor(y))
        crop_right = min(
            int(image_info["width"]), math.ceil(x + w)
        )
        crop_bottom = min(
            int(image_info["height"]), math.ceil(y + h)
        )

        crop_file = (
            f"{split_name}/{class_info['code']}/"
            f"image_{image_id:06d}_ann_{annotation_id:06d}.png"
        )

        classification_crop_rows.append({
            "split": split_name,
            "group_id": split_info["group_id"],
            "dataset": split_info["dataset"],
            "image_id": image_id,
            "annotation_id": annotation_id,
            "drug_code": class_info["code"],
            "category_id": category_id,
            "model_class_id": int(class_info["model_class_id"]),
            "image_path": split_info["image_path"],
            "image_width": int(image_info["width"]),
            "image_height": int(image_info["height"]),
            "bbox_x": x,
            "bbox_y": y,
            "bbox_width": w,
            "bbox_height": h,
            "crop_left": crop_left,
            "crop_top": crop_top,
            "crop_right": crop_right,
            "crop_bottom": crop_bottom,
            "crop_file": crop_file,
        })

classification_crop_manifest = pd.DataFrame(
    classification_crop_rows
).sort_values(
    ["split", "image_id", "annotation_id"]
).reset_index(drop=True)

classification_train_manifest = classification_crop_manifest.loc[
    classification_crop_manifest["split"].eq("train")
].copy()

classification_validation_manifest = (
    classification_crop_manifest.loc[
        classification_crop_manifest["split"].eq("validation")
    ].copy()
)

# 3. 누수 방지 조건 확인
assert classification_crop_manifest["annotation_id"].is_unique
assert classification_crop_manifest["crop_file"].is_unique
assert len(classification_train_manifest) == 31689
assert len(classification_validation_manifest) == 7919

for field in ["image_id", "group_id"]:
    train_values = set(classification_train_manifest[field])
    validation_values = set(classification_validation_manifest[field])
    assert train_values.isdisjoint(validation_values), (
        f"Train/Validation의 {field}가 겹칩니다."
    )

assert classification_train_manifest["category_id"].nunique() == 118
assert classification_validation_manifest["category_id"].nunique() == 118

# 4. 분류 담당자용 클래스별 수량
classification_class_summary = (
    classification_crop_manifest.groupby([
        "drug_code", "category_id", "model_class_id", "split"
    ])
    .agg(
        crop_count=("annotation_id", "size"),
        image_count=("image_id", "nunique"),
        group_count=("group_id", "nunique"),
    )
    .reset_index()
)

# 5. CSV와 사용 설명 저장
CLASSIFICATION_SHARE_DIR = (
    SPLIT_RESULT_DIR / "classification_share"
)
CLASSIFICATION_SHARE_DIR.mkdir(parents=True, exist_ok=True)

classification_tables = {
    "classification_crop_manifest.csv": classification_crop_manifest,
    "classification_train.csv": classification_train_manifest,
    "classification_validation.csv": classification_validation_manifest,
    "classification_class_summary.csv": classification_class_summary,
    "split_manifest.csv": classification_split_manifest,
    "class_mapping.csv": classification_mapping,
}

for filename, table in classification_tables.items():
    table.to_csv(
        CLASSIFICATION_SHARE_DIR / filename,
        index=False,
        encoding="utf-8-sig",
    )

classification_readme = """# 탐지·분류 공통 분할 사용 안내

## 분할 원칙
탐지 모델에서 확정한 이미지별 Train/Validation 배정을 그대로 사용합니다.
분류용 알약 이미지를 자른 뒤 다시 무작위 분할하지 않습니다.
같은 조합 그룹은 한쪽 분할에만 속합니다.
증강 결과도 원본 이미지의 split과 group_id를 상속합니다.

## 파일
- classification_crop_manifest.csv: 전체 박스별 목록
- classification_train.csv: 분류 Train 박스 목록
- classification_validation.csv: 분류 Validation 박스 목록
- classification_class_summary.csv: 클래스별 박스·이미지·그룹 수
- split_manifest.csv: 탐지와 공유하는 원본 이미지 분할
- class_mapping.csv: 공통 클래스 매핑

## 이미지 연결
image_path의 select/는 기존 SELECT 폴더를 가리킵니다.
image_path의 codeit/는 검증 완료 대회 ZIP을 푼 폴더를 가리킵니다.
각 출처 폴더 내부의 images 폴더에서 원본 이미지를 읽습니다.

## 자르기
PIL 이미지의 crop 메서드에 다음 좌표를 전달합니다.
(crop_left, crop_top, crop_right, crop_bottom)
오른쪽과 아래쪽 경계는 PIL 방식대로 끝점을 포함하지 않습니다.
좌표는 원본 이미지 크기 기준입니다.
이미지를 먼저 축소했다면 좌표도 함께 변환해야 합니다.
crop_file은 향후 저장할 상대 경로이며 아직 파일을 생성하지 않았습니다.
별도 여백은 추가하지 않았습니다.

## 클래스 번호
분류 학습용 라벨은 model_class_id 0~117을 사용합니다.
category_id는 원본 알약 코드의 숫자이므로 그대로 분류 출력 번호로 쓰지 않습니다.
출력 클래스 수는 118입니다.

## 추가 데이터
별도 단일 이미지나 외부 데이터를 추가할 경우 이 목록만으로
중복이나 누수가 자동 방지되지는 않습니다.
기존 Validation 이미지 및 관련 촬영 출처와의 중복을 별도로 확인합니다.

## 평가 주의
Validation 알약의 정답 박스를 잘라 평가한 분류 성능은
정답 위치가 주어진 조건의 성능입니다.
탐지 모델이 예측한 박스를 사용하는 전체 파이프라인 성능과 구분합니다.
"""

(CLASSIFICATION_SHARE_DIR / "README.md").write_text(
    classification_readme,
    encoding="utf-8",
)

# 6. 이미지 없는 팀원 공유용 ZIP
CLASSIFICATION_SHARE_ZIP = (
    SPLIT_RESULT_DIR / "classification_share.zip"
)

with zipfile.ZipFile(
    CLASSIFICATION_SHARE_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as z:
    for filename in [*classification_tables, "README.md"]:
        z.write(
            CLASSIFICATION_SHARE_DIR / filename,
            arcname=filename,
        )

with zipfile.ZipFile(CLASSIFICATION_SHARE_ZIP, "r") as z:
    assert z.testzip() is None

print("========== 6-2. 탐지·분류 공통 분할 목록 저장 ==========")
print(f"Train 알약 박스: {len(classification_train_manifest):,}개")
print(
    f"Validation 알약 박스: "
    f"{len(classification_validation_manifest):,}개"
)
print("양쪽 포함 클래스: 각각 118종")
print("분할 간 원본 이미지 ID 중복: 0개")
print("분할 간 조합 그룹 중복: 0개")

print("\n[Validation 클래스별 수량: 처음 10종]")
display(
    classification_class_summary.loc[
        classification_class_summary["split"].eq("validation")
    ].head(10)
)

print(f"\nCSV 저장 폴더: {CLASSIFICATION_SHARE_DIR}")
print(f"팀원 공유용 ZIP: {CLASSIFICATION_SHARE_ZIP}")
print("※ 이미지 자르기·복사 및 기존 분할 변경 없음")

### 6-3. 추가 오류 제외 및 분할 갱신

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import re
import shutil
import zipfile
import numpy as np
import pandas as pd

# 1. 기존 학습 자료 읽기
old_package_dir = Path(TRAINING_PACKAGE_DIR)

old_split_manifest = pd.read_csv(
    old_package_dir / "split_manifest.csv",
    encoding="utf-8-sig",
)

old_coco = {
    split_name: json.loads(
        (
            old_package_dir
            / "annotations"
            / f"{split_name}.json"
        ).read_text(encoding="utf-8")
    )
    for split_name in ["train", "validation"]
}

correction_targets = [
    {
        "original_stem":
            "K-012420-016551-024850-027926_0_2_0_2_70_000_200",
        "drug_code": "K-016551",
        "width": 23,
        "height": 497,
    },
    {
        "original_stem":
            "K-012420-016548-031705-033208_0_2_0_2_90_000_200",
        "drug_code": "K-031705",
        "width": 27,
        "height": 435,
    },
    {
        "original_stem":
            "K-012247-016548-019607-029345_0_2_0_2_90_000_200",
        "drug_code": "K-019607",
        "width": 159,
        "height": 15,
    },
    {
        "original_stem":
            "K-012420-016548-021771-027926_0_2_0_2_70_000_200",
        "drug_code": "K-012420",
        "width": 19,
        "height": 192,
    },
    {
        "original_stem":
            "K-012420-016548-021026-024850_0_2_0_2_90_000_200",
        "drug_code": "K-021026",
        "width": 19,
        "height": 178,
    },
]

target_lookup = {
    record["original_stem"]: record
    for record in correction_targets
}

if not old_split_manifest["image_id"].is_unique:
    raise ValueError("기존 분할 목록에 중복 이미지 ID가 있습니다.")

manifest_lookup = (
    old_split_manifest.set_index("image_id").to_dict("index")
)

matched_target_rows = []
all_box_rows = []

# 2. 지정 이미지와 bbox 확인 및 구조 검사 재현
for split_name, document in old_coco.items():
    image_lookup = {
        image["id"]: image for image in document["images"]
    }

    for annotation in document["annotations"]:
        image = image_lookup[annotation["image_id"]]
        name = Path(image["file_name"]).stem
        match = re.search(r"(K-\d{6}.*)$", name)
        original_stem = match.group(1) if match else name

        x, y, w, h = map(float, annotation["bbox"])
        short_side = min(w, h)
        elongation = (
            max(w, h) / short_side
            if short_side > 0 else np.inf
        )

        row = {
            "split": split_name,
            "image_id": int(image["id"]),
            "annotation_id": int(annotation["id"]),
            "drug_code": annotation["drug_code"],
            "image_path": image["file_name"],
            "original_stem": original_stem,
            "bbox": annotation["bbox"],
            "width": w,
            "height": h,
            "short_side": short_side,
            "long_short_ratio": elongation,
        }
        all_box_rows.append(row)

        target = target_lookup.get(original_stem)

        if (
            target is None
            or annotation["drug_code"] != target["drug_code"]
        ):
            continue

        if not (
            np.isclose(w, target["width"])
            and np.isclose(h, target["height"])
        ):
            raise ValueError(
                f"지정한 bbox 크기와 현재 라벨이 다릅니다: {original_stem}"
            )

        split_info = manifest_lookup[int(image["id"])]

        matched_target_rows.append({
            **row,
            "dataset": split_info["dataset"],
            "original_image_id": split_info["original_image_id"],
            "original_json_file": split_info["original_json_file"],
            "좌표와_크기_조건_통과": bool(
                x >= 0 and y >= 0 and w > 0 and h > 0
            ),
            "이미지_범위_조건_통과": bool(
                x >= 0 and y >= 0
                and x + w <= image["width"]
                and y + h <= image["height"]
            ),
            "면적_일치_조건_통과": bool(np.isclose(
                float(annotation["area"]),
                w * h,
                rtol=1e-9,
                atol=1e-6,
            )),
            "제외_사유": (
                "시각 검토에서 bbox가 알약을 감싸지 않고 "
                "주로 배경을 포함하는 오류 확인"
            ),
        })

correction_details = pd.DataFrame(matched_target_rows)

if (
    len(correction_details) != 5
    or correction_details["original_stem"].nunique() != 5
    or correction_details["image_id"].nunique() != 5
):
    display(correction_details)
    raise ValueError(
        "지정한 5장을 각각 하나의 이미지로 확정하지 못했습니다. "
        "새 파일은 저장하지 않았습니다."
    )

removed_image_ids = set(correction_details["image_id"])

# 3. 같은 유형의 추가 후보 검사
# 후보 조건일 뿐, 다른 이미지를 자동 제외하지 않음
all_box_checks = pd.DataFrame(all_box_rows)

thin_box_candidates = all_box_checks.loc[
    all_box_checks["short_side"].le(30)
    | all_box_checks["long_short_ratio"].ge(8)
].copy()

remaining_thin_candidates = thin_box_candidates.loc[
    ~thin_box_candidates["image_id"].isin(removed_image_ids)
].copy()

# 4. 기존 배정을 유지하면서 5장만 제외
new_split_manifest = old_split_manifest.loc[
    ~old_split_manifest["image_id"].isin(removed_image_ids)
].copy()

new_coco = {}
revision_summary_rows = []
removed_annotation_rows = []

for split_name, old_document in old_coco.items():
    document = dict(old_document)

    document["images"] = [
        image for image in old_document["images"]
        if image["id"] not in removed_image_ids
    ]
    document["annotations"] = [
        annotation for annotation in old_document["annotations"]
        if annotation["image_id"] not in removed_image_ids
    ]

    new_coco[split_name] = document

    for annotation in old_document["annotations"]:
        if annotation["image_id"] in removed_image_ids:
            removed_annotation_rows.append({
                "split": split_name,
                "image_id": annotation["image_id"],
                "annotation_id": annotation["id"],
                "drug_code": annotation["drug_code"],
                "category_id": annotation["category_id"],
                "bbox": annotation["bbox"],
            })

    revision_summary_rows.append({
        "split": split_name,
        "수정전_이미지": len(old_document["images"]),
        "추가제외_이미지": (
            len(old_document["images"]) - len(document["images"])
        ),
        "수정후_이미지": len(document["images"]),
        "수정전_박스": len(old_document["annotations"]),
        "추가제외_박스": (
            len(old_document["annotations"])
            - len(document["annotations"])
        ),
        "수정후_박스": len(document["annotations"]),
        "수정후_클래스": len({
            annotation["category_id"]
            for annotation in document["annotations"]
        }),
    })

revision_summary = pd.DataFrame(revision_summary_rows)
removed_annotations = pd.DataFrame(removed_annotation_rows)

# 5. 분할 및 연결 검증
assert len(new_split_manifest) == len(old_split_manifest) - 5

old_assignments = old_split_manifest.set_index("image_id")["split"]
new_assignments = new_split_manifest.set_index("image_id")["split"]

assert old_assignments.loc[
    new_assignments.index
].equals(new_assignments)

train_groups = set(
    new_split_manifest.loc[
        new_split_manifest["split"].eq("train"), "group_id"
    ]
)
validation_groups = set(
    new_split_manifest.loc[
        new_split_manifest["split"].eq("validation"), "group_id"
    ]
)

assert train_groups.isdisjoint(validation_groups)
assert set(new_split_manifest["image_id"]).isdisjoint(
    removed_image_ids
)

for split_name, document in new_coco.items():
    expected_ids = set(
        new_split_manifest.loc[
            new_split_manifest["split"].eq(split_name), "image_id"
        ]
    )

    assert {
        image["id"] for image in document["images"]
    } == expected_ids

    assert all(
        annotation["image_id"] in expected_ids
        for annotation in document["annotations"]
    )

# 6. 클래스별 분포 다시 계산
class_rows = [
    {
        "split": split_name,
        "category_id": annotation["category_id"],
        "image_id": annotation["image_id"],
    }
    for split_name, document in new_coco.items()
    for annotation in document["annotations"]
]

class_records = pd.DataFrame(class_rows)

mapping = pd.read_csv(
    old_package_dir / "class_mapping.csv",
    encoding="utf-8-sig",
)

new_class_distribution = pd.DataFrame(
    index=pd.Index(
        sorted(mapping["category_id"].astype(int)),
        name="category_id",
    )
)

for split_name in ["train", "validation"]:
    subset = class_records.loc[
        class_records["split"].eq(split_name)
    ]

    new_class_distribution[f"{split_name}_boxes"] = (
        subset.groupby("category_id").size()
    )
    new_class_distribution[f"{split_name}_images"] = (
        subset.groupby("category_id")["image_id"].nunique()
    )

new_class_distribution = (
    new_class_distribution.fillna(0).astype(int)
)

class_totals = (
    new_class_distribution["train_boxes"]
    + new_class_distribution["validation_boxes"]
)

new_class_distribution["validation_box_pct"] = (
    new_class_distribution["validation_boxes"]
    .div(class_totals.replace(0, np.nan))
    .mul(100)
    .round(2)
)

missing_split_classes = new_class_distribution.loc[
    new_class_distribution["train_boxes"].eq(0)
    | new_class_distribution["validation_boxes"].eq(0)
]

# 7. 새 버전 저장
REVISION_DIR = old_package_dir.parent / (
    "revision_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
NEW_TRAINING_PACKAGE_DIR = REVISION_DIR / "training_package"

(NEW_TRAINING_PACKAGE_DIR / "annotations").mkdir(
    parents=True, exist_ok=False
)

for split_name, document in new_coco.items():
    (
        NEW_TRAINING_PACKAGE_DIR
        / "annotations"
        / f"{split_name}.json"
    ).write_text(
        json.dumps(
            document,
            ensure_ascii=False,
            allow_nan=False,
        ),
        encoding="utf-8",
    )

for filename in ["pill_data.py", "class_mapping.csv"]:
    shutil.copy2(
        old_package_dir / filename,
        NEW_TRAINING_PACKAGE_DIR / filename,
    )

new_split_manifest.to_csv(
    NEW_TRAINING_PACKAGE_DIR / "split_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

for split_name in ["train", "validation"]:
    new_split_manifest.loc[
        new_split_manifest["split"].eq(split_name)
    ].to_csv(
        NEW_TRAINING_PACKAGE_DIR / f"{split_name}_manifest.csv",
        index=False,
        encoding="utf-8-sig",
    )

new_class_distribution.to_csv(
    NEW_TRAINING_PACKAGE_DIR / "split_class_distribution.csv",
    encoding="utf-8-sig",
)

correction_details.to_csv(
    NEW_TRAINING_PACKAGE_DIR / "additional_exclusions.csv",
    index=False,
    encoding="utf-8-sig",
)

removed_annotations.to_csv(
    NEW_TRAINING_PACKAGE_DIR / "removed_annotations.csv",
    index=False,
    encoding="utf-8-sig",
)

revision_summary.to_csv(
    NEW_TRAINING_PACKAGE_DIR / "revision_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

remaining_thin_candidates.to_csv(
    NEW_TRAINING_PACKAGE_DIR / "remaining_thin_bbox_candidates.csv",
    index=False,
    encoding="utf-8-sig",
)

# 분류 담당자에게 전달할 원본 이미지 기준 목록
classification_split_reference = new_split_manifest.copy()

classification_split_reference["original_image_stem"] = (
    classification_split_reference["file_name"].map(
        lambda name: (
            re.search(r"(K-\d{6}.*)$", Path(name).stem).group(1)
            if re.search(r"(K-\d{6}.*)$", Path(name).stem)
            else Path(name).stem
        )
    )
)

classification_split_reference.to_csv(
    NEW_TRAINING_PACKAGE_DIR / "classification_split_reference.csv",
    index=False,
    encoding="utf-8-sig",
)

old_settings_path = old_package_dir / "split_settings.json"
new_settings = (
    json.loads(old_settings_path.read_text(encoding="utf-8"))
    if old_settings_path.is_file()
    else {}
)

new_settings.update({
    "revision": "exclude_5_confirmed_bbox_errors",
    "resplit": False,
    "removed_image_ids": sorted(
        int(value) for value in removed_image_ids
    ),
    "train_images": int(
        new_split_manifest["split"].eq("train").sum()
    ),
    "validation_images": int(
        new_split_manifest["split"].eq("validation").sum()
    ),
    "train_groups": len(train_groups),
    "validation_groups": len(validation_groups),
    "classes_in_both": int(
        (
            new_class_distribution["train_boxes"].gt(0)
            & new_class_distribution["validation_boxes"].gt(0)
        ).sum()
    ),
})

(NEW_TRAINING_PACKAGE_DIR / "split_settings.json").write_text(
    json.dumps(new_settings, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# 8. 검사 누락 원인과 수정 근거 기록
revision_report = f"""# 추가 bbox 오류 확인 및 데이터 수정

## 확인한 오류

시각 검토에서 5개 bbox가 알약을 감싸지 않고 주로 배경을 포함하는
명확한 위치·크기 오류임을 확인하였다.
해당 이미지 5장과 연결된 모든 annotation을 제외하였다.
상세 파일명과 좌표는 additional_exclusions.csv에 기록하였다.

## 구조 검사로 놓칠 수 있는 이유

- 양수인 너비·높이는 실제 물체를 감싸는지를 보장하지 않는다.
- 이미지 범위 안에 있어도 박스가 배경에 위치할 수 있다.
- area와 너비×높이가 같아도 박스 위치·크기는 잘못될 수 있다.
- 박스 중첩 검사는 다른 박스와의 관계를 확인하며,
  단독으로 잘못 배치된 얇은 박스를 반드시 검출하지 않는다.
- 실물 길이 대비 크기 검사는 긴 변을 사용하므로,
  긴 변은 정상이고 짧은 변만 비정상인 오류를 놓칠 수 있다.
- 클래스별 IQR 후보의 표본 검토는 전체 라벨 검증이 아니다.
- 전체 Train/Validation 크기 분포가 비슷해도 개별 오류는 남을 수 있다.

위 내용은 검사 설계의 한계이다.
5개 사례가 과거 IQR 후보에 실제 포함됐는지, 시각 표본으로 표시됐는지는
원래 검사 기록을 확인해야 하며 이 수정 작업만으로 확정하지 않는다.
이번에 재계산한 구조 검사 결과는 additional_exclusions.csv에 기록하였다.

## 보완 검사

짧은 변 30px 이하 또는 긴 변/짧은 변 비율 8 이상을 추가 후보로 찾았다.
이 기준은 수동 검토용 참고 기준이며 자동 제외 기준이 아니다.
확인된 5장 외 추가 후보: {len(remaining_thin_candidates)}개 박스.
상세 목록은 remaining_thin_bbox_candidates.csv에 기록하였다.

## 분할 처리

남은 이미지의 기존 Train/Validation 배정 및 ID는 유지하였다.
재분할하지 않았으며 같은 조합 그룹의 분할 간 중복은 없다.
새 수량은 revision_summary.csv에 기록하였다.

## 팀원 사용 방법

기존 이미지 폴더는 그대로 사용한다.
이 버전의 training_package 폴더를 별도 위치에 풀고,
로더 실행 셀의 PACKAGE_DIR을 새 폴더로 지정한다.
이미 실행 중인 Dataset과 DataLoader는 새 패키지로 다시 생성한다.

분류 담당자는 classification_split_reference.csv를 사용한다.
추가 제외한 원본 이미지에서 잘라 만든 분류 샘플도 함께 제외한다.

이전 분포 그래프, 제외 요약 및 기존 분류용 목록은 이전 버전 자료이다.
새 버전 기준으로 다시 집계해야 한다.
기존 integrated_labels.zip도 이전 버전이며 이번 수정으로 갱신되지 않는다.

## 현재 한계

추가 후보가 남아 있다면 해당 후보의 시각 검토가 필요하다.
이번 저장 완료는 나머지 라벨 전체가 정상이라는 판정을 의미하지 않는다.
"""

(NEW_TRAINING_PACKAGE_DIR / "revision_report.md").write_text(
    revision_report,
    encoding="utf-8",
)

readme = """# 수정된 공통 학습 패키지

확인된 bbox 오류 이미지 5장을 추가 제외한 버전입니다.
기존 이미지 폴더와 클래스 매핑은 그대로 사용합니다.
Train/Validation 배정은 재생성하지 않았습니다.

- annotations/train.json: 수정된 Train 라벨
- annotations/validation.json: 수정된 Validation 라벨
- split_manifest.csv: 수정된 전체 이미지 분할 목록
- train_manifest.csv: 수정된 Train 이미지 목록
- validation_manifest.csv: 수정된 Validation 이미지 목록
- class_mapping.csv: 기존과 동일한 클래스 매핑
- pill_data.py: 기존과 동일한 로더
- split_class_distribution.csv: 수정된 클래스별 수량
- classification_split_reference.csv: 분류 담당자용 원본 분할 목록
- additional_exclusions.csv: 확인된 5개 오류와 원본 파일 정보
- removed_annotations.csv: 이미지와 함께 제외된 전체 박스
- remaining_thin_bbox_candidates.csv: 추가 시각 검토 후보
- revision_report.md: 수정 근거 및 검사 한계

이전 폴더에 덮어 풀지 말고 별도 위치에 풀어 사용하세요.
로더의 PACKAGE_DIR을 새 폴더로 지정하고 로더를 다시 생성하세요.
YOLOX 실행 시 실제 이미지 경로가 연결된 라벨은 새로 생성됩니다.
"""

(NEW_TRAINING_PACKAGE_DIR / "README.md").write_text(
    readme, encoding="utf-8"
)

# 9. 공유용 ZIP 저장 및 내용 확인
NEW_TRAINING_PACKAGE_ZIP = REVISION_DIR / "training_package.zip"

package_files = sorted(
    path
    for path in NEW_TRAINING_PACKAGE_DIR.rglob("*")
    if path.is_file()
)

with zipfile.ZipFile(
    NEW_TRAINING_PACKAGE_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as z:
    for path in package_files:
        z.write(
            path,
            arcname=path.relative_to(
                NEW_TRAINING_PACKAGE_DIR
            ).as_posix(),
        )

with zipfile.ZipFile(NEW_TRAINING_PACKAGE_ZIP, "r") as z:
    assert z.testzip() is None

    for split_name, expected_document in new_coco.items():
        saved_document = json.loads(
            z.read(
                f"annotations/{split_name}.json"
            ).decode("utf-8")
        )
        assert saved_document == expected_document

# 10. 결과 출력
print("\n========== 6-3. 추가 오류 제외 및 분할 갱신 ==========")

print("\n[확인된 5개 오류와 구조 검사 결과]")
display(correction_details[[
    "original_stem", "drug_code", "split", "bbox",
    "좌표와_크기_조건_통과",
    "이미지_범위_조건_통과",
    "면적_일치_조건_통과",
]])

print("\n[분할별 변경 수량]")
display(revision_summary)

print("나머지 이미지의 기존 분할 및 ID 유지")
print("Train/Validation 조합 그룹 중복: 0개")
print(
    f"추가 시각 검토 후보: {len(remaining_thin_candidates):,}개 박스, "
    f"{remaining_thin_candidates['image_id'].nunique():,}장"
)

if not missing_split_classes.empty:
    print("\n[수정 후 한쪽 분할에 박스가 없는 클래스]")
    display(missing_split_classes)
else:
    print("118종 모두 Train과 Validation에 유지됨")

if not remaining_thin_candidates.empty:
    print("\n[추가 후보: 최대 20개 — 자동 제외하지 않음]")
    display(remaining_thin_candidates.head(20))

print(f"\n새 패키지 폴더: {NEW_TRAINING_PACKAGE_DIR}")
print(f"새 공유용 ZIP: {NEW_TRAINING_PACKAGE_ZIP}")
print("※ 기존 이미지·라벨·분할 파일 변경 없음")

### 6-4. 수정된 분포·분할 목록·제외 내역 및 근거 저장

In [ ]:
# 기존 제외 요약·상세 파일 위치 복구
audit_restore_dir = INTEGRATION_RESULT_DIR / "exclusion_audit"

required_audit_files = {
    "class_exclusion_summary.csv": "class_exclusion_summary",
    "excluded_image_details.csv": "excluded_image_details",
}

audit_restore_sources = {}
audit_restore_missing = []

for filename, variable_name in required_audit_files.items():
    expected_path = audit_restore_dir / filename

    if expected_path.is_file():
        audit_restore_sources[filename] = expected_path
        continue

    search_root = DATA_PATH.parent.parent
    candidates = sorted(
        path
        for path in search_root.rglob(filename)
        if path != expected_path
        and not any(
            part.startswith("revision_")
            for part in path.relative_to(search_root).parts
        )
    )

    if len(candidates) == 1:
        audit_restore_sources[filename] = candidates[0]

    elif len(candidates) > 1:
        print(f"\n{filename} 후보가 여러 개입니다:")
        for path in candidates:
            print(path)
        audit_restore_missing.append(filename)

    elif isinstance(globals().get(variable_name), pd.DataFrame):
        audit_restore_sources[filename] = globals()[variable_name].copy()

    else:
        audit_restore_missing.append(filename)

# 모두 확보한 경우에만 복구
if audit_restore_missing:
    print("\n자동 복구하지 못한 파일:")
    for filename in audit_restore_missing:
        print(filename)
    print("6-4 셀은 아직 다시 실행하지 마세요.")
else:
    audit_restore_dir.mkdir(parents=True, exist_ok=True)

    for filename, source in audit_restore_sources.items():
        destination = audit_restore_dir / filename

        if isinstance(source, pd.DataFrame):
            if not destination.exists():
                source.to_csv(
                    destination,
                    index=False,
                    encoding="utf-8-sig",
                )
            print(f"메모리 집계에서 복구: {destination}")

        elif source != destination:
            if not destination.exists():
                destination.write_bytes(source.read_bytes())
            print(f"기존 파일에서 복구: {source}")
        else:
            print(f"기존 위치 확인: {destination}")

    print("\n복구 완료")
    print("앞서 오류가 난 6-4 셀 전체를 다시 실행하면 됩니다.")
    print("6-3의 5장 제외 작업은 다시 실행하지 않아도 됩니다.")

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

# 1. 수정 버전 파일 확인
package_dir = Path(NEW_TRAINING_PACKAGE_DIR)

revised_manifest = pd.read_csv(
    package_dir / "split_manifest.csv",
    encoding="utf-8-sig",
)

revised_coco = {
    split_name: json.loads(
        (
            package_dir / "annotations" / f"{split_name}.json"
        ).read_text(encoding="utf-8")
    )
    for split_name in ["train", "validation"]
}

additional_exclusions = pd.read_csv(
    package_dir / "additional_exclusions.csv",
    encoding="utf-8-sig",
)
removed_box_records = pd.read_csv(
    package_dir / "removed_annotations.csv",
    encoding="utf-8-sig",
)

shared_mapping = pd.read_csv(
    package_dir / "class_mapping.csv",
    encoding="utf-8-sig",
)

assert additional_exclusions["image_id"].is_unique
assert len(additional_exclusions) == 5
assert not revised_manifest["image_id"].isin(
    additional_exclusions["image_id"]
).any()

# 2. 클래스별 이미지·박스 분포 재집계
revised_box_rows = []

for split_name, document in revised_coco.items():
    for annotation in document["annotations"]:
        revised_box_rows.append({
            "split": split_name,
            "image_id": annotation["image_id"],
            "annotation_id": annotation["id"],
            "category_id": annotation["category_id"],
            "drug_code": annotation["drug_code"],
        })

revised_boxes = pd.DataFrame(revised_box_rows)

revised_class_distribution = (
    shared_mapping[["code", "category_id", "model_class_id"]]
    .rename(columns={"code": "drug_code"})
    .set_index("category_id")
)

for split_name in ["train", "validation"]:
    subset = revised_boxes.loc[
        revised_boxes["split"].eq(split_name)
    ]

    revised_class_distribution[f"{split_name}_boxes"] = (
        subset.groupby("category_id").size()
    )
    revised_class_distribution[f"{split_name}_images"] = (
        subset.groupby("category_id")["image_id"].nunique()
    )

count_columns = [
    "train_boxes", "validation_boxes",
    "train_images", "validation_images",
]
revised_class_distribution[count_columns] = (
    revised_class_distribution[count_columns]
    .fillna(0)
    .astype(int)
)

revised_class_distribution["total_boxes"] = (
    revised_class_distribution["train_boxes"]
    + revised_class_distribution["validation_boxes"]
)
revised_class_distribution["validation_box_pct"] = (
    revised_class_distribution["validation_boxes"]
    .div(
        revised_class_distribution["total_boxes"].replace(0, np.nan)
    )
    .mul(100)
    .round(2)
)

revised_class_distribution = revised_class_distribution.sort_values(
    "total_boxes", ascending=False
)

revised_class_distribution.to_csv(
    package_dir / "class_distribution.csv",
    encoding="utf-8-sig",
)
revised_class_distribution.to_csv(
    package_dir / "split_class_distribution.csv",
    encoding="utf-8-sig",
)

# Train/Validation 목록을 현재 버전으로 저장
for split_name in ["train", "validation"]:
    revised_manifest.loc[
        revised_manifest["split"].eq(split_name)
    ].to_csv(
        package_dir / f"{split_name}_manifest.csv",
        index=False,
        encoding="utf-8-sig",
    )

# 3. 기존 제외 내역에 추가 5장의 전체 클래스 연결
old_audit_dir = INTEGRATION_RESULT_DIR / "exclusion_audit"

old_exclusion_summary = pd.read_csv(
    old_audit_dir / "class_exclusion_summary.csv",
    encoding="utf-8-sig",
)
old_exclusion_details = pd.read_csv(
    old_audit_dir / "excluded_image_details.csv",
    encoding="utf-8-sig",
).fillna("")

additional_lookup = (
    additional_exclusions.set_index("image_id").to_dict("index")
)

extra_detail_rows = []

for record in removed_box_records.to_dict("records"):
    source = additional_lookup[record["image_id"]]

    # 이번 5장은 SELECT 원본 JSON과 직접 연결
    if source["dataset"] != "select":
        raise ValueError(
            "추가 제외 중 대회 이미지가 있습니다. "
            "대회 원본 JSON 연결을 별도로 확인해야 합니다."
        )

    extra_detail_rows.append({
        "데이터": "SELECT",
        "클래스명": record["drug_code"],
        "이미지 ID": int(source["original_image_id"]),
        "이미지 파일명": Path(source["image_path"]).name,
        "JSON 경로": source["original_json_file"],
        "검증 단계": "6-3 추가 bbox 오류 시각 검토",
        "제외 사유": (
            "이미지 내 bbox가 알약을 감싸지 않고 주로 배경을 포함하여 "
            "해당 이미지와 전체 annotation 제외"
        ),
    })

extra_details = pd.DataFrame(extra_detail_rows).drop_duplicates(
    ["데이터", "클래스명", "JSON 경로", "이미지 ID"]
)

revised_exclusion_details = (
    pd.concat(
        [old_exclusion_details, extra_details],
        ignore_index=True,
    )
    .drop_duplicates([
        "데이터", "클래스명", "JSON 경로",
        "이미지 ID", "검증 단계", "제외 사유",
    ])
    .sort_values(["데이터", "클래스명", "이미지 파일명"])
    .reset_index(drop=True)
)

def join_reason_values(values):
    return " / ".join(sorted({
        str(value).strip()
        for value in values
        if pd.notna(value) and str(value).strip()
    }))

# 검증 전 전체 수량은 유지하고 제외 수만 다시 계산
revised_exclusion_summary_rows = []

for record in old_exclusion_summary.to_dict("records"):
    detail = revised_exclusion_details.loc[
        revised_exclusion_details["데이터"].eq(record["데이터"])
        & revised_exclusion_details["클래스명"].eq(record["클래스명"])
    ]

    excluded_images = detail["이미지 ID"].nunique()
    excluded_jsons = detail["JSON 경로"].nunique()
    retained_images = int(record["전체 이미지 수"]) - excluded_images

    if retained_images < 0:
        raise ValueError("제외 이미지 집계가 전체 이미지 수를 초과했습니다.")

    revised_exclusion_summary_rows.append({
        "데이터": record["데이터"],
        "클래스명": record["클래스명"],
        "연결된 JSON 개수": int(record["연결된 JSON 개수"]),
        "제외 JSON 수": excluded_jsons,
        "전체 이미지 수": int(record["전체 이미지 수"]),
        "제외 이미지 수": excluded_images,
        "유지 이미지 수": retained_images,
        "클래스 전체 제외": retained_images == 0,
        "제외된 사유": join_reason_values(detail["제외 사유"]),
    })

revised_exclusion_summary_all = pd.DataFrame(
    revised_exclusion_summary_rows
)

# 요청 목적에 맞게 실제 제외가 발생한 클래스만 요약 CSV에 포함
revised_exclusion_summary = (
    revised_exclusion_summary_all.loc[
        revised_exclusion_summary_all["제외 이미지 수"].gt(0)
    ]
    .sort_values(
        ["데이터", "제외 이미지 수", "클래스명"],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

revised_exclusion_summary.to_csv(
    package_dir / "class_exclusion_summary.csv",
    index=False,
    encoding="utf-8-sig",
)
revised_exclusion_details.to_csv(
    package_dir / "excluded_image_details.csv",
    index=False,
    encoding="utf-8-sig",
)

# 출처별 실제 제외 이미지 수는 중복 없이 확인
revised_excluded_image_counts = (
    revised_exclusion_details.groupby("데이터")["이미지 ID"].nunique()
)

assert revised_excluded_image_counts["SELECT"] == 136
assert revised_excluded_image_counts["대회"] == 17

# 4. 클래스별 박스 수와 분할 비율 그래프
plot_table = revised_class_distribution
positions = np.arange(len(plot_table))

fig, (ax_count, ax_ratio) = plt.subplots(
    1, 2,
    figsize=(18, max(12, len(plot_table) * 0.28)),
    sharey=True,
    gridspec_kw={"width_ratios": [2, 1]},
)

ax_count.barh(
    positions, plot_table["train_boxes"],
    color="steelblue", label="Train",
)
ax_count.barh(
    positions, plot_table["validation_boxes"],
    left=plot_table["train_boxes"],
    color="darkorange", label="Validation",
)

for position, (_, row) in enumerate(plot_table.iterrows()):
    ax_count.text(
        row["total_boxes"] + plot_table["total_boxes"].max() * 0.01,
        position,
        f"{int(row['train_boxes'])} / {int(row['validation_boxes'])}",
        va="center",
        fontsize=7,
    )

ax_count.set_yticks(positions)
ax_count.set_yticklabels(plot_table["drug_code"], fontsize=8)
ax_count.invert_yaxis()
ax_count.set_xlim(0, plot_table["total_boxes"].max() * 1.22)
ax_count.set_xlabel("Number of boxes")
ax_count.set_title("Updated box counts | Train / Validation")
ax_count.legend(loc="lower right")
ax_count.grid(axis="x", alpha=0.2)
ax_count.set_axisbelow(True)

train_pct = (
    plot_table["train_boxes"]
    .div(plot_table["total_boxes"].replace(0, np.nan))
    .mul(100)
)

ax_ratio.barh(positions, train_pct, color="steelblue")
ax_ratio.barh(
    positions,
    plot_table["validation_box_pct"],
    left=train_pct,
    color="darkorange",
)
ax_ratio.axvline(80, color="black", linestyle="--", linewidth=1)

for position, percentage in enumerate(plot_table["validation_box_pct"]):
    ax_ratio.text(
        101, position, f"{percentage:.1f}%",
        va="center", fontsize=7,
    )

ax_ratio.set_xlim(0, 114)
ax_ratio.set_xticks([0, 20, 40, 60, 80, 100])
ax_ratio.set_xlabel("Share within each class (%)")
ax_ratio.set_title("Updated split ratio | Validation %")

plt.tight_layout()

fig.savefig(
    package_dir / "class_box_distribution.png",
    dpi=160,
    bbox_inches="tight",
)
fig.savefig(
    package_dir / "class_box_distribution.pdf",
    bbox_inches="tight",
)
plt.show()

# 5. 실제 수량을 반영한 노트북 기록 생성
train_document = revised_coco["train"]
validation_document = revised_coco["validation"]

train_count = len(train_document["images"])
validation_count = len(validation_document["images"])
total_count = train_count + validation_count

train_boxes_count = len(train_document["annotations"])
validation_boxes_count = len(validation_document["annotations"])

both_class_count = int(
    (
        revised_class_distribution["train_boxes"].gt(0)
        & revised_class_distribution["validation_boxes"].gt(0)
    ).sum()
)

pending_candidates = pd.read_csv(
    package_dir / "remaining_thin_bbox_candidates.csv",
    encoding="utf-8-sig",
)

notebook_revision_note = f"""## 6-4. 추가 bbox 오류 제외 및 데이터셋 갱신

### 확인한 문제
시각 검토에서 5개 bbox가 알약을 감싸지 않고 주로 배경을 포함하는 오류를 확인하였다.
좌표는 수정하지 않고 해당 이미지 5장과 연결된 모든 annotation을 제외하였다.

### 기존 검사의 한계
- 좌표·크기가 양수이고 이미지 범위 안에 있어도 실제 물체를 감싸지 않을 수 있다.
- 면적이 너비×높이와 일치하는 것은 라벨의 의미적 정확성을 보장하지 않는다.
- 중첩 검사는 다른 박스와 겹치지 않는 단독 위치 오류를 놓칠 수 있다.
- 실물 길이 대비 비율은 긴 변을 사용하므로 짧은 변의 이상을 놓칠 수 있다.
- 통계적 이상치 후보의 표본 검토는 전체 박스의 시각 검증이 아니다.
- Train/Validation 전체 분포가 유사해도 개별 라벨 오류는 남을 수 있다.

각 사례가 과거 IQR 후보에 포함됐는지, 실제 시각 검토 대상으로 표시됐는지는
당시 검사 기록을 확인해야 한다. 모든 검사를 통과했다고 단정하지 않는다.

### 수정 방법
- 나머지 이미지의 기존 Train/Validation 배정과 ID를 유지하였다.
- 무작위 재분할은 수행하지 않았다.
- 기존 이미지 파일은 유지하고 학습 라벨과 사용 목록에서 제외하였다.
- 새 분할 목록, 클래스 분포, 제외 요약·상세 파일을 생성하였다.

### 수정 후 수량

| 구분 | 이미지 | 박스 |
|---|---:|---:|
| Train | {train_count:,} | {train_boxes_count:,} |
| Validation | {validation_count:,} | {validation_boxes_count:,} |
| 전체 | {total_count:,} | {train_boxes_count + validation_boxes_count:,} |

- 이미지 비율: Train {train_count / total_count:.2%}, Validation {validation_count / total_count:.2%}
- 양쪽에 모두 포함된 클래스: {both_class_count}종
- 누적 제외 이미지: SELECT 136장, 대회 17장
- 같은 조합 그룹의 분할 간 중복 없음
- 나머지 이미지의 기존 분할 변경 없음

### 추가 검토와 공유
- 짧은 변 30px 이하 또는 긴 변/짧은 변 비율 8 이상을 추가 후보로 검사하였다.
- 확인된 5장 외 남은 후보: {len(pending_candidates):,}개 박스.
- 후보는 자동 제외하지 않았으며 시각 검토 후 판단한다.
- 분류 데이터도 추가 제외 원본에서 잘라 만든 샘플을 제외한다.
- 팀원은 수정된 training_package를 사용하고 Dataset·DataLoader를 다시 생성한다.
- 이전 분포 그래프와 제외 내역은 구버전 자료로 구분한다.
"""

(package_dir / "notebook_revision_note.md").write_text(
    notebook_revision_note,
    encoding="utf-8",
)
display(Markdown(notebook_revision_note))

# 6. 새 공유용 ZIP을 갱신
with zipfile.ZipFile(
    NEW_TRAINING_PACKAGE_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as z:
    for path in sorted(package_dir.rglob("*")):
        if path.is_file():
            z.write(
                path,
                arcname=path.relative_to(package_dir).as_posix(),
            )

with zipfile.ZipFile(NEW_TRAINING_PACKAGE_ZIP, "r") as z:
    assert z.testzip() is None

print("\n========== 6-4. 갱신 완료 ==========")
print("\n[출처별 누적 제외 이미지 수]")
display(revised_excluded_image_counts.rename("제외 이미지 수"))

print("\n[제외가 발생한 클래스 요약: 처음 20행]")
display(revised_exclusion_summary.head(20))

print("\n[추가 제외 이미지의 기존 분할]")
display(additional_exclusions[[
    "original_stem", "drug_code", "split"
]])

print(f"\n결과 폴더: {package_dir}")
print(f"교체할 새 공유용 ZIP: {NEW_TRAINING_PACKAGE_ZIP}")
print("노트북 근거: notebook_revision_note.md")
print("클래스 분포: class_distribution.csv")
print("Train 목록: train_manifest.csv")
print("Validation 목록: validation_manifest.csv")
print("제외 요약: class_exclusion_summary.csv")
print("제외 상세: excluded_image_details.csv")
print("※ 제외 요약에는 제외 이미지가 1장 이상인 클래스만 포함했습니다.")
print("※ 클래스별 제외 수의 합은 실제 제외 이미지 수와 다를 수 있습니다.")

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

# 추가 5장을 제외한 새 학습 패키지 사용
if "NEW_TRAINING_PACKAGE_DIR" not in globals():
    raise RuntimeError(
        "새 학습 패키지 경로 변수가 없습니다. "
        "추가 5장 제외 작업에서 출력된 training_package 폴더 경로가 필요합니다."
    )

stats_package_dir = Path(NEW_TRAINING_PACKAGE_DIR)
stats_results = {}

for split_name in ["train", "validation"]:
    label_path = (
        stats_package_dir / "annotations" / f"{split_name}.json"
    )

    with label_path.open("r", encoding="utf-8") as file:
        labels = json.load(file)

    image_lookup = {
        image["id"]: image
        for image in labels["images"]
    }

    box_rows = []

    for annotation in labels["annotations"]:
        _, _, width, height = map(float, annotation["bbox"])
        image = image_lookup[annotation["image_id"]]
        image_area = float(image["width"]) * float(image["height"])

        box_rows.append({
            "width": width,
            "height": height,
            "area": width * height,
            "relative_area_pct": width * height / image_area * 100,
            "aspect_ratio": width / height,
        })

    boxes = pd.DataFrame(box_rows)

    stats_results[split_name] = {
        "평균 너비 (px)": boxes["width"].mean(),
        "평균 높이 (px)": boxes["height"].mean(),
        "평균 면적 (px²)": boxes["area"].mean(),
        "이미지 대비 평균 박스 면적 (%)": (
            boxes["relative_area_pct"].mean()
        ),
        "평균 종횡비": boxes["aspect_ratio"].mean(),
        "중앙값 종횡비": boxes["aspect_ratio"].median(),
    }

    print(
        f"{split_name}: "
        f"이미지 {len(image_lookup):,}장 / "
        f"박스 {len(boxes):,}개"
    )

updated_box_comparison = pd.DataFrame(stats_results).rename(
    columns={"train": "Train", "validation": "Validation"}
)

print("\n========== 박스 크기·종횡비 비교 ==========")
display(updated_box_comparison.round(2))

print("※ 종횡비 = 너비 ÷ 높이")
print("※ 파일 수정·삭제 없음")

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# 추가 5장 제외 후의 학습용 JSON 사용
hist_package_dir = Path(NEW_TRAINING_PACKAGE_DIR)
hist_values = {}

for split_name in ["train", "validation"]:
    with (
        hist_package_dir / "annotations" / f"{split_name}.json"
    ).open("r", encoding="utf-8") as file:
        labels = json.load(file)

    image_lookup = {
        image["id"]: image
        for image in labels["images"]
    }

    relative_areas = []
    aspect_ratios = []

    for annotation in labels["annotations"]:
        _, _, width, height = map(float, annotation["bbox"])
        image = image_lookup[annotation["image_id"]]
        image_area = float(image["width"]) * float(image["height"])

        relative_areas.append(width * height / image_area * 100)
        aspect_ratios.append(width / height)

    hist_values[split_name] = {
        "relative_area": np.asarray(relative_areas),
        "aspect_ratio": np.asarray(aspect_ratios),
    }

    print(
        f"{split_name}: 이미지 {len(image_lookup):,}장 / "
        f"박스 {len(relative_areas):,}개"
    )

# 양쪽에 동일한 구간을 적용하고, 각 분할의 전체 박스 수로 정규화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title, xlabel in [
    (
        axes[0],
        "relative_area",
        "Box area / image area",
        "Relative area (%)",
    ),
    (
        axes[1],
        "aspect_ratio",
        "Box aspect ratio",
        "Width / height",
    ),
]:
    combined = np.concatenate([
        hist_values["train"][metric],
        hist_values["validation"][metric],
    ])
    bins = np.histogram_bin_edges(combined, bins=50)

    for split_name, color in [
        ("train", "steelblue"),
        ("validation", "darkorange"),
    ]:
        values = hist_values[split_name][metric]

        ax.hist(
            values,
            bins=bins,
            weights=np.full(len(values), 100.0 / len(values)),
            histtype="step",
            linewidth=2,
            color=color,
            label=split_name,
        )

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Boxes (%)")
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=100))
    ax.legend()
    ax.grid(axis="y", alpha=0.2)

fig.tight_layout()

# 학습 패키지 밖에 저장하여 기존 ZIP 내용과 혼동 방지
hist_output_dir = hist_package_dir.parent / "distribution_review_updated"
hist_output_dir.mkdir(parents=True, exist_ok=True)
hist_output_path = hist_output_dir / "box_area_aspect_distribution.png"

fig.savefig(hist_output_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"\n그래프 저장: {hist_output_path}")
print("※ 각 분할의 전체 박스를 100%로 계산했습니다.")
print("※ 극단값을 포함한 전체 범위를 표시했습니다.")
print("※ 학습용 JSON과 분할 목록은 변경하지 않았습니다.")

In [ ]:
import json
import re
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 1. 갱신된 학습 패키지와 기존 원본 메타데이터
color_package_dir = Path(NEW_TRAINING_PACKAGE_DIR)
color_metadata_path = Path(CODEIT_RESULT_DIR) / "source_metadata.json"

with color_metadata_path.open("r", encoding="utf-8") as file:
    color_metadata = json.load(file)

# 2. 원본 메타데이터에서 클래스별 색상 기록 수집
color_records_by_class = {}

def collect_color_metadata(node):
    if isinstance(node, list):
        for item in node:
            collect_color_metadata(item)
        return

    if not isinstance(node, dict):
        return

    if "color_class1" in node or "color_class2" in node:
        code = None

        for field in ["dl_mapping_code", "drug_N", "drug_code"]:
            match = re.fullmatch(
                r"K-(\d{6})", str(node.get(field, "")).strip()
            )
            if match:
                code = int(match.group(1))
                break

        colors = []
        for field in ["color_class1", "color_class2"]:
            value = node.get(field)
            if value is None:
                continue

            value = str(value).strip()
            if value.lower() not in {"", "nan", "none", "null"}:
                if value not in colors:
                    colors.append(value)

        if code is not None and colors:
            color_label = " + ".join(sorted(colors))
            color_records_by_class.setdefault(code, set()).add(
                color_label
            )

    for value in node.values():
        if isinstance(value, (dict, list)):
            collect_color_metadata(value)

collect_color_metadata(color_metadata)

if not color_records_by_class:
    raise ValueError(
        "source_metadata.json에서 클래스 코드와 색상 정보를 "
        "연결하지 못했습니다. 색상 집계는 수행하지 않았습니다."
    )

color_conflicts = {
    code: values
    for code, values in color_records_by_class.items()
    if len(values) > 1
}

color_lookup = {
    code: next(iter(values))
    for code, values in color_records_by_class.items()
    if len(values) == 1
}

# 3. 갱신된 전체 annotation에 색상 연결
color_box_rows = []
color_image_counts = {}

for split_name in ["train", "validation"]:
    with (
        color_package_dir / "annotations" / f"{split_name}.json"
    ).open("r", encoding="utf-8") as file:
        labels = json.load(file)

    color_image_counts[split_name] = len(labels["images"])

    for annotation in labels["annotations"]:
        category_id = int(annotation["category_id"])

        if category_id in color_conflicts:
            color_label = "색상 기록 상충"
            status = "상충"
        elif category_id in color_lookup:
            color_label = color_lookup[category_id]
            status = "연결"
        else:
            color_label = "정보 연결 안 됨"
            status = "미연결"

        color_box_rows.append({
            "split": split_name,
            "image_id": annotation["image_id"],
            "annotation_id": annotation["id"],
            "category_id": category_id,
            "drug_code": f"K-{category_id:06d}",
            "color": color_label,
            "color_status": status,
        })

updated_color_boxes = pd.DataFrame(color_box_rows)

# 4. 연결률과 색상별 구성 비율
updated_color_coverage = (
    updated_color_boxes.assign(
        linked=lambda frame: frame["color_status"].eq("연결")
    )
    .groupby("split")
    .agg(
        boxes=("annotation_id", "size"),
        boxes_with_color=("linked", "sum"),
    )
    .reindex(["train", "validation"])
)

updated_color_coverage["coverage_pct"] = (
    updated_color_coverage["boxes_with_color"]
    / updated_color_coverage["boxes"] * 100
)

updated_color_counts = pd.crosstab(
    updated_color_boxes["color"],
    updated_color_boxes["split"],
).reindex(columns=["train", "validation"], fill_value=0)

# 분모는 각 분할의 전체 박스 수이며, 미연결·상충도 포함
updated_color_pct = updated_color_counts.div(
    updated_color_counts.sum(axis=0), axis=1
) * 100

updated_color_comparison = pd.DataFrame({
    "Train 박스 수": updated_color_counts["train"],
    "Validation 박스 수": updated_color_counts["validation"],
    "Train 비율 (%)": updated_color_pct["train"],
    "Validation 비율 (%)": updated_color_pct["validation"],
    "차이 (%p)": (
        updated_color_pct["train"]
        - updated_color_pct["validation"]
    ).abs(),
})

known_color_comparison = updated_color_comparison.drop(
    index=["정보 연결 안 됨", "색상 기록 상충"],
    errors="ignore",
)

max_color_difference = known_color_comparison["차이 (%p)"].max()
max_difference_color = known_color_comparison["차이 (%p)"].idxmax()

# 5. 결과 출력
print("========== 추가 제외 후 색상 분포 비교 ==========")

for split_name in ["train", "validation"]:
    print(
        f"{split_name}: 이미지 {color_image_counts[split_name]:,}장 / "
        f"박스 {updated_color_coverage.loc[split_name, 'boxes']:,}개"
    )

print("\n[색상 정보 연결 범위]")
display(updated_color_coverage.round(2))

print("\n[색상별 구성 — 각 분할의 전체 박스 기준]")
display(updated_color_comparison.round(2))

if color_conflicts:
    print("\n[색상 기록이 상충하여 연결을 보류한 클래스]")
    display(pd.DataFrame([
        {
            "클래스": f"K-{code:06d}",
            "원본 색상 기록": " / ".join(sorted(values)),
        }
        for code, values in sorted(color_conflicts.items())
    ]))

# 6. 그래프 출력 — 한글 글꼴 문제를 피하도록 색상에 번호 부여
color_graph_data = updated_color_pct.copy()
color_graph_codes = [
    f"C{index + 1}" for index in range(len(color_graph_data))
]

color_graph_key = pd.DataFrame({
    "그래프 코드": color_graph_codes,
    "색상": color_graph_data.index,
})

color_graph_data.index = color_graph_codes

fig, ax = plt.subplots(figsize=(12, 5))
color_graph_data.plot.bar(
    ax=ax,
    color=["steelblue", "darkorange"],
    rot=0,
)
ax.set_title("Color composition after additional exclusions")
ax.set_xlabel("Color code")
ax.set_ylabel("Boxes (%)")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()

# 학습 라벨과 기존 ZIP은 변경하지 않음
color_output_dir = (
    color_package_dir.parent / "distribution_review_updated"
)
color_output_dir.mkdir(parents=True, exist_ok=True)

fig.savefig(
    color_output_dir / "color_distribution.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

display(color_graph_key)

# 7. CSV 저장
updated_color_coverage.to_csv(
    color_output_dir / "color_coverage.csv",
    encoding="utf-8-sig",
)
updated_color_comparison.to_csv(
    color_output_dir / "color_distribution.csv",
    encoding="utf-8-sig",
)
color_graph_key.to_csv(
    color_output_dir / "color_graph_key.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n[노션 기록용 수치]")
print(
    "Train 색상 정보 연결률: "
    f"{updated_color_coverage.loc['train', 'coverage_pct']:.2f}%"
)
print(
    "Validation 색상 정보 연결률: "
    f"{updated_color_coverage.loc['validation', 'coverage_pct']:.2f}%"
)
print(
    "연결된 색상 범주 중 구성 비율의 최대 차이: "
    f"{max_difference_color}, {max_color_difference:.2f}%p"
)
print("\n※ 색상 비율의 분모는 각 분할의 전체 박스 수입니다.")
print("※ 제품 메타데이터 기준이며 픽셀 색상 측정 결과가 아닙니다.")
print("※ 학습 라벨·분할 목록·기존 ZIP은 변경하지 않았습니다.")
print(f"\n결과 저장 폴더: {color_output_dir}")

In [ ]:
from pathlib import Path
import pandas as pd

check_package_dir = Path(NEW_TRAINING_PACKAGE_DIR)

for filename in [
    "class_exclusion_summary.csv",
    "excluded_image_details.csv",
    "additional_exclusions.csv",
]:
    path = check_package_dir / filename

    if path.is_file():
        table = pd.read_csv(path, encoding="utf-8-sig")
        print(f"[저장 확인] {filename}: {len(table):,}행")
    else:
        print(f"[파일 없음] {filename}")

detail_path = check_package_dir / "excluded_image_details.csv"

if detail_path.is_file():
    details = pd.read_csv(detail_path, encoding="utf-8-sig")

    print("\n[데이터별 제외 이미지 수]")
    display(
        details.groupby("데이터")["이미지 파일명"]
        .nunique()
        .rename("제외 이미지 수")
        .to_frame()
    )

print(f"\n확인 폴더: {check_package_dir}")
print("※ 파일 읽기만 수행합니다.")